# Unified Multi-Class Skin Disease Classification
### An explainable and calibrated two-stage stacking ensemble of CNN, ViT, and YOLO models

| Part | Description |
|---|---|
| Part 1 | Per-class analysis: the best model for each disease |
| Part 2 | Stacking ensemble meta-classifier with temperature calibration |
| Part 3 | Grad-CAM and occlusion sensitivity: where each model attends |
| Part 4 | Attention alignment against lesion masks |
| Part 5 | Two-stage prediction: ensemble with a per-class specialist cross-check |

Saved models and results are loaded from Drive when available; otherwise they are generated and cached.


In [ ]:
# Block 0: Install & Imports
!pip install ultralytics --quiet

import os, gc, glob, zipfile, shutil, pickle, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import cv2
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

mixed_precision.set_global_policy('mixed_float16')

from google.colab import drive
drive.mount('/content/drive')

print('Mixed Precision Enabled.')
print('All imports done.')

# Reproducibility seeds
import random
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
print('Random seeds set (42) for reproducibility.')

In [ ]:
# Block 1: Drive paths and configuration

ROOT_DIR = '/content/drive/MyDrive/THESIS'
DRIVE_WORKSPACE = os.path.join(ROOT_DIR, '10_final_run')

MODELS_DIR      = os.path.join(DRIVE_WORKSPACE, 'saved_models')
LOGS_DIR        = os.path.join(DRIVE_WORKSPACE, 'training_logs')
PREDS_DIR       = os.path.join(DRIVE_WORKSPACE, 'predictions')
PLOTS_DIR       = os.path.join(DRIVE_WORKSPACE, 'analytics_plots')
GRADCAM_DIR     = os.path.join(DRIVE_WORKSPACE, 'gradcam_results')
PERCLASS_DIR    = os.path.join(DRIVE_WORKSPACE, 'perclass_analysis')
RESULTS_DIR     = os.path.join(DRIVE_WORKSPACE, 'final_results')

for d in [
    MODELS_DIR,
    LOGS_DIR,
    PREDS_DIR,
    PLOTS_DIR,
    GRADCAM_DIR,
    PERCLASS_DIR,
    RESULTS_DIR
]:
    os.makedirs(d, exist_ok=True)

# Dataset

dataset_final_path = os.path.join(
    DRIVE_WORKSPACE,
    'Skin_Dataset_10_Classes'
)

if not os.path.exists(dataset_final_path):
    raise FileNotFoundError(
        f"Dataset not found:\n{dataset_final_path}"
    )

print("Dataset found.")
print(dataset_final_path)

CLASSES = [
    'Acne',
    'Actinic keratosis',
    'Basal cell carcinoma',
    'Chickenpox',
    'Measles',
    'Melanocytic nevus',
    'Normal  Unknown',
    'Tinea',
    'Vascular lesion',
    'Vitiligo'
]

ALL_MODELS = [
    'EfficientNetB0',
    'MobileNetV2',
    'ConvNeXtTiny',
    'DenseNet121',
    'Xception',
    'ResNet50V2',
    'InceptionV3',
    'EfficientNetB1',
    'DenseNet169',
    'VGG16',
    'ViT_B16',
    'YOLO11m_cls'
]

# TOP3 is not fixed here.
# It is computed automatically in Block 6, after ALL 12 models
# (10 CNN + ViT + YOLO) have been trained/loaded, based on real
# test accuracy.

BATCH_SIZE = 64
IMG_SIZE = (256, 256)

MODELS_CONFIG = {
    'EfficientNetB0': (tf.keras.applications.EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input),
    'MobileNetV2': (tf.keras.applications.MobileNetV2, tf.keras.applications.mobilenet_v2.preprocess_input),
    'ConvNeXtTiny': (tf.keras.applications.ConvNeXtTiny, tf.keras.applications.convnext.preprocess_input),
    'DenseNet121': (tf.keras.applications.DenseNet121, tf.keras.applications.densenet.preprocess_input),
    'Xception': (tf.keras.applications.Xception, tf.keras.applications.xception.preprocess_input),
    'ResNet50V2': (tf.keras.applications.ResNet50V2, tf.keras.applications.resnet_v2.preprocess_input),
    'InceptionV3': (tf.keras.applications.InceptionV3, tf.keras.applications.inception_v3.preprocess_input),
    'EfficientNetB1': (tf.keras.applications.EfficientNetB1, tf.keras.applications.efficientnet.preprocess_input),
    'DenseNet169': (tf.keras.applications.DenseNet169, tf.keras.applications.densenet.preprocess_input),
    'VGG16': (tf.keras.applications.VGG16, tf.keras.applications.vgg16.preprocess_input),
}

print(f'Workspace: {DRIVE_WORKSPACE}')
print(f'Dataset: {dataset_final_path}')
print(f'Classes ({len(CLASSES)}): {CLASSES}')
print(f'Total models configured: {len(ALL_MODELS)} (TOP3 will be selected dynamically later)')

In [ ]:
# Block 2: Load Datasets
for f in glob.glob('/content/*_cache*'):
    try: os.remove(f)
    except: pass

test_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(dataset_final_path, 'test'),
    shuffle=False, batch_size=BATCH_SIZE,
    image_size=IMG_SIZE, label_mode='categorical'
)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(dataset_final_path, 'valid') if os.path.exists(os.path.join(dataset_final_path, 'valid')) else os.path.join(dataset_final_path, 'val'),
    shuffle=False, batch_size=BATCH_SIZE,
    image_size=IMG_SIZE, label_mode='categorical'
)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(dataset_final_path, 'train'),
    shuffle=True, batch_size=BATCH_SIZE,
    image_size=IMG_SIZE, label_mode='categorical'
)

class_names = test_dataset.class_names
nb_classes  = len(class_names)

AUTOTUNE     = tf.data.AUTOTUNE
test_dataset = test_dataset.cache('/content/test_cache').prefetch(AUTOTUNE)
val_dataset  = val_dataset.cache('/content/val_cache').prefetch(AUTOTUNE)

y_test_true = np.concatenate([y for x, y in test_dataset], axis=0).argmax(axis=1)
y_val_true  = np.concatenate([y for x, y in val_dataset],  axis=0).argmax(axis=1)

# Class weights
train_labels_for_weights = []
for i, class_name in enumerate(class_names):
    class_dir  = os.path.join(dataset_final_path, 'train', class_name)
    num_images = len([f for f in os.listdir(class_dir) if os.path.isfile(os.path.join(class_dir, f))])
    train_labels_for_weights.extend([i] * num_images)
class_weights_array = compute_class_weight('balanced', classes=np.unique(train_labels_for_weights), y=train_labels_for_weights)
class_weight_dict   = {i: w for i, w in enumerate(class_weights_array)}

# Augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.5),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])
train_dataset_aug = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

print(f'Classes: {class_names}')
print(f'Test samples : {len(y_test_true)}')
print(f'Val samples  : {len(y_val_true)}')

In [ ]:
# Block 2-A: Dataset and mask report (fully dynamic; nothing hardcoded)
import os, glob, re
import numpy as np, pandas as pd, cv2
from collections import Counter
from PIL import Image

# ---- Only these are configurable paths; everything else is derived ----
DATA_ROOT     = dataset_final_path
REAL_MASK_DIR = '/content/drive/MyDrive/THESIS/30_masks/ISIC_masks'

IMG_EXT = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff',
           '*.JPG', '*.JPEG', '*.PNG')

def list_images(folder):
    fs = []
    for e in IMG_EXT:
        fs += glob.glob(os.path.join(folder, e))
    return fs

def find_mask(img_path, mask_dir):
    base = os.path.splitext(os.path.basename(img_path))[0]
    core = re.match(r'(ISIC_\d+)', base)
    core = core.group(1) if core else None
    for cand in [base + '.png', base + '_segmentation.png', base + '.PNG',
                 (core + '.png') if core else None]:
        if cand and os.path.exists(os.path.join(mask_dir, cand)):
            return os.path.join(mask_dir, cand)
    return None

# ---- Derive splits and classes from the folder tree (no fixed lists) ----
splits = sorted([d for d in os.listdir(DATA_ROOT)
                 if os.path.isdir(os.path.join(DATA_ROOT, d))])
ref_split = 'test' if 'test' in splits else (splits[0] if splits else None)
classes = sorted([c for c in os.listdir(os.path.join(DATA_ROOT, ref_split))
                  if os.path.isdir(os.path.join(DATA_ROOT, ref_split, c))]) if ref_split else []

# ---- Detect which classes actually have masks available (dynamic) ----
maskable_classes = []
if os.path.isdir(REAL_MASK_DIR):
    for cls in classes:
        imgs = list_images(os.path.join(DATA_ROOT, ref_split, cls))
        if any(find_mask(ip, REAL_MASK_DIR) for ip in imgs):
            maskable_classes.append(cls)

# ============================ PART 1: DATASET ============================
print('=' * 72)
print(' PART 1: DATASET SUMMARY')
print('=' * 72)
print(f' Root    : {DATA_ROOT}')
print(f' Splits  : {splits}   |   Classes detected : {len(classes)}')

rows = []
for cls in classes:
    row = {'Class': cls}
    for sp in splits:
        row[sp] = len(list_images(os.path.join(DATA_ROOT, sp, cls)))
    row['Total'] = sum(row[sp] for sp in splits)
    rows.append(row)
df = pd.DataFrame(rows)
if not df.empty:
    totals = {'Class': 'TOTAL', **{sp: int(df[sp].sum()) for sp in splits},
              'Total': int(df['Total'].sum())}
    df = pd.concat([df, pd.DataFrame([totals])], ignore_index=True)
    print('\n  Image counts per class:')
    print(df.to_string(index=False))

    pc = df[df['Class'] != 'TOTAL']['Total']
    spread = int(pc.max() - pc.min())
    print(f'\n  Grand total : {int(pc.sum())}   |   Min/Max per class : {int(pc.min())}/{int(pc.max())}'
          f' |   Spread : {spread}   |   Balanced : {"yes" if spread <= max(5, 0.02*pc.max()) else "no"}')

# ---- image properties, derived from actual files ----
print('\n  Image properties (sampled from each class):')
prop = []
for cls in classes:
    fs = list_images(os.path.join(DATA_ROOT, ref_split, cls))
    if fs:
        im = Image.open(fs[0])
        prop.append({'Class': cls, 'Size': f'{im.size[0]}x{im.size[1]}',
                     'Mode': im.mode, 'Format': im.format})
if prop:
    pdf = pd.DataFrame(prop)
    common_size = pdf['Size'].mode()[0]
    print(pdf.to_string(index=False))
    print(f' Most common size: {common_size}   |   uniform: {"yes" if pdf["Size"].nunique()==1 else "no"}')

# ==================== PART 2: NAMING / SOURCE (dynamic) =================
print('\n' + '=' * 72)
print(' PART 2: FILE-NAMING PATTERN PER CLASS')
print('=' * 72)
for cls in classes:
    files = [os.path.basename(f) for f in list_images(os.path.join(DATA_ROOT, ref_split, cls))]
    if not files:
        continue
    isic = sum(bool(re.match(r'ISIC_\d+', f)) for f in files)
    exts = Counter(os.path.splitext(f)[1].lower() for f in files)
    tag  = f'{isic}/{len(files)} ISIC-named' if isic else 'non-ISIC / custom naming'
    print(f' {cls:<24} {tag}   extensions: {dict(exts)}')

# ======================= PART 3: MASK REPORT (dynamic) =================
print('\n' + '=' * 72)
print(' PART 3: LESION-MASK REPORT')
print('=' * 72)
if not os.path.isdir(REAL_MASK_DIR):
    print(f' Mask directory not found: {REAL_MASK_DIR}')
elif not maskable_classes:
    print(f' Mask directory exists but no image names matched any mask in {REAL_MASK_DIR}')
else:
    print(f' Mask directory   : {REAL_MASK_DIR}')
    print(f' Classes with masks: {maskable_classes}')
    mrows = []; t_img = t_msk = 0; fg = []; dims = []
    for cls in maskable_classes:
        imgs = list_images(os.path.join(DATA_ROOT, ref_split, cls))
        matched = [ip for ip in imgs if find_mask(ip, REAL_MASK_DIR)]
        mrows.append({'Class': cls, 'Images': len(imgs), 'With_mask': len(matched),
                      'Coverage_%': round(100*len(matched)/len(imgs), 1) if imgs else 0})
        t_img += len(imgs); t_msk += len(matched)
        for mp in [find_mask(ip, REAL_MASK_DIR) for ip in matched[:15]]:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            if m is not None:
                dims.append(m.shape); fg.append((m > 127).mean())
    mrows.append({'Class': 'TOTAL', 'Images': t_img, 'With_mask': t_msk,
                  'Coverage_%': round(100*t_msk/t_img, 1) if t_img else 0})
    print('\n  Mask coverage:')
    print(pd.DataFrame(mrows).to_string(index=False))

    if dims:
        h = np.array([d[0] for d in dims]); w = np.array([d[1] for d in dims]); fg = np.array(fg)
        uniq_vals = np.unique(cv2.imread(find_mask(
            [ip for ip in list_images(os.path.join(DATA_ROOT, ref_split, maskable_classes[0]))
             if find_mask(ip, REAL_MASK_DIR)][0], REAL_MASK_DIR), cv2.IMREAD_GRAYSCALE))
        print('\n  Mask properties (sampled):')
        print(f' Size range       : {w.min()}x{h.min()} to {w.max()}x{h.max()}')
        print(f' Lesion area       : mean {fg.mean()*100:.1f}%  (min {fg.min()*100:.1f}%, max {fg.max()*100:.1f}%)')
        print(f' Pixel values      : {sorted(uniq_vals.tolist())[:5]} ... (binary: background/lesion)')
        uncovered = t_img - t_msk
        if uncovered:
            print(f'\n   {uncovered} test images have no matching mask (no public segmentation available).')
print('\n' + '=' * 72)

---
## 2.B Dataset Composition and Provenance

The dataset used in this study is not drawn from a single public benchmark. It is a custom 10-class collection assembled by combining images from several publicly available Kaggle dermatology datasets, balanced to approximately 450-476 images per class. The class set was chosen to unify three families of skin conditions within a single classifier: dermoscopic pigmented lesions (Actinic keratosis, Basal cell carcinoma, Melanocytic nevus, Vascular lesion), infectious exanthems (Chickenpox, Measles), and common inflammatory or pigmentary conditions (Acne, Tinea, Vitiligo, Normal/Unknown).

The code cell below documents, per class, the exact source dataset(s), the number of images drawn from each, and the corresponding Kaggle link, so the composition is fully reproducible. The broader candidate-source exploration and target-count planning (covering 19-22 candidate classes and a 500-image per-class target before the final 10-class scope was fixed) is documented in the accompanying dataset-planning spreadsheets (`Dataset_Count.xlsx` and `500-Image_Balanced_Dataset_Generation_Plan.xlsx`).

The following points should be stated explicitly when reporting this dataset in the thesis:
- The train/validation/test split method (random, stratified by class) and whether it is patient-disjoint or source-disjoint.
- Whether duplicate or near-duplicate images were checked for across splits, to rule out data leakage.
- What the "Normal / Unknown" class contains.
- That dermoscopic close-ups and clinical full-region photographs are mixed by design: this mixture, combined with the inclusion of infectious exanthems alongside dermoscopic lesions, is the unifying contribution of this dataset relative to prior single-source benchmarks.


In [ ]:
# Block 2-B: Dataset Provenance and Source Composition
# This project combines images from multiple publicly available Kaggle skin
# disease datasets into a single balanced 10-class dataset. The table below
# documents, for every class used in this study, which source dataset(s)
# contributed images and how many, so the composition can be audited and
# reproduced. Figures are taken from the dataset-planning spreadsheets
# (Dataset_Count.xlsx, 500-Image_Balanced_Dataset_Generation_Plan.xlsx).

dataset_provenance = [
    {'Class': 'Acne', 'Images': 476,
     'Source Dataset': 'Skin Disease Dataset (pacificrm)',
     'Source Link': 'https://www.kaggle.com/datasets/pacificrm/skindiseasedataset'},
    {'Class': 'Actinic keratosis', 'Images': 476,
     'Source Dataset': 'ISIC Skin Disease Image Dataset - Labelled (riyaelizashaju)',
     'Source Link': 'https://www.kaggle.com/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled'},
    {'Class': 'Basal cell carcinoma', 'Images': 476,
     'Source Dataset': 'ISIC Skin Disease Image Dataset - Labelled (riyaelizashaju)',
     'Source Link': 'https://www.kaggle.com/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled'},
    {'Class': 'Chickenpox', 'Images': 343,
     'Source Dataset': 'Skin Disease Raw Dataset (devdope)',
     'Source Link': 'https://www.kaggle.com/datasets/devdope/skin-disease-raw-dataset'},
    {'Class': 'Chickenpox', 'Images': 107,
     'Source Dataset': 'Monkeypox Skin Image Dataset - MSID (dipuiucse)',
     'Source Link': 'https://www.kaggle.com/datasets/dipuiucse/monkeypoxskinimagedataset'},
    {'Class': 'Measles', 'Images': 294,
     'Source Dataset': 'Skin Disease Raw Dataset (devdope)',
     'Source Link': 'https://www.kaggle.com/datasets/devdope/skin-disease-raw-dataset'},
    {'Class': 'Measles', 'Images': 91,
     'Source Dataset': 'Monkeypox Skin Image Dataset - MSID (dipuiucse)',
     'Source Link': 'https://www.kaggle.com/datasets/dipuiucse/monkeypoxskinimagedataset'},
    {'Class': 'Melanocytic nevus', 'Images': 476,
     'Source Dataset': 'ISIC Skin Disease Image Dataset - Labelled (riyaelizashaju)',
     'Source Link': 'https://www.kaggle.com/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled'},
    {'Class': 'Normal  Unknown', 'Images': 476,
     'Source Dataset': 'Skin Disease Dataset (pacificrm)',
     'Source Link': 'https://www.kaggle.com/datasets/pacificrm/skindiseasedataset'},
    {'Class': 'Tinea', 'Images': 476,
     'Source Dataset': 'Skin Disease Dataset (pacificrm)',
     'Source Link': 'https://www.kaggle.com/datasets/pacificrm/skindiseasedataset'},
    {'Class': 'Vascular lesion', 'Images': 253,
     'Source Dataset': 'ISIC Skin Disease Image Dataset - Labelled (riyaelizashaju)',
     'Source Link': 'https://www.kaggle.com/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled'},
    {'Class': 'Vascular lesion', 'Images': 223,
     'Source Dataset': 'Skin Disease Image Dataset - Balanced (riyaelizashaju)',
     'Source Link': 'https://www.kaggle.com/datasets/riyaelizashaju/skin-disease-image-dataset-balanced'},
    {'Class': 'Vitiligo', 'Images': 476,
     'Source Dataset': 'Skin Disease Dataset (pacificrm)',
     'Source Link': 'https://www.kaggle.com/datasets/pacificrm/skindiseasedataset'},
]

provenance_df = pd.DataFrame(dataset_provenance)
class_totals  = provenance_df.groupby('Class')['Images'].sum().reindex(CLASSES)

print('=' * 72)
print('  DATASET PROVENANCE: source composition per class')
print('=' * 72)
print(provenance_df.to_string(index=False))
print()
print('Per-class totals (all sources combined):')
print(class_totals.to_string())
print(f'\nGrand total images in the combined dataset: {int(class_totals.sum())}')
print(
    '\nNote: several classes were planned at a 500-image target. Where a '
    'candidate source overlapped with a source already included (for '
    'example, the Extended MSID dataset duplicates images already drawn '
    'from MSID for Measles and Chickenpox) or a supplementary source could '
    'not be located, the achieved count is below 500. The full sourcing '
    'plan, including rejected and duplicate sources, is documented in the '
    'accompanying dataset planning spreadsheets.'
)

# Save the provenance table for inclusion in the thesis appendix
provenance_df.to_csv(os.path.join(LOGS_DIR, 'dataset_provenance.csv'), index=False)

# ---- Chart 1: per-class image count, stacked by contributing source dataset ----
pivot = provenance_df.pivot_table(
    index='Class', columns='Source Dataset', values='Images', aggfunc='sum'
).reindex(CLASSES)

fig, ax = plt.subplots(figsize=(13, 7))
bottom = np.zeros(len(pivot))
colors = plt.cm.tab20(np.linspace(0, 1, len(pivot.columns)))
for color, source in zip(colors, pivot.columns):
    values = pivot[source].fillna(0).values
    ax.bar(pivot.index, values, bottom=bottom, label=source, color=color)
    bottom += values

ax.set_ylabel('Number of images')
ax.set_title('Combined Dataset Composition by Class and Source Dataset')
ax.set_xticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.index, rotation=45, ha='right')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.35), ncol=2, fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'dataset_provenance_by_source.png'), dpi=300, bbox_inches='tight')
plt.show()

# ---- Chart 2: per-class image count by dataset split (train / valid / test) ----
split_df = df[df['Class'] != 'TOTAL'].set_index('Class')[splits]
ax = split_df.plot(kind='bar', figsize=(13, 6), width=0.8)
ax.set_ylabel('Number of images')
ax.set_title('Per-Class Image Count by Dataset Split')
ax.set_xticklabels(split_df.index, rotation=45, ha='right')
plt.legend(title='Split')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'per_class_split_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Training details viewer

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from IPython.display import display, HTML

# Path setup
LOGS_DIR = '/content/drive/MyDrive/THESIS/10_final_run/training_logs'
MODELS = ['EfficientNetB0', 'MobileNetV2', 'ConvNeXtTiny', 'DenseNet121',
          'Xception', 'ResNet50V2', 'InceptionV3', 'EfficientNetB1',
          'DenseNet169', 'VGG16', 'ViT_B16', 'YOLO11m_cls']

print("="*80)
print("📊 TRAINING DETAILS VIEWER")
print("="*80)

# 1. SUMMARY TABLE - All models at a glance

print("\n" + "="*80)
print("1️⃣ SUMMARY TABLE (Final Epoch Performance)")
print("="*80)

summary_data = []

for model_name in MODELS:
    log_path = os.path.join(LOGS_DIR, f'{model_name}_training.csv')

    if os.path.exists(log_path):
        df = pd.read_csv(log_path)
        last_row = df.iloc[-1]

        # Find best validation accuracy
        best_val_acc = df['val_accuracy'].max()
        best_epoch = df[df['val_accuracy'] == best_val_acc]['epoch'].values[0]

        summary_data.append({
            'Model': model_name,
            'Epochs': len(df),
            'Final Train Acc': f"{last_row['accuracy']:.4f}",
            'Final Val Acc': f"{last_row['val_accuracy']:.4f}",
            'Best Val Acc': f"{best_val_acc:.4f}",
            'Best Epoch': best_epoch,
            'Final Train Loss': f"{last_row['loss']:.4f}",
            'Final Val Loss': f"{last_row['val_loss']:.4f}"
        })

summary_df = pd.DataFrame(summary_data)
display(HTML(summary_df.to_html(index=False)))

# Save summary
summary_df.to_csv(os.path.join(LOGS_DIR, 'training_summary.csv'), index=False)
print("\n✅ Summary saved to: training_logs/training_summary.csv")

# 2. EPOCH-WISE DETAILS - Pick a model to see all epochs

print("\n" + "="*80)
print("2️⃣ EPOCH-WISE DETAILS (Choose a model)")
print("="*80)

# You can change this to any model name
MODEL_TO_SHOW = 'ConvNeXtTiny'  # Change this to see another model

log_path = os.path.join(LOGS_DIR, f'{MODEL_TO_SHOW}_training.csv')

if os.path.exists(log_path):
    df = pd.read_csv(log_path)

    # Check which columns exist
    available_cols = list(df.columns)
    print(f"\n📁 Model: {MODEL_TO_SHOW}")
    print(f"📊 Total Epochs: {len(df)}")
    print(f"📈 Best Val Accuracy: {df['val_accuracy'].max():.4f} at Epoch {df['val_accuracy'].idxmax()}")
    print(f"📉 Best Val Loss: {df['val_loss'].min():.4f} at Epoch {df['val_loss'].idxmin()}")
    print(f"📋 Available Columns: {available_cols}")

    # Select columns to display (only those that exist)
    display_cols = ['epoch']
    for col in ['accuracy', 'val_accuracy', 'loss', 'val_loss']:
        if col in available_cols:
            display_cols.append(col)
    if 'lr' in available_cols:
        display_cols.append('lr')

    print("\n" + "-"*80)
    print("First 5 Epochs (Phase 1 - Head Training):")
    print("-"*80)
    display(df[display_cols].head(5))

    print("\n" + "-"*80)
    print("Last 5 Epochs (Final Phase):")
    print("-"*80)
    display(df[display_cols].tail(5))

    print("\n" + "-"*80)
    print("Top 5 Epochs by Validation Accuracy:")
    print("-"*80)
    display(df.nlargest(5, 'val_accuracy')[display_cols])

    # Show Phase 1 vs Phase 2 split (if epochs > 10)
    if len(df) > 10:
        phase1 = df[df['epoch'] < 10]
        phase2 = df[df['epoch'] >= 10]

        print("\n" + "-"*80)
        print("Phase 1 vs Phase 2 Comparison:")
        print("-"*80)
        comparison = pd.DataFrame({
            'Phase': ['Phase 1 (Head Only)', 'Phase 2 (Fine-tuning)'],
            'Epochs': [len(phase1), len(phase2)],
            'Start Val Acc': [phase1['val_accuracy'].iloc[0], phase2['val_accuracy'].iloc[0]],
            'End Val Acc': [phase1['val_accuracy'].iloc[-1], phase2['val_accuracy'].iloc[-1]],
            'Best Val Acc': [phase1['val_accuracy'].max(), phase2['val_accuracy'].max()]
        })
        display(HTML(comparison.to_html(index=False)))

# 3. TRAINING CURVES - Visualize any model

print("\n" + "="*80)
print("3️⃣ TRAINING CURVES (Visualization)")
print("="*80)

MODEL_TO_PLOT = 'ConvNeXtTiny'  # Change this to see another model

log_path = os.path.join(LOGS_DIR, f'{MODEL_TO_PLOT}_training.csv')

if os.path.exists(log_path):
    hist = pd.read_csv(log_path)
    x = np.arange(len(hist))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy plot
    axes[0].plot(x, hist['accuracy'], label='Train Accuracy', linewidth=2)
    axes[0].plot(x, hist['val_accuracy'], label='Validation Accuracy', linewidth=2)
    if len(hist) > 10:
        axes[0].axvline(x=9.5, color='red', linestyle='--', alpha=0.5, label='Phase 1 → Phase 2')
    axes[0].set_title(f'{MODEL_TO_PLOT} - Training & Validation Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss plot
    axes[1].plot(x, hist['loss'], label='Train Loss', linewidth=2)
    axes[1].plot(x, hist['val_loss'], label='Validation Loss', linewidth=2)
    if len(hist) > 10:
        axes[1].axvline(x=9.5, color='red', linestyle='--', alpha=0.5, label='Phase 1 → Phase 2')
    axes[1].set_title(f'{MODEL_TO_PLOT} - Training & Validation Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Save plot
    plt.savefig(os.path.join(LOGS_DIR, f'{MODEL_TO_PLOT}_training_curves.png'), dpi=200, bbox_inches='tight')
    print(f"✅ Plot saved to: training_logs/{MODEL_TO_PLOT}_training_curves.png")

# 4. COMPARE ALL MODELS - Bar Charts

print("\n" + "="*80)
print("4️⃣ MODEL COMPARISON CHARTS")
print("="*80)

if len(summary_data) > 0:
    comp_df = pd.DataFrame(summary_data)
    comp_df['Best Val Acc'] = comp_df['Best Val Acc'].astype(float)
    comp_df['Final Val Acc'] = comp_df['Final Val Acc'].astype(float)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Best Validation Accuracy
    sorted_df = comp_df.sort_values('Best Val Acc', ascending=True)
    colors = ['gold' if x == sorted_df['Best Val Acc'].max() else 'steelblue' for x in sorted_df['Best Val Acc']]
    axes[0].barh(sorted_df['Model'], sorted_df['Best Val Acc'], color=colors)
    axes[0].set_title('Best Validation Accuracy by Model')
    axes[0].set_xlabel('Accuracy')
    axes[0].set_xlim(0.85, 0.97)

    # Add value labels
    for i, v in enumerate(sorted_df['Best Val Acc']):
        axes[0].text(v + 0.002, i, f'{v:.2%}', va='center')

    # Final Validation Accuracy
    sorted_df = comp_df.sort_values('Final Val Acc', ascending=True)
    colors = ['gold' if x == sorted_df['Final Val Acc'].max() else 'steelblue' for x in sorted_df['Final Val Acc']]
    axes[1].barh(sorted_df['Model'], sorted_df['Final Val Acc'], color=colors)
    axes[1].set_title('Final Validation Accuracy by Model')
    axes[1].set_xlabel('Accuracy')
    axes[1].set_xlim(0.85, 0.97)

    for i, v in enumerate(sorted_df['Final Val Acc']):
        axes[1].text(v + 0.002, i, f'{v:.2%}', va='center')

    plt.tight_layout()
    plt.show()

    # Save chart
    plt.savefig(os.path.join(LOGS_DIR, 'model_comparison_chart.png'), dpi=200, bbox_inches='tight')
    print("✅ Comparison chart saved to: training_logs/model_comparison_chart.png")

# 5. ALL CSV FILES - List and Preview

print("\n" + "="*80)
print("5️⃣ ALL CSV FILES - Available Data")
print("="*80)

# List all CSV files
csv_files = [f for f in os.listdir(LOGS_DIR) if f.endswith('.csv')]

print("\n📁 Available CSV files:")
for i, f in enumerate(csv_files, 1):
    file_path = os.path.join(LOGS_DIR, f)
    size = os.path.getsize(file_path) / 1024  # KB
    print(f"   {i}. {f} ({size:.1f} KB)")

# Quick view of summary CSV
print("\n" + "-"*80)
print("📊 Training Summary Preview:")
print("-"*80)
if os.path.exists(os.path.join(LOGS_DIR, 'training_summary.csv')):
    summary_df = pd.read_csv(os.path.join(LOGS_DIR, 'training_summary.csv'))
    display(summary_df)

# 6. EXPORT OPTIONS

print("\n" + "="*80)
print("6️⃣ EXPORT OPTIONS")
print("="*80)

print("\n📥 To download any CSV file, run:")
print("   from google.colab import files")
print("   files.download('path/to/file.csv')")

print("\n📥 To download all CSV files as ZIP, run:")
print("   import zipfile")
print("   from google.colab import files")
print("   with zipfile.ZipFile('all_training_logs.zip', 'w') as zipf:")
print("       for f in os.listdir(LOGS_DIR):")
print("           if f.endswith('.csv'):")
print("               zipf.write(os.path.join(LOGS_DIR, f), f)")
print("   files.download('all_training_logs.zip')")

print("\n" + "="*80)
print("✅ ALL TRAINING DETAILS VIEWED SUCCESSFULLY!")
print("="*80)

In [ ]:
# Block 3: Load All 10 CNN Models
# AUTO-SKIP: model saved in Drive -> load directly
# NOT SAVED: train from scratch -> save to Drive
loaded_models  = {}
test_preds_all = {}
val_preds_all  = {}
model_acc      = {}

for model_name, (ModelClass, preprocessor) in MODELS_CONFIG.items():
    print(f"\n{'='*60}\nProcessing: {model_name}\n{'='*60}")

    model_path     = os.path.join(MODELS_DIR, f'{model_name}_final.keras')
    preds_path     = os.path.join(PREDS_DIR,  f'{model_name}_preds.npy')
    val_preds_path = os.path.join(PREDS_DIR,  f'{model_name}_val_preds.npy')
    log_path       = os.path.join(LOGS_DIR,   f'{model_name}_training.csv')

    # AUTO-SKIP
    if os.path.exists(model_path) and os.path.exists(preds_path) and os.path.exists(val_preds_path):
        print(f'{model_name} already trained. Loading...')
        model      = keras.models.load_model(model_path)
        test_preds = np.load(preds_path)
        val_preds  = np.load(val_preds_path)

    # TRAIN FROM SCRATCH
    else:
        print(f'{model_name} not found. Training...')
        inputs     = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
        x          = preprocessor(inputs)
        base_model = ModelClass(weights='imagenet', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), include_top=False)
        base_model.trainable = False
        x          = base_model(x, training=False)
        x          = GlobalAveragePooling2D()(x)
        x          = BatchNormalization()(x)
        x          = Dropout(0.4)(x)
        outputs    = Dense(nb_classes, activation='softmax', dtype='float32')(x)
        model      = keras.Model(inputs, outputs)

        checkpoint    = ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, verbose=1)
        reduce_lr     = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7, verbose=1)
        csv_logger    = CSVLogger(log_path, append=True)
        early_stop_p1 = EarlyStopping(monitor='val_loss', patience=4,  restore_best_weights=True)
        early_stop_p2 = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

        print('Phase 1: Training head...')
        model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=['accuracy']
        )
        model.fit(train_dataset_aug, epochs=10, validation_data=val_dataset,
                  class_weight=class_weight_dict,
                  callbacks=[checkpoint, reduce_lr, early_stop_p1, csv_logger])

        print('Phase 2: Fine-tuning...')
        base_model.trainable = True
        model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=5e-5, weight_decay=1e-5),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=['accuracy']
        )
        model.fit(train_dataset_aug, epochs=100, validation_data=val_dataset,
                  class_weight=class_weight_dict,
                  callbacks=[checkpoint, reduce_lr, early_stop_p2, csv_logger])

        model      = keras.models.load_model(model_path)
        test_preds = model.predict(test_dataset, verbose=0)
        val_preds  = model.predict(val_dataset,  verbose=0)
        np.save(preds_path,     test_preds)
        np.save(val_preds_path, val_preds)
        print(f'{model_name} saved to Drive.')

    acc = accuracy_score(y_test_true, test_preds.argmax(axis=1))
    print(f'[{model_name}] Accuracy: {acc*100:.2f}%')

    loaded_models[model_name]  = model
    test_preds_all[model_name] = test_preds
    val_preds_all[model_name]  = val_preds
    model_acc[model_name]      = acc

    try: del base_model
    except: pass
    gc.collect()

print(f'\n{len(loaded_models)}/10 models ready.')

In [ ]:
# Block 3-D: Training Curves (Accuracy and Loss vs. Training Step) for the CNNs
# Reads the per-model CSV logs written during training (Block 3, CSVLogger)
# and plots training/validation accuracy and loss curves. This works whether
# a model was trained in the current session or loaded from Drive in a prior
# session, as long as its training log was produced at least once. The
# x-axis is a running step index rather than the raw epoch column, because
# the two-phase (head, then fine-tune) training restarts epoch numbering at
# zero for the second phase.

cnn_model_names = list(MODELS_CONFIG.keys())
n_cols = 5
n_rows = int(np.ceil(len(cnn_model_names) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols * 2, figsize=(5 * n_cols, 4 * n_rows))
axes = np.array(axes).reshape(n_rows, n_cols * 2)

for idx, model_name in enumerate(cnn_model_names):
    row, col = divmod(idx, n_cols)
    ax_acc  = axes[row, col * 2]
    ax_loss = axes[row, col * 2 + 1]
    log_path = os.path.join(LOGS_DIR, f'{model_name}_training.csv')
    if not os.path.exists(log_path):
        ax_acc.set_title(f'{model_name}: log not found', fontsize=8)
        ax_acc.axis('off')
        ax_loss.axis('off')
        continue

    hist = pd.read_csv(log_path)
    x = np.arange(len(hist))

    ax_acc.plot(x, hist['accuracy'], label='Train')
    ax_acc.plot(x, hist['val_accuracy'], label='Validation')
    ax_acc.set_title(f'{model_name} -- Accuracy', fontsize=9)
    ax_acc.set_xlabel('Training step'); ax_acc.set_ylabel('Accuracy')
    ax_acc.legend(fontsize=7)

    ax_loss.plot(x, hist['loss'], label='Train')
    ax_loss.plot(x, hist['val_loss'], label='Validation')
    ax_loss.set_title(f'{model_name} -- Loss', fontsize=9)
    ax_loss.set_xlabel('Training step'); ax_loss.set_ylabel('Loss')
    ax_loss.legend(fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'cnn_training_curves.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Block 3-B: ViT-B16 (Vision Transformer)
print(f"\n{'='*60}\nProcessing: ViT_B16\n{'='*60}")

# Step 1: Install required packages
get_ipython().system('pip install -q validators')
get_ipython().system('pip install -q --no-deps vit-keras')

# Step 2: Imports
import os
import gc
import sys
import types
import numpy as np

# Step 3: Create fake TensorFlow Addons module
if 'tensorflow_addons' not in sys.modules:
    sys.modules['tensorflow_addons'] = types.ModuleType('tensorflow_addons')

import keras
from vit_keras import vit
import vit_keras.layers as vit_layers
from sklearn.metrics import accuracy_score
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, CSVLogger

# Step 4: Keras 3 Monkey Patches
# --- Fix Layer Names (replace '/' with '_' in names) ---
old_init = keras.layers.Layer.__init__
def new_init(self, *args, **kwargs):
    if 'name' in kwargs and isinstance(kwargs['name'], str):
        kwargs['name'] = kwargs['name'].replace('/', '_')
    old_init(self, *args, **kwargs)

# --- Fix Training Argument (add training=None) ---
def make_new_call(old_func):
    def new_call(self, inputs, training=None, **kwargs):
        return old_func(self, inputs)
    return new_call

vit_layers.ClassToken.call = make_new_call(vit_layers.ClassToken.call)
vit_layers.AddPositionEmbs.call = make_new_call(vit_layers.AddPositionEmbs.call)
vit_layers.MultiHeadSelfAttention.call = make_new_call(vit_layers.MultiHeadSelfAttention.call)

old_tb_call = vit_layers.TransformerBlock.call
def new_tb_call(self, inputs, training=None, **kwargs):
    return old_tb_call(self, inputs, training=training if training is not None else False)
vit_layers.TransformerBlock.call = new_tb_call

# Step 5: Model Training
model_name_vit      = 'ViT_B16'
vit_model_path      = os.path.join(MODELS_DIR, f'{model_name_vit}_final.keras')
vit_preds_path      = os.path.join(PREDS_DIR,  f'{model_name_vit}_preds.npy')
vit_val_preds_path  = os.path.join(PREDS_DIR,  f'{model_name_vit}_val_preds.npy')
vit_log_path        = os.path.join(LOGS_DIR,   f'{model_name_vit}_training.csv')

try:
    if os.path.exists(vit_model_path) and os.path.exists(vit_preds_path) and os.path.exists(vit_val_preds_path):
        print(f'{model_name_vit} already trained. Loading...')
        vit_full_model = keras.models.load_model(vit_model_path)
        vit_test_preds = np.load(vit_preds_path)
        vit_val_preds  = np.load(vit_val_preds_path)
    else:
        print(f'{model_name_vit} not found. Training...')

        # Enable name-fix patch to prevent naming errors
        keras.layers.Layer.__init__ = new_init

        vit_base = vit.vit_b16(
            image_size=IMG_SIZE[0],
            pretrained=True,
            include_top=False,
            pretrained_top=False,
        )
        vit_base.trainable = False

        # Disable name-fix patch after model creation
        keras.layers.Layer.__init__ = old_init

        inputs  = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
        x       = keras.layers.Rescaling(scale=1./127.5, offset=-1)(inputs)
        x       = vit_base(x)
        x       = keras.layers.BatchNormalization()(x)
        x       = keras.layers.Dropout(0.4)(x)
        outputs = keras.layers.Dense(nb_classes, activation='softmax', dtype='float32')(x)
        vit_full_model = keras.Model(inputs, outputs)

        checkpoint    = ModelCheckpoint(vit_model_path, monitor='val_accuracy', save_best_only=True, verbose=1)
        reduce_lr     = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7, verbose=1)
        csv_logger    = CSVLogger(vit_log_path, append=True)
        early_stop_p1 = EarlyStopping(monitor='val_loss', patience=4,  restore_best_weights=True)
        early_stop_p2 = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

        print('Phase 1: Training head...')
        vit_full_model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=['accuracy']
        )
        vit_full_model.fit(train_dataset_aug, epochs=10, validation_data=val_dataset,
                            class_weight=class_weight_dict,
                            callbacks=[checkpoint, reduce_lr, early_stop_p1, csv_logger])

        print('Phase 2: Fine-tuning...')
        vit_base.trainable = True
        vit_full_model.compile(
            optimizer=keras.optimizers.AdamW(learning_rate=5e-5, weight_decay=1e-5),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=['accuracy']
        )
        vit_full_model.fit(train_dataset_aug, epochs=100, validation_data=val_dataset,
                            class_weight=class_weight_dict,
                            callbacks=[checkpoint, reduce_lr, early_stop_p2, csv_logger])

        vit_full_model = keras.models.load_model(vit_model_path)
        vit_test_preds = vit_full_model.predict(test_dataset, verbose=0)
        vit_val_preds  = vit_full_model.predict(val_dataset,  verbose=0)
        np.save(vit_preds_path,     vit_test_preds)
        np.save(vit_val_preds_path, vit_val_preds)
        print(f'{model_name_vit} saved to Drive.')

    acc_vit = accuracy_score(y_test_true, vit_test_preds.argmax(axis=1))
    print(f'[{model_name_vit}] Accuracy: {acc_vit*100:.2f}%')

    loaded_models[model_name_vit]  = vit_full_model
    test_preds_all[model_name_vit] = vit_test_preds
    val_preds_all[model_name_vit]  = vit_val_preds
    model_acc[model_name_vit]      = acc_vit

    try: del vit_base
    except: pass

except Exception as e:
    import traceback
    traceback.print_exc()
    print(f' ViT_B16 failed, skipping this model. Error: {e}')

import gc
gc.collect()
print(f'\nViT block done. Total models ready: {len(loaded_models)}')

In [ ]:
# Block 3-E: ViT-B16 Training Curve (Accuracy and Loss vs. Training Step)
if os.path.exists(vit_log_path):
    vit_hist = pd.read_csv(vit_log_path)
    x = np.arange(len(vit_hist))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(x, vit_hist['accuracy'], label='Train')
    axes[0].plot(x, vit_hist['val_accuracy'], label='Validation')
    axes[0].set_title('ViT_B16 -- Accuracy')
    axes[0].set_xlabel('Training step'); axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(x, vit_hist['loss'], label='Train')
    axes[1].plot(x, vit_hist['val_loss'], label='Validation')
    axes[1].set_title('ViT_B16 -- Loss')
    axes[1].set_xlabel('Training step'); axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'vit_training_curve.png'), dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('ViT_B16 training log not found; skipping the training-curve plot.')

In [ ]:
# Block 3-C: YOLO11m-cls (classification variant, NOT object detection)
# AUTO-SKIP: weights saved in Drive -> load directly
# NOT SAVED: train from scratch -> save to Drive
print(f"\n{'='*60}\nProcessing: YOLO11m_cls\n{'='*60}")

import os
import glob
import shutil
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score
import gc

# Auto-install ultralytics if missing
try:
    from ultralytics import YOLO
except ImportError:
    import subprocess
    import sys
    print("Installing ultralytics...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
    from ultralytics import YOLO

model_name_yolo      = 'YOLO11m_cls'
yolo_weights_dir     = os.path.join(MODELS_DIR, model_name_yolo)
yolo_best_path       = os.path.join(yolo_weights_dir, 'best.pt')
yolo_preds_path      = os.path.join(PREDS_DIR, f'{model_name_yolo}_preds.npy')
yolo_val_preds_path  = os.path.join(PREDS_DIR, f'{model_name_yolo}_val_preds.npy')
os.makedirs(yolo_weights_dir, exist_ok=True)

try:
    # AUTO-SKIP
    if os.path.exists(yolo_best_path):
        print(f'{model_name_yolo} already trained. Loading...')
        yolo_model = YOLO(yolo_best_path)
    # TRAIN FROM SCRATCH
    else:
        print(f'{model_name_yolo} not found. Training...')
        yolo_model = YOLO('yolo11m-cls.pt')
        train_results = yolo_model.train(
            data=dataset_final_path,   # expects train/ and val/ subfolders -- already present
            epochs=100, imgsz=IMG_SIZE[0], batch=32, device='cuda'
        )
        # copy best.pt from the Ultralytics run folder into our Drive structure
        best_src = os.path.join(train_results.save_dir, 'weights', 'best.pt')
        shutil.copy(best_src, yolo_best_path)
        # Persist the Ultralytics training log so training curves can be
        # plotted in Block 3-F even in a later session.
        yolo_results_csv = os.path.join(train_results.save_dir, 'results.csv')
        if os.path.exists(yolo_results_csv):
            shutil.copy(yolo_results_csv, os.path.join(LOGS_DIR, 'YOLO11m_cls_training.csv'))
        yolo_model = YOLO(yolo_best_path)
        print(f'{model_name_yolo} saved to Drive.')

    # YOLO's internal class order (model.names) may NOT match class_names order,
    # so every prediction is explicitly remapped back into class_names order.
    yolo_idx_to_name  = yolo_model.names                      # e.g. {0: 'Acne', 1: 'Chickenpox', ...}
    name_to_keras_idx = {name: i for i, name in enumerate(class_names)}
    reorder           = [name_to_keras_idx[yolo_idx_to_name[i]] for i in range(len(yolo_idx_to_name))]

    def yolo_predict_split(split_name, expected_len):
        """Run YOLO11m-cls over a split folder, in the EXACT SAME file order Keras used."""
        split_path = os.path.join(dataset_final_path, split_name)

        # Use Keras to fetch the exact same files it uses in Block 2
        print(f"Fetching file paths for {split_name} split using Keras dataset...")
        tmp_ds = tf.keras.utils.image_dataset_from_directory(
            split_path,
            shuffle=False,
            batch_size=1,
            image_size=IMG_SIZE,
            label_mode='categorical'
        )
        imgs = tmp_ds.file_paths

        probs_list = []
        for img_path in imgs:
            res = yolo_model(img_path, verbose=False)[0]
            raw_probs = res.probs.data.cpu().numpy()       # YOLO's own class order
            fixed_probs = np.zeros_like(raw_probs)
            fixed_probs[reorder] = raw_probs               # remap into class_names order
            probs_list.append(fixed_probs)

        arr = np.array(probs_list)
        assert arr.shape[0] == expected_len, (
            f'Mismatch in {split_name}: got {arr.shape[0]} predictions, '
            f'expected {expected_len}. Check that Keras and YOLO see the same files.'
        )
        return arr

    if os.path.exists(yolo_preds_path) and os.path.exists(yolo_val_preds_path):
        print('YOLO predictions already saved. Loading...')
        yolo_test_preds = np.load(yolo_preds_path)
        yolo_val_preds  = np.load(yolo_val_preds_path)
    else:
        print('Running YOLO inference on test set...')
        yolo_test_preds = yolo_predict_split('test', len(y_test_true))
        print('Running YOLO inference on val set...')
        yolo_val_preds  = yolo_predict_split('val',  len(y_val_true))
        np.save(yolo_preds_path,     yolo_test_preds)
        np.save(yolo_val_preds_path, yolo_val_preds)
        print('YOLO predictions saved to Drive.')

    acc_yolo = accuracy_score(y_test_true, yolo_test_preds.argmax(axis=1))
    print(f'[{model_name_yolo}] Accuracy: {acc_yolo*100:.2f}%')

    loaded_models[model_name_yolo]  = yolo_model
    test_preds_all[model_name_yolo] = yolo_test_preds
    val_preds_all[model_name_yolo]  = yolo_val_preds
    model_acc[model_name_yolo]      = acc_yolo

except Exception as e:
    print(f' YOLO11m_cls failed, skipping this model. Error: {e}')

gc.collect()
print(f'\nYOLO block done. Total models ready: {len(loaded_models)}')

---
## PART 1: Per-Class Model Analysis
Which model performs best for which disease?


In [ ]:
# Block 4: Per-Class F1 Analysis
from sklearn.metrics import f1_score
import pandas as pd

print(f"\n{'='*60}\nGenerating Per-Class F1 Scores for ALL models...\n{'='*60}")

# Forcefully calculate again (bypassing the AUTO-SKIP check)
f1_csv_path = os.path.join(RESULTS_DIR, 'per_class_f1_scores.csv')

# Calculate F1 scores for each model, for each class
records = []
for m_name, preds in val_preds_all.items():
    pred_classes = preds.argmax(axis=1)
    # Get F1 score per class (returns array of shape (nb_classes,))
    f1_per_class = f1_score(y_val_true, pred_classes, average=None)

    for i, cls_name in enumerate(class_names):
        records.append({
            'Model': m_name,
            'Disease': cls_name,
            'F1_Score': f1_per_class[i]
        })

df_f1 = pd.DataFrame(records)

# Find the best model for each disease
best_models_per_disease = df_f1.loc[df_f1.groupby('Disease')['F1_Score'].idxmax()]
best_models_per_disease = best_models_per_disease.reset_index(drop=True)

print("\n--- BEST MODEL PER DISEASE ---")
print(best_models_per_disease.to_string(index=False))

# Overwrite the old CSV file with the new one
df_f1.to_csv(f1_csv_path, index=False)
print('\nSaved new results to Drive.')

In [ ]:
# Block 5: Per-class F1 heatmap chart
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
import numpy as np

heatmap_path = os.path.join(PERCLASS_DIR, 'perclass_f1_heatmap.png')

# Create pivot table for heatmap using df_f1 from Block 4
perclass_df = df_f1.pivot(index='Disease', columns='Model', values='F1_Score')
best_model_per_class = perclass_df.idxmax(axis=1)
best_f1_per_class = perclass_df.max(axis=1)

print('Generating new heatmap for all 12 models...')
fig, axes = plt.subplots(1, 2, figsize=(22, 7))

# Plot 1: Heatmap
sns.heatmap(
    perclass_df, annot=True, fmt='.2f', cmap='YlOrRd',
    ax=axes[0], cbar=True, annot_kws={'size': 8}
)
axes[0].set_title('Per-Class F1 Score — All Models vs All Diseases', fontsize=12)
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', labelsize=9)

# Plot 2: Bar chart
unique_models  = list(dict.fromkeys(best_model_per_class.values))
color_palette  = plt.cm.Set3(np.linspace(0, 1, len(unique_models)))
model_color    = {m: color_palette[i] for i, m in enumerate(unique_models)}
bar_colors     = [model_color.get(best_model_per_class[c], 'gray') for c in class_names]

bars = axes[1].barh(class_names, best_f1_per_class.values, color=bar_colors)
axes[1].set_xlabel('Best F1 Score')
axes[1].set_title('Best Model Per Disease', fontsize=12)
axes[1].set_xlim(0, 1.15)
for i, cls in enumerate(class_names):
    axes[1].text(
        best_f1_per_class[cls] + 0.01, i,
        f'{best_model_per_class[cls]} ({best_f1_per_class[cls]:.2f})',
        va='center', fontsize=8
    )

plt.tight_layout()
plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'New Heatmap saved -> {heatmap_path}')

---
## PART 2: Stacking Ensemble

In [ ]:
# Block 6: Stacking ensemble meta-classifier

# The TOP-3 models are selected automatically by validation accuracy across
# all trained models (10 CNN + ViT + YOLO); the list is not hardcoded.
# Dynamic Top-3 -- chosen from VALIDATION accuracy so the test set stays
# fully held out from model selection (matches the methodology).
model_val_acc = {
    name: accuracy_score(y_val_true, preds.argmax(axis=1))
    for name, preds in val_preds_all.items()
}
TOP3 = [name for name, _ in sorted(model_val_acc.items(), key=lambda x: x[1], reverse=True)[:3]]
print('Dynamically selected Top-3 models (by validation accuracy):')
for _name in TOP3:
    print(f' {_name}: val {model_val_acc[_name]*100:.2f}%  |  test {model_acc[_name]*100:.2f}%')
print()

meta_path      = os.path.join(MODELS_DIR, 'meta_classifier.pkl')
top3_available = [m for m in TOP3 if m in loaded_models]

X_val_meta  = np.concatenate([val_preds_all[m]  for m in top3_available], axis=1)
X_test_meta = np.concatenate([test_preds_all[m] for m in top3_available], axis=1)

if False:
    print('Meta-classifier already trained. Loading...')
    with open(meta_path, 'rb') as f:
        meta_model = pickle.load(f)

# TRAIN
else:
    print('Training meta-classifier...')
    meta_model = LogisticRegression(max_iter=2000, C=0.1, class_weight='balanced')
    meta_model.fit(X_val_meta, y_val_true)
    with open(meta_path, 'wb') as f:
        pickle.dump(meta_model, f)
    print(f'Meta-classifier saved -> {meta_path}')

ensemble_preds = meta_model.predict(X_test_meta)
ensemble_acc   = accuracy_score(y_test_true, ensemble_preds)

print('\n' + '='*55)
print('MODEL LEADERBOARD')
print('='*55)
for name, acc in sorted(model_acc.items(), key=lambda x: x[1], reverse=True):
    tag = ' <- top 3' if name in TOP3 else ''
    print(f' {name:<22} {acc*100:.2f}%{tag}')
print(f' {"-"*40}')
print(f' {"Stacking Ensemble":<22} {ensemble_acc*100:.2f}%  <- best')
print('='*55)

print('\n--- ENSEMBLE CLASSWISE PERFORMANCE ---')
print(classification_report(y_test_true, ensemble_preds, target_names=class_names))

pd.DataFrame(
    classification_report(y_test_true, ensemble_preds, target_names=class_names, output_dict=True)
).transpose().to_csv(os.path.join(LOGS_DIR, 'ensemble_classification_report.csv'))

# Confusion matrix
cm = confusion_matrix(y_test_true, ensemble_preds)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Stacking Ensemble — Confusion Matrix', fontsize=14)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'ensemble_confusion_matrix.png'), dpi=300)
plt.show()
print('Ensemble results saved.')

In [ ]:
# Block 6-A2: ROC Curves and AUC (One-vs-Rest, per class + micro/macro average)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_bin     = label_binarize(y_test_true, classes=list(range(nb_classes)))
ensemble_proba = meta_model.predict_proba(X_test_meta)

fpr, tpr, roc_auc = {}, {}, {}
for i in range(nb_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], ensemble_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr['micro'], tpr['micro'], _ = roc_curve(y_test_bin.ravel(), ensemble_proba.ravel())
roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

all_fpr  = np.unique(np.concatenate([fpr[i] for i in range(nb_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(nb_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= nb_classes
fpr['macro'], tpr['macro'] = all_fpr, mean_tpr
roc_auc['macro'] = auc(fpr['macro'], tpr['macro'])

plt.figure(figsize=(9, 8))
colors = plt.cm.tab10(np.linspace(0, 1, nb_classes))
for i, color in zip(range(nb_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=1.3,
             label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')

plt.plot(fpr['micro'], tpr['micro'], color='deeppink', linestyle=':', lw=2.5,
         label=f'Micro-average (AUC = {roc_auc["micro"]:.3f})')
plt.plot(fpr['macro'], tpr['macro'], color='navy', linestyle=':', lw=2.5,
         label=f'Macro-average (AUC = {roc_auc["macro"]:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')

plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('Stacking Ensemble -- ROC Curves (One-vs-Rest)')
plt.legend(loc='lower right', fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'ensemble_roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

roc_auc_df = pd.DataFrame({
    'Class': class_names + ['Micro-average', 'Macro-average'],
    'AUC': [roc_auc[i] for i in range(nb_classes)] + [roc_auc['micro'], roc_auc['macro']]
})
roc_auc_df.to_csv(os.path.join(LOGS_DIR, 'ensemble_roc_auc.csv'), index=False)
print(roc_auc_df.to_string(index=False))

In [ ]:
# Block 6-B: Temperature Scaling for Ensemble Calibration
# Post-hoc calibration via temperature scaling (Guo et al., 2017).
# NOTE: the C=0.1 regularized meta-learner is UNDER-confident, so the optimal
# temperature is T < 1, which SHARPENS the posteriors toward accuracy (lowers ECE).

import numpy as np
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score
from scipy.optimize import minimize_scalar

def temperature_scale(logits, T):
    """Apply temperature scaling to logits."""
    scaled = logits / T
    exp_scaled = np.exp(scaled - np.max(scaled, axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def ece_loss(T, logits, labels, n_bins=15):
    """ECE as a function of temperature, for optimization."""
    probs = temperature_scale(logits, T)
    confs = np.max(probs, axis=1)
    preds = np.argmax(probs, axis=1)
    accs  = (preds == labels).astype(float)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confs > bin_boundaries[i]) & (confs <= bin_boundaries[i+1])
        if mask.sum() == 0: continue
        ece += (mask.sum() / len(labels)) * abs(accs[mask].mean() - confs[mask].mean())
    return ece

# Get raw logits from meta-classifier (before softmax)
X_val_meta  = np.concatenate([val_preds_all[m] for m in TOP3], axis=1)
X_test_meta = np.concatenate([test_preds_all[m] for m in TOP3], axis=1)
val_logits  = meta_model.decision_function(X_val_meta)
test_logits = meta_model.decision_function(X_test_meta)

# Find optimal temperature on validation set
result = minimize_scalar(ece_loss, bounds=(0.1, 10.0), method='bounded',
                         args=(val_logits, y_val_true))
optimal_T = result.x

# Apply to test set
calibrated_probs = temperature_scale(test_logits, optimal_T)
calibrated_preds = np.argmax(calibrated_probs, axis=1)
calibrated_acc   = accuracy_score(y_test_true, calibrated_preds)

# ECE before vs after
from sklearn.metrics import accuracy_score
ece_before = ece_loss(1.0, test_logits, y_test_true)
ece_after  = ece_loss(optimal_T, test_logits, y_test_true)

print(f'Optimal Temperature: {optimal_T:.3f}')
print(f'ECE Before: {ece_before:.4f}')
print(f'ECE After:  {ece_after:.4f}  (↓ {(ece_before-ece_after)/ece_before*100:.1f}%)')
print(f'Accuracy:   {calibrated_acc*100:.2f}% (was {ensemble_acc*100:.2f}%)')

In [ ]:
# Block 6-D: Reliability Diagram (Calibration Curve, Before vs. After Scaling)
raw_probs = temperature_scale(test_logits, 1.0)

def reliability_bins(probs, labels, n_bins=15):
    confs = np.max(probs, axis=1)
    preds = np.argmax(probs, axis=1)
    accs  = (preds == labels).astype(float)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_centers, bin_acc, bin_conf = [], [], []
    for i in range(n_bins):
        mask = (confs > bin_boundaries[i]) & (confs <= bin_boundaries[i + 1])
        if mask.sum() == 0:
            continue
        bin_centers.append((bin_boundaries[i] + bin_boundaries[i + 1]) / 2)
        bin_acc.append(accs[mask].mean())
        bin_conf.append(confs[mask].mean())
    return np.array(bin_centers), np.array(bin_acc), np.array(bin_conf)

centers_raw, acc_raw, conf_raw = reliability_bins(raw_probs, y_test_true)
centers_cal, acc_cal, conf_cal = reliability_bins(calibrated_probs, y_test_true)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
panels = [
    (axes[0], centers_raw, acc_raw, f'Before Calibration (ECE = {ece_before:.4f})'),
    (axes[1], centers_cal, acc_cal, f'After Calibration, T = {optimal_T:.3f} (ECE = {ece_after:.4f})'),
]
for ax, centers, accs, title in panels:
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.bar(centers, accs, width=1 / 15, edgecolor='black', alpha=0.75, label='Observed accuracy')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'ensemble_reliability_diagram.png'), dpi=300, bbox_inches='tight')
plt.show()

---
---
### Calibration Finding

The stacked ensemble was the most accurate model (96.94%) but, before calibration, the most poorly calibrated: its Expected Calibration Error (0.2086) exceeded that of every individual base model (ECE 0.042 to 0.107). This miscalibration is a result of under-confidence rather than over-confidence: the L2-regularized (C = 0.1), class-balanced logistic-regression meta-learner shrinks its logits and produces hedged posteriors whose mean confidence falls well below the ensemble's 96.94% accuracy. Temperature scaling with T = 0.54 (< 1), fit on the validation set, sharpens these posteriors toward the accuracy level, reducing ensemble ECE to 0.0143, a 93.1% reduction, with no change in accuracy. All ensemble confidences reported by the final two-stage predictor use these calibrated probabilities.

This result evaluates trustworthiness in addition to accuracy, which is consistent with the evaluation gap identified in the accompanying literature review. Note the direction of the effect: T < 1 increases confidence, so the meta-learner was under-confident, the opposite of the over-confidence pattern that Guo et al. (2017) describe for deep networks directly. The reliability diagram below (Block 6-D) visualizes this before/after effect.


In [ ]:
# Block 6-C: Statistical significance -- McNemar test (ensemble vs best single model)
# Confirms whether the ensemble's gain over the top single model is real, not noise.
try:
    from statsmodels.stats.contingency_tables import mcnemar
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    from statsmodels.stats.contingency_tables import mcnemar

best_single  = max(model_acc, key=model_acc.get)         # best single model by test accuracy
ens_correct  = (ensemble_preds == y_test_true)
base_correct = (test_preds_all[best_single].argmax(axis=1) == y_test_true)

n11 = int(np.sum( ens_correct &  base_correct))
n10 = int(np.sum( ens_correct & ~base_correct))   # ensemble right, base wrong
n01 = int(np.sum(~ens_correct &  base_correct))   # base right, ensemble wrong
n00 = int(np.sum(~ens_correct & ~base_correct))
table = [[n11, n10], [n01, n00]]

res = mcnemar(table, exact=True)   # exact binomial (fine for small discordant counts)
print('='*60)
print(' McNemar test: Stacking Ensemble vs best single model')
print('='*60)
print(f' Ensemble accuracy       : {ensemble_acc*100:.2f}%')
print(f' {best_single} accuracy   : {model_acc[best_single]*100:.2f}%')
print(f' Ensemble-only-correct   : {n10}')
print(f' {best_single}-only-correct : {n01}')
print(f' McNemar p-value         : {res.pvalue:.4f}')
if res.pvalue < 0.05:
    print(' => Ensemble improvement IS statistically significant (p < 0.05).')
else:
    print(' => Improvement is NOT significant at 0.05 -- report this honestly;')
    print(' with only ~720 test images a ~1% gap can be within noise.')
print('='*60)

# Save for the paper
import csv
with open(os.path.join(RESULTS_DIR, 'mcnemar_ensemble_vs_best.csv'), 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['best_single', 'ensemble_acc', 'best_acc', 'n_ens_only', 'n_base_only', 'p_value'])
    w.writerow([best_single, ensemble_acc, model_acc[best_single], n10, n01, res.pvalue])
print('Saved -> mcnemar_ensemble_vs_best.csv')

---
## PART 3: Grad-CAM — Where Does the Model Look?
Red = high attention | Yellow = moderate | Blue = low attention


In [ ]:
# Block 7: Grad-CAM Helper Functions (+ Occlusion Sensitivity for ViT/YOLO, see 7-B)
# Works for nested models (ConvNeXtTiny, Xception, EfficientNet etc.)
import os
import urllib.request
from google.colab import files
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
from PIL import Image

def get_gradcam_heatmap(model, img_tensor, model_name=None):
    """
    Super safe Grad-CAM that explicitly skips InputLayers and handles
    all functional rewiring smoothly.
    """
    base_model = None
    last_feature_layer = None
    base_model_idx = -1

    for i, layer in enumerate(model.layers):
        if isinstance(layer, tf.keras.Model) or (hasattr(layer, 'layers') and len(getattr(layer, 'layers', [])) > 0):
            base_model = layer
            base_model_idx = i
            break

    if base_model is None: return None, None

    for sub_layer in reversed(base_model.layers):
        try:
            shape = sub_layer.output_shape
            if isinstance(shape, list): shape = shape[0]
            if len(shape) == 4 and shape[1] is not None and shape[1] > 1:
                last_feature_layer = sub_layer
                break
        except: pass

    if last_feature_layer is None: return None, None

    try:
        inner_grad_model = tf.keras.Model(
            inputs  = base_model.input,
            outputs = [last_feature_layer.output, base_model.output]
        )
    except: return None, None

    x = img_tensor
    # Skip InputLayer to avoid TypeError
    for layer in model.layers[:base_model_idx]:
        if layer.__class__.__name__ == 'InputLayer':
            continue
        try:
            x = layer(x, training=False)
        except TypeError:
            x = layer(x)

    with tf.GradientTape() as tape:
        feature_maps, base_out = inner_grad_model(x, training=False)
        tape.watch(feature_maps)

        top_x = base_out
        for layer in model.layers[base_model_idx+1:]:
            try:
                top_x = layer(top_x, training=False)
            except TypeError:
                top_x = layer(top_x)

        final_preds = top_x
        pred_idx    = tf.argmax(final_preds[0])
        cls_score   = final_preds[:, pred_idx]

    grads = tape.gradient(cls_score, feature_maps)
    if grads is None: return None, None

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap      = feature_maps[0] @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0)
    max_val      = tf.reduce_max(heatmap)
    if max_val > 0: heatmap = heatmap / max_val

    return heatmap.numpy(), int(pred_idx)

def apply_heatmap_overlay(img_path, heatmap, alpha=0.45):
    """Overlay Grad-CAM heatmap on original image."""
    img             = np.array(Image.open(img_path).convert('RGB').resize(IMG_SIZE))
    heatmap_resized = cv2.resize(heatmap.astype(np.float32), IMG_SIZE)
    heatmap_uint8   = np.uint8(255 * heatmap_resized)
    heatmap_color   = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color   = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay         = np.uint8(img * (1 - alpha) + heatmap_color * alpha)
    return overlay

def get_bbox_from_heatmap(heatmap, img_size, threshold=0.5):
    """Extract bounding box from Grad-CAM heatmap."""
    heatmap_resized = cv2.resize(heatmap.astype(np.float32), img_size)
    binary          = (heatmap_resized > threshold).astype(np.uint8)
    contours, _     = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        binary      = (heatmap_resized > 0.3).astype(np.uint8)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        largest    = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)
        return x, y, x+w, y+h

    return None

# Block 7-B: Universal (Architecture-Agnostic) Heatmap for ViT / YOLO
# Grad-CAM above needs a spatial conv feature map, which ViT_B16 (patch
# embedding + transformer, no conv map) and YOLO11m_cls (different
# framework -- Ultralytics/PyTorch, not Keras) don't have. So instead of
# skipping them, we use Occlusion Sensitivity Mapping (Zeiler & Fergus,
# 2014): systematically grey-out small regions of the image and measure
# how much the predicted-class confidence drops. Big drop = model was
# relying on that region = heatmap. Works identically for ANY classifier
# (CNN, ViT, YOLO, ...) since it only calls predict(), no internal layer
# access needed -- so all 12 models get a real, comparable heatmap.

def make_predict_fn(model_name, model):
    """
    Returns a function: (H,W,3) uint8 RGB numpy image -> 1D probability
    vector over CLASSES, regardless of whether the model is a Keras CNN/ViT
    model or an Ultralytics YOLO classifier.
    """
    if model_name == 'YOLO11m_cls':
        idx_to_name = model.names
        name_to_idx = {c: i for i, c in enumerate(class_names)}
        def fn(img_array):
            res       = model(img_array, verbose=False)[0]
            raw_probs = res.probs.data.cpu().numpy()
            fixed     = np.zeros_like(raw_probs)
            for yolo_i, cname in idx_to_name.items():
                fixed[name_to_idx[cname]] = raw_probs[yolo_i]
            return fixed
        return fn
    else:
        def fn(img_array):
            arr = np.expand_dims(img_array.astype(np.float32), axis=0)
            return model.predict(arr, verbose=0)[0]
        return fn

def get_occlusion_heatmap(predict_fn, img_path, target_idx=None, grid_size=8, occlusion_value=127):
    """
    Model-agnostic explainability heatmap via systematic occlusion.
    Returns (heatmap, target_idx). heatmap shape = (grid_size, grid_size);
    apply_heatmap_overlay() / get_bbox_from_heatmap() already cv2.resize()
    any-shape heatmap to IMG_SIZE, so this is a drop-in replacement for the
    Grad-CAM heatmap wherever it's used (Block 8, 9, 10).
    """
    orig_img = np.array(Image.open(img_path).convert('RGB').resize(IMG_SIZE))
    H, W, _  = orig_img.shape

    base_probs = predict_fn(orig_img)
    if target_idx is None:
        target_idx = int(np.argmax(base_probs))
    base_score = base_probs[target_idx]

    step_h, step_w = H // grid_size, W // grid_size
    heatmap = np.zeros((grid_size, grid_size), dtype=np.float32)

    for i in range(grid_size):
        for j in range(grid_size):
            occluded = orig_img.copy()
            y1, y2 = i * step_h, min((i + 1) * step_h, H)
            x1, x2 = j * step_w, min((j + 1) * step_w, W)
            occluded[y1:y2, x1:x2, :] = occlusion_value
            probs = predict_fn(occluded)
            drop  = base_score - probs[target_idx]
            heatmap[i, j] = max(drop, 0)

    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()

    return heatmap, target_idx

# Models with no spatial conv feature map -- these get Occlusion Sensitivity
# instead of Grad-CAM. All 10 CNN models keep using the sharper Grad-CAM above.
NON_CONV_MODELS = ('ViT_B16', 'YOLO11m_cls')

def get_heatmap_for_model(model_name, model, img_path, img_arr=None):
    """
    Single dispatch point used everywhere (Block 8, 9, 10): Grad-CAM for
    CNNs, Occlusion Sensitivity for ViT/YOLO. Returns (heatmap, pred_idx).
    Falls back to Occlusion if Grad-CAM produces a near-zero heatmap.
    """
    if model_name in NON_CONV_MODELS:
        predict_fn = make_predict_fn(model_name, model)
        return get_occlusion_heatmap(predict_fn, img_path)
    else:
        heatmap, pred_idx = get_gradcam_heatmap(model, img_arr, model_name=model_name)
        if heatmap is None or np.max(heatmap) < 0.05:
            predict_fn = make_predict_fn(model_name, model)
            return get_occlusion_heatmap(predict_fn, img_path)
        return heatmap, pred_idx

print('Universal heatmap (Grad-CAM + Occlusion) functions ready.')

# Block 7-C: Complete Publication-Ready Explainability Pipeline
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from datetime import datetime
import tensorflow as tf

# CONFIGURATION
# Set to True to generate new heatmaps, False to load the latest existing results
NEW_GENERATION = False  # <-- CHANGE THIS TO False TO LOAD LATEST RESULTS

# Ensure directories exist
GRADCAM_DIR = '/content/drive/MyDrive/THESIS/10_final_run/gradcam_results'
os.makedirs(GRADCAM_DIR, exist_ok=True)

# The Top-3 models should already be defined in your notebook as TOP3
# If not, you can define them here:
# TOP3 = ['ConvNeXtTiny', 'ViT_B16', 'DenseNet169']

# PHASE 1: IMAGE SELECTION & METADATA MANAGEMENT

def load_or_select_images(generate_new=False):
    """
    Load pre-selected images or select new ones.
    This ensures the SAME images are used for all models.
    """
    selected_images_file = os.path.join(GRADCAM_DIR, 'selected_images.json')

    if not generate_new and os.path.exists(selected_images_file):
        print("[OK] Loading existing image selection...")
        with open(selected_images_file, 'r') as f:
            selected_images = json.load(f)
        return selected_images

    print("[OK] Selecting new images (one per class)...")
    selected_images = {}

    for class_name in class_names:
        class_path = os.path.join(dataset_final_path, 'test', class_name)
        if os.path.isdir(class_path):
            # Get all image files in the class directory
            image_files = [f for f in os.listdir(class_path)
                          if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]
            if image_files:
                # Sort for deterministic selection
                image_files.sort()
                # Select the first image (or you could use a specific index)
                selected_images[class_name] = {
                    'file_name': image_files[0],
                    'file_path': os.path.join(class_path, image_files[0])
                }

    # Save the current selection
    with open(selected_images_file, 'w') as f:
        json.dump(selected_images, f, indent=4)

    # Also save a timestamped history file
    history_file = os.path.join(GRADCAM_DIR,
                                f'selected_images_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json')
    with open(history_file, 'w') as f:
        json.dump(selected_images, f, indent=4)

    print(f"[OK] Selected images saved to {selected_images_file}")
    print(f"[OK] History saved to {history_file}")
    return selected_images

def manage_latest_generation(metadata=None, save=True):
    """
    Manage the latest_generation.json pointer file.
    This allows us to always load the most recent results.
    """
    latest_file = os.path.join(GRADCAM_DIR, 'latest_generation.json')

    if save and metadata is not None:
        with open(latest_file, 'w') as f:
            json.dump(metadata, f, indent=4)
        print(f"[OK] Latest generation pointer saved to {latest_file}")
        return metadata
    elif os.path.exists(latest_file):
        with open(latest_file, 'r') as f:
            return json.load(f)
    else:
        return None

# PHASE 2: HEATMAP FUSION (ENSEMBLE HEATMAP)

def get_fusion_heatmap(heatmaps, method='mean'):
    """
    Fuse heatmaps from multiple models into a single ensemble heatmap.

    Args:
        heatmaps: List of heatmap arrays (numpy arrays)
        method: 'mean' or 'max' for fusion strategy

    Returns:
        Fused and normalized heatmap
    """
    if not heatmaps:
        return None

    # Filter out None values
    valid_heatmaps = [h for h in heatmaps if h is not None]
    if not valid_heatmaps:
        return None

    # Resize all heatmaps to IMG_SIZE
    resized_heatmaps = []
    for hmap in valid_heatmaps:
        # Handle both 2D and 3D heatmaps
        if len(hmap.shape) == 2:
            hmap_resized = cv2.resize(hmap.astype(np.float32), IMG_SIZE)
        else:
            # If it's a 3D heatmap (e.g., occlusion grid), resize
            hmap_resized = cv2.resize(hmap.astype(np.float32), IMG_SIZE)
        resized_heatmaps.append(hmap_resized)

    # Stack and fuse
    stacked = np.stack(resized_heatmaps, axis=0)

    if method == 'mean':
        fused = np.mean(stacked, axis=0)
    elif method == 'max':
        fused = np.max(stacked, axis=0)
    else:
        raise ValueError(f"Unknown fusion method: {method}")

    # Normalize to [0, 1]
    max_val = np.max(fused)
    if max_val > 0:
        fused = fused / max_val

    return fused

def get_heatmap_for_fixed_image(model_name, model, img_path):
    """
    Dispatch function: Grad-CAM for CNNs, Occlusion Sensitivity for ViT/YOLO.
    This uses the functions defined in Block 7.
    """
    try:
        # Load image as tensor for Grad-CAM
        img = np.array(Image.open(img_path).convert('RGB').resize(IMG_SIZE))
        img_tensor = np.expand_dims(img.astype(np.float32), axis=0)

        # Use the existing get_heatmap_for_model function from Block 7
        heatmap, pred_idx = get_heatmap_for_model(model_name, model, img_path, img_arr=img_tensor)
        return heatmap, pred_idx
    except Exception as e:
        print(f" [FAILED] Error generating heatmap for {model_name}: {e}")
        return None, None

# PHASE 3: PUBLICATION-READY GRID GENERATION

def create_publication_grid(original_img_path, model_overlays, class_name, img_name, save_dir):

    # Load original image
    orig_img = np.array(Image.open(original_img_path).convert('RGB').resize(IMG_SIZE))

    # Determine grid layout
    n_images = 1 + len(model_overlays)  # Original + overlays
    n_cols = min(n_images, 4)  # Max 4 columns
    n_rows = (n_images + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))

    # Flatten axes for easier indexing
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    # Plot original image
    axes[0].imshow(orig_img)
    axes[0].set_title('Original', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Plot each model overlay
    for i, (model_name, overlay_path) in enumerate(model_overlays):
        overlay_img = np.array(Image.open(overlay_path))
        axes[i + 1].imshow(overlay_img)
        axes[i + 1].set_title(f'{model_name}', fontsize=14, fontweight='bold')
        axes[i + 1].axis('off')

    # Hide any unused subplots
    for i in range(n_images, len(axes)):
        axes[i].axis('off')

    # Add a main title
    fig.suptitle(f'Class: {class_name}', fontsize=16, fontweight='bold', y=1.02)

    plt.tight_layout()

    # Save with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = os.path.join(save_dir, f"{img_name}_{class_name}_grid_{timestamp}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    return save_path

# PHASE 4: MAIN GENERATION PIPELINE

def run_explainability_pipeline(force_generate=False):

    # Check if we should load existing results
    if not force_generate:
        latest_metadata = manage_latest_generation(save=False)
        if latest_metadata:
            print("\n" + "="*60)
            print("LOADING LATEST GENERATION RESULTS")
            print("="*60)
            print(f"Generation time: {latest_metadata.get('generation_time', 'unknown')}")
            print(f"Models: {latest_metadata.get('top_3_models', [])}")
            print("="*60)

            # Load selected images from metadata
            selected_images = latest_metadata.get('selected_images', {})

            # Build paths dictionary
            result = {
                'generation_time': latest_metadata.get('generation_time'),
                'selected_images': selected_images,
                'top_3_models': latest_metadata.get('top_3_models'),
                'grid_dir': latest_metadata.get('grid_dir'),
                'model_dirs': {},
                'ensemble_dir': os.path.join(GRADCAM_DIR, 'Ensemble'),
                'grid_paths': {}
            }

            # Find all overlay paths for each class
            for class_name, img_info in selected_images.items():
                img_name = os.path.splitext(img_info['file_name'])[0]
                result['grid_paths'][class_name] = {}

                # Check each model's overlay
                for model_name in latest_metadata.get('top_3_models', []):
                    overlay_path = os.path.join(GRADCAM_DIR, model_name,
                                               f"{img_name}_{class_name}_overlay.png")
                    if os.path.exists(overlay_path):
                        result['grid_paths'][class_name][model_name] = overlay_path

                # Check ensemble
                ensemble_path = os.path.join(GRADCAM_DIR, 'Ensemble',
                                            f"{img_name}_{class_name}_ensemble.png")
                if os.path.exists(ensemble_path):
                    result['grid_paths'][class_name]['Ensemble'] = ensemble_path

            return result

    # --- GENERATE NEW RESULTS ---
    print("\n" + "="*60)
    print("GENERATING NEW EXPLAINABILITY RESULTS")
    print("="*60)

    # Step 1: Select images
    selected_images = load_or_select_images(generate_new=True)

    # Step 2: Verify models are loaded
    available_models = [m for m in TOP3 if m in loaded_models]
    if not available_models:
        print("[FAILED] Error: No models found in loaded_models.")
        print(f" Available models: {list(loaded_models.keys())}")
        return None

    print(f"[OK] Available models: {available_models}")

    # Step 3: Create directories
    for model_name in available_models + ['Ensemble']:
        os.makedirs(os.path.join(GRADCAM_DIR, model_name), exist_ok=True)

    grid_dir = os.path.join(GRADCAM_DIR, 'Grid')
    os.makedirs(grid_dir, exist_ok=True)

    # Step 4: Generate heatmaps for each image
    all_results = {}

    for class_name, img_info in selected_images.items():
        img_path = img_info['file_path']
        img_name = os.path.splitext(img_info['file_name'])[0]

        print(f"\n▶ Processing {class_name}")
        print(f" Image: {img_info['file_name']}")

        model_heatmaps = {}
        model_overlays = []

        for model_name in available_models:
            print(f" -> Generating heatmap for {model_name}...")

            model = loaded_models[model_name]
            heatmap, pred_idx = get_heatmap_for_fixed_image(model_name, model, img_path)

            if heatmap is not None and np.max(heatmap) > 0.01:
                # Save raw heatmap
                heatmap_path = os.path.join(GRADCAM_DIR, model_name,
                                           f"{img_name}_{class_name}_heatmap.npy")
                np.save(heatmap_path, heatmap)

                # Generate and save overlay
                overlay = apply_heatmap_overlay(img_path, heatmap)
                overlay_path = os.path.join(GRADCAM_DIR, model_name,
                                           f"{img_name}_{class_name}_overlay.png")
                plt.imsave(overlay_path, overlay.astype(np.uint8))

                model_heatmaps[model_name] = heatmap
                model_overlays.append((model_name, overlay_path))
                print(f" [OK] Saved overlay: {overlay_path}")
            else:
                print(f" [FAILED] Heatmap generation failed or heatmap was empty")

        # Step 5: Generate Ensemble Heatmap
        if len(model_heatmaps) >= 2:
            print(f" -> Generating ensemble heatmap...")
            heatmaps_for_fusion = [h for h in model_heatmaps.values() if h is not None]
            ensemble_heatmap = get_fusion_heatmap(heatmaps_for_fusion, method='mean')

            if ensemble_heatmap is not None:
                # Save ensemble heatmap
                ensemble_heatmap_path = os.path.join(GRADCAM_DIR, 'Ensemble',
                                                    f"{img_name}_{class_name}_heatmap.npy")
                np.save(ensemble_heatmap_path, ensemble_heatmap)

                # Generate and save ensemble overlay
                ensemble_overlay = apply_heatmap_overlay(img_path, ensemble_heatmap)
                ensemble_overlay_path = os.path.join(GRADCAM_DIR, 'Ensemble',
                                                    f"{img_name}_{class_name}_ensemble.png")
                plt.imsave(ensemble_overlay_path, ensemble_overlay.astype(np.uint8))

                # Add ensemble to overlays for grid
                model_overlays.append(('Ensemble', ensemble_overlay_path))
                print(f" [OK] Saved ensemble overlay: {ensemble_overlay_path}")

        # Step 6: Create publication-ready grid
        if model_overlays:
            print(f" -> Creating publication grid...")
            grid_path = create_publication_grid(
                img_path, model_overlays, class_name, img_name, grid_dir
            )
            print(f" [OK] Grid saved: {grid_path}")

            # Store results
            all_results[class_name] = {
                'image_path': img_path,
                'model_heatmaps': model_heatmaps,
                'overlays': model_overlays,
                'grid_path': grid_path
            }

    # Step 7: Save metadata
    metadata = {
        'generation_time': datetime.now().isoformat(),
        'selected_images': selected_images,
        'top_3_models': TOP3,
        'available_models': available_models,
        'img_size': IMG_SIZE,
        'grid_dir': grid_dir,
        'num_classes': len(class_names)
    }
    manage_latest_generation(metadata, save=True)

    # Step 8: Return results summary
    result = {
        'generation_time': metadata['generation_time'],
        'selected_images': selected_images,
        'top_3_models': TOP3,
        'grid_dir': grid_dir,
        'model_dirs': {m: os.path.join(GRADCAM_DIR, m) for m in available_models},
        'ensemble_dir': os.path.join(GRADCAM_DIR, 'Ensemble'),
        'grid_paths': {cls: data['grid_path'] for cls, data in all_results.items()}
    }

    print("\n" + "="*60)
    print("[OK] EXPLAINABILITY PIPELINE COMPLETE.")
    print("="*60)
    print(f" Generated results for {len(all_results)} classes")
    print(f" Models used: {available_models}")
    print(f" Results saved to: {GRADCAM_DIR}")
    print(f" Grid images saved to: {grid_dir}")
    print("="*60)

    return result

# PHASE 5: EXECUTION

# Run the pipeline
results = run_explainability_pipeline(force_generate=NEW_GENERATION)

# Print summary of available results
if results:
    print("\n" + "="*60)
    print("AVAILABLE RESULTS SUMMARY")
    print("="*60)
    print(f"Generation time: {results.get('generation_time', 'unknown')}")
    print(f"Grid directory: {results.get('grid_dir', 'unknown')}")
    print(f"Number of classes processed: {len(results.get('selected_images', {}))}")

    # Show which images are available
    selected_images = results.get('selected_images', {})
    for class_name, img_info in selected_images.items():
        print(f"\n{class_name}:")
        print(f" Image: {img_info['file_name']}")

        # Check which overlays exist
        img_name = os.path.splitext(img_info['file_name'])[0]
        for model_name in results.get('top_3_models', []) + ['Ensemble']:
            overlay_path = os.path.join(GRADCAM_DIR, model_name,
                                       f"{img_name}_{class_name}_overlay.png")
            if model_name == 'Ensemble':
                overlay_path = os.path.join(GRADCAM_DIR, model_name,
                                           f"{img_name}_{class_name}_ensemble.png")
            if os.path.exists(overlay_path):
                print(f" [OK] {model_name}: {os.path.basename(overlay_path)}")

    print("\nTo load these results later, set NEW_GENERATION = False")
    print("="*60)

# QUICK ACCESS FUNCTIONS FOR LATER USE

def load_latest_results():
    """Quick function to load the latest results."""
    metadata = manage_latest_generation(save=False)
    if metadata:
        print("[OK] Loaded latest generation results")
        return metadata
    else:
        print("[FAILED] No results found. Please generate first.")
        return None

def get_overlay_path(class_name, model_name):
    """Get the overlay path for a specific class and model."""
    if model_name == 'Ensemble':
        # For ensemble, the filename is different
        img_info = load_latest_results().get('selected_images', {}).get(class_name, {})
        img_name = os.path.splitext(img_info.get('file_name', ''))[0]
        return os.path.join(GRADCAM_DIR, 'Ensemble',
                           f"{img_name}_{class_name}_ensemble.png")
    else:
        img_info = load_latest_results().get('selected_images', {}).get(class_name, {})
        img_name = os.path.splitext(img_info.get('file_name', ''))[0]
        return os.path.join(GRADCAM_DIR, model_name,
                           f"{img_name}_{class_name}_overlay.png")

print("\n[OK] Pipeline ready.")
print(" - To view results, check the 'gradcam_results' folder in Drive")
print(" - Use load_latest_results() to get metadata")
print(" - Use get_overlay_path(class_name, model_name) to get overlay paths")

In [ ]:
# Block 8: Generate heatmaps for Top-3 models + Ensemble across all 10 classes
import glob
import os
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime
import cv2

# CONFIGURATION - CHANGE THIS TO CONTROL GENERATION
# Set to True to generate new heatmaps, False to load existing ones
NEW_GENERATION = False  # <-- CHANGE THIS TO False TO LOAD EXISTING RESULTS

print("\n" + "="*60)
print("BLOCK 8: GENERATING HEATMAPS WITH ENSEMBLE")
print("="*60)
print(f"NEW_GENERATION = {NEW_GENERATION}")
if NEW_GENERATION:
    print("[INFO] Mode: GENERATING NEW heatmaps")
else:
    print("[INFO] Mode: LOADING EXISTING heatmaps")
print("="*60)

# STEP 1: ENSURE IMAGE SELECTION EXISTS

# Make sure GRADCAM_DIR exists
os.makedirs(GRADCAM_DIR, exist_ok=True)

selected_images_file = os.path.join(GRADCAM_DIR, 'selected_images.json')

# ONLY create new selection if NEW_GENERATION is True
if NEW_GENERATION or not os.path.exists(selected_images_file):
    print("[WARNING] Creating new image selection...")
    selected_images = {}
    for class_name in class_names:
        class_path = os.path.join(dataset_final_path, 'test', class_name)
        if os.path.isdir(class_path):
            image_files = [f for f in os.listdir(class_path)
                          if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]
            if image_files:
                image_files.sort()
                selected_images[class_name] = {
                    'file_name': image_files[0],
                    'file_path': os.path.join(class_path, image_files[0])
                }
                print(f" [OK] {class_name}: {image_files[0]}")

    with open(selected_images_file, 'w') as f:
        json.dump(selected_images, f, indent=4)
    print(f"[OK] Saved to {selected_images_file}")
else:
    print("[OK] Loading existing image selection...")
    with open(selected_images_file, 'r') as f:
        selected_images = json.load(f)

print(f"\n[OK] Using {len(selected_images)} fixed images (one per class)")

# STEP 2: CREATE DIRECTORIES

for model_name in TOP3 + ['Ensemble']:
    os.makedirs(os.path.join(GRADCAM_DIR, model_name), exist_ok=True)

grid_dir = os.path.join(GRADCAM_DIR, 'Grid')
os.makedirs(grid_dir, exist_ok=True)

# STEP 3: HELPER FUNCTIONS

def get_model_heatmap(model_name, model, img_path, class_name, force_generate=False):
    """Generate or load heatmap for a single model."""
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    model_save_dir = os.path.join(GRADCAM_DIR, model_name)

    overlay_path = os.path.join(model_save_dir, f'{img_name}_{class_name}_overlay.png')
    heatmap_path = os.path.join(model_save_dir, f'{img_name}_{class_name}_heatmap.npy')

    # Try to load existing (if NOT force_generate)
    if not force_generate and os.path.exists(overlay_path) and os.path.exists(heatmap_path):
        heatmap = np.load(heatmap_path)
        overlay = np.array(Image.open(overlay_path))
        return heatmap, overlay, True  # loaded from cache

    # Generate new
    print(f" Generating {model_name} heatmap...")
    img_arr = np.expand_dims(
        tf.keras.utils.img_to_array(
            tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
        ), axis=0
    )

    try:
        heatmap, pred_idx = get_heatmap_for_model(model_name, model, img_path, img_arr)

        if heatmap is not None and np.max(heatmap) > 0.01:
            np.save(heatmap_path, heatmap)
            overlay = apply_heatmap_overlay(img_path, heatmap)
            Image.fromarray(overlay).save(overlay_path)
            return heatmap, overlay, False  # newly generated
        else:
            overlay = np.array(Image.open(img_path).convert('RGB').resize(IMG_SIZE))
            return None, overlay, False
    except Exception as e:
        print(f" Error: {e}")
        overlay = np.array(Image.open(img_path).convert('RGB').resize(IMG_SIZE))
        return None, overlay, False

def get_ensemble_heatmap(heatmaps, img_path, class_name, img_name, force_generate=False):
    """Generate or load ensemble heatmap."""
    ensemble_dir = os.path.join(GRADCAM_DIR, 'Ensemble')

    overlay_path = os.path.join(ensemble_dir, f'{img_name}_{class_name}_ensemble.png')
    heatmap_path = os.path.join(ensemble_dir, f'{img_name}_{class_name}_heatmap.npy')

    if not force_generate and os.path.exists(overlay_path) and os.path.exists(heatmap_path):
        heatmap = np.load(heatmap_path)
        overlay = np.array(Image.open(overlay_path))
        return heatmap, overlay, True  # loaded from cache

    # Generate new ensemble
    valid_heatmaps = [h for h in heatmaps if h is not None]
    if len(valid_heatmaps) < 2:
        return None, None, False

    # Fuse using mean
    resized_heatmaps = []
    for h in valid_heatmaps:
        h_resized = cv2.resize(h.astype(np.float32), IMG_SIZE)
        resized_heatmaps.append(h_resized)

    fused = np.mean(np.stack(resized_heatmaps, axis=0), axis=0)

    # Normalize
    max_val = np.max(fused)
    if max_val > 0:
        fused = fused / max_val

    # Save
    np.save(heatmap_path, fused)
    overlay = apply_heatmap_overlay(img_path, fused)
    Image.fromarray(overlay).save(overlay_path)

    return fused, overlay, False  # newly generated

def create_grid(original_img, model_overlays, class_name, img_name, save_dir):
    """Create grid with Original + Top-3 models + Ensemble."""
    n_images = 1 + len(model_overlays)
    n_cols = min(n_images, 5)
    n_rows = (n_images + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    # Original
    axes[0].imshow(original_img)
    axes[0].set_title('Original', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Model overlays
    for i, (model_name, overlay) in enumerate(model_overlays):
        axes[i + 1].imshow(overlay)
        title = f'{model_name} ★' if model_name == 'Ensemble' else model_name
        color = 'darkred' if model_name == 'Ensemble' else 'black'
        axes[i + 1].set_title(title, fontsize=14, fontweight='bold', color=color)
        axes[i + 1].axis('off')

    # Hide unused
    for i in range(n_images, len(axes)):
        axes[i].axis('off')

    fig.suptitle(f'Class: {class_name}', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = os.path.join(save_dir, f'{img_name}_{class_name}_grid_{timestamp}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    return save_path

# STEP 4: GENERATE OR LOAD HEATMAPS FOR ALL CLASSES

all_results = {}

for class_name, img_info in selected_images.items():
    img_path = img_info['file_path']
    img_name = os.path.splitext(img_info['file_name'])[0]

    print(f"\n▶ Processing {class_name}")
    print(f" Image: {img_info['file_name']}")

    # Load original
    original_img = np.array(Image.open(img_path).convert('RGB').resize(IMG_SIZE))

    # Collect heatmaps and overlays
    model_heatmaps = {}
    model_overlays = []

    # Process each Top-3 model
    for model_name in TOP3:
        if model_name not in loaded_models:
            print(f" [WARNING] {model_name} not loaded, skipping")
            continue

        heatmap, overlay, cached = get_model_heatmap(
            model_name,
            loaded_models[model_name],
            img_path,
            class_name,
            force_generate=NEW_GENERATION  # <-- KEY: Pass the flag here
        )

        if heatmap is not None:
            model_heatmaps[model_name] = heatmap
        model_overlays.append((model_name, overlay))
        status = "loaded" if cached else ("generated" if heatmap is not None else "failed")
        print(f" [OK] {model_name}: {status}")

    # Generate Ensemble
    print(f" -> Ensemble...")
    ensemble_heatmap, ensemble_overlay, cached = get_ensemble_heatmap(
        list(model_heatmaps.values()),
        img_path,
        class_name,
        img_name,
        force_generate=NEW_GENERATION  # <-- KEY: Pass the flag here
    )

    if ensemble_heatmap is not None:
        model_overlays.append(('Ensemble', ensemble_overlay))
        status = "loaded" if cached else "generated"
        print(f" [OK] Ensemble: {status}")
    else:
        print(f" [WARNING] Ensemble could not be generated (need at least 2 models)")

    # Create grid (always create if NEW_GENERATION or grid doesn't exist)
    grid_path = None
    if NEW_GENERATION:
        grid_path = create_grid(
            original_img, model_overlays, class_name, img_name, grid_dir
        )
        print(f" [OK] Grid generated: {os.path.basename(grid_path)}")
    else:
        # Try to find existing grid
        grid_pattern = os.path.join(grid_dir, f'{img_name}_{class_name}_grid_*.png')
        existing_grids = glob.glob(grid_pattern)
        if existing_grids:
            grid_path = existing_grids[-1]  # Use the latest
            print(f" [OK] Grid loaded: {os.path.basename(grid_path)}")
        else:
            # No grid exists, generate it even if NEW_GENERATION is False
            print(f" [WARNING] No existing grid found, generating...")
            grid_path = create_grid(
                original_img, model_overlays, class_name, img_name, grid_dir
            )
            print(f" [OK] Grid generated: {os.path.basename(grid_path)}")

    all_results[class_name] = {
        'image_path': img_path,
        'grid_path': grid_path,
        'model_heatmaps': model_heatmaps,
        'ensemble_heatmap': ensemble_heatmap
    }

# STEP 5: VERIFICATION AND SUMMARY

print("\n" + "="*60)
print("VERIFICATION SUMMARY")
print("="*60)
print(f"Mode: {'GENERATING NEW' if NEW_GENERATION else 'LOADING EXISTING'}")

all_complete = True
for class_name in selected_images:
    img_name = os.path.splitext(selected_images[class_name]['file_name'])[0]
    print(f"\n{class_name}:")

    for model_name in TOP3:
        if model_name in loaded_models:
            path = os.path.join(GRADCAM_DIR, model_name, f'{img_name}_{class_name}_overlay.png')
            status = "[OK]" if os.path.exists(path) else "[FAILED]"
            print(f" {status} {model_name}")

    path = os.path.join(GRADCAM_DIR, 'Ensemble', f'{img_name}_{class_name}_ensemble.png')
    status = "[OK]" if os.path.exists(path) else "[FAILED]"
    print(f" {status} Ensemble")

    if all_results[class_name]['grid_path'] and os.path.exists(all_results[class_name]['grid_path']):
        print(f" [OK] Grid: {os.path.basename(all_results[class_name]['grid_path'])}")
    else:
        print(f" [FAILED] Grid: MISSING")

print("\n" + "="*60)
print("[OK] BLOCK 8 COMPLETE.")
print("="*60)
print(f" Mode: {'NEW GENERATION' if NEW_GENERATION else 'LOADED FROM CACHE'}")
print(f" Results saved to: {GRADCAM_DIR}")
print(f" Grids saved to: {grid_dir}")
print("="*60)

# STEP 6: QUICK LOAD FUNCTION FOR BLOCK 9

def load_heatmap_results(class_name=None):
    """Load heatmap results for use in Block 9."""
    results = {}

    for cls_name, img_info in selected_images.items():
        if class_name and cls_name != class_name:
            continue

        img_name = os.path.splitext(img_info['file_name'])[0]
        results[cls_name] = {'overlays': {}, 'heatmaps': {}}

        for model_name in TOP3:
            if model_name in loaded_models:
                overlay_path = os.path.join(GRADCAM_DIR, model_name,
                                           f'{img_name}_{cls_name}_overlay.png')
                heatmap_path = os.path.join(GRADCAM_DIR, model_name,
                                           f'{img_name}_{cls_name}_heatmap.npy')

                if os.path.exists(overlay_path):
                    results[cls_name]['overlays'][model_name] = np.array(Image.open(overlay_path))
                if os.path.exists(heatmap_path):
                    results[cls_name]['heatmaps'][model_name] = np.load(heatmap_path)

        # Ensemble
        overlay_path = os.path.join(GRADCAM_DIR, 'Ensemble',
                                   f'{img_name}_{cls_name}_ensemble.png')
        heatmap_path = os.path.join(GRADCAM_DIR, 'Ensemble',
                                   f'{img_name}_{cls_name}_heatmap.npy')

        if os.path.exists(overlay_path):
            results[cls_name]['overlays']['Ensemble'] = np.array(Image.open(overlay_path))
        if os.path.exists(heatmap_path):
            results[cls_name]['heatmaps']['Ensemble'] = np.load(heatmap_path)

    return results

print("\n[OK] Use load_heatmap_results() to load results in Block 9")

print("\n[OK] To view a sample grid:")
print(" from IPython.display import Image as IPImage")
print(" IPImage(all_results['Acne']['grid_path'])")

In [ ]:
# Block 9: Heatmap comparison for one disease across three models + Ensemble
# Uses the reproducible pipeline (same images as Block 8)

import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime

# CONFIGURATION - CHANGE THIS TO CONTROL GENERATION
# Set to True to generate new comparison, False to load existing
NEW_GENERATION = False  # <-- CHANGE THIS TO False TO LOAD EXISTING

COMPARISON_VERSION = "v2" # Change to v2, v3, etc. if needed

print("\n" + "="*80)
print("BLOCK 9: HEATMAP COMPARISON")
print("="*80)
print(f"NEW_GENERATION = {NEW_GENERATION}")
if NEW_GENERATION:
    print("[INFO] Mode: GENERATING NEW comparison image")
else:
    print("[INFO] Mode: LOADING EXISTING comparison image")
print("="*80)

# STEP 1: LOAD FIXED IMAGE SELECTION

selected_images_file = os.path.join(GRADCAM_DIR, 'selected_images.json')

if os.path.exists(selected_images_file):
    with open(selected_images_file, 'r') as f:
        selected_images = json.load(f)
    print(f"[OK] Loaded {len(selected_images)} fixed images from selection")
else:
    print("[WARNING] No selected_images.json found. Please run Block 8 first.")
    # Fallback: use first image from each class
    selected_images = {}
    for class_name in class_names:
        class_path = os.path.join(dataset_final_path, 'test', class_name)
        if os.path.isdir(class_path):
            image_files = [f for f in os.listdir(class_path)
                          if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if image_files:
                image_files.sort()
                selected_images[class_name] = {
                    'file_name': image_files[0],
                    'file_path': os.path.join(class_path, image_files[0])
                }

# STEP 2: SEARCH FOR EXISTING COMPARISON IMAGE

comparison_pattern = os.path.join(
    GRADCAM_DIR,
    f"gradcam_comparison_{COMPARISON_VERSION}_*.png"
)
saved_files = glob.glob(comparison_pattern)

# STEP 3: LOAD EXISTING COMPARISON IMAGE (if NEW_GENERATION is False)

if (not NEW_GENERATION) and len(saved_files) > 0:
    # Sort by modification time to get the latest
    latest_file = max(saved_files, key=os.path.getmtime)

    print("\n" + "="*80)
    print("[INFO] LOADING LATEST COMPARISON IMAGE")
    print("="*80)
    print(f"File: {os.path.basename(latest_file)}")
    print(f"Path: {latest_file}")
    print("="*80)

    # Display the image
    plt.figure(figsize=(20, 45))
    img = np.array(Image.open(latest_file))
    plt.imshow(img)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    print("\n[OK] Comparison image loaded successfully.")
    print(f"To regenerate, set NEW_GENERATION = True and re-run this block.")

    # Store the path for later use
    comparison_path = latest_file

# STEP 4: GENERATE NEW COMPARISON IMAGE

else:
    if (not NEW_GENERATION) and len(saved_files) == 0:
        print("[WARNING] No saved comparison image found.")
        print("[INFO] Generating a new comparison image...\n")

    print("\n" + "="*80)
    print("[INFO] GENERATING NEW COMPARISON IMAGE")
    print("="*80)
    print(f"Models: {TOP3} + Ensemble")
    print(f"Classes: {len(class_names)}")
    print("="*80)

    # Determine layout
    # Columns: Original + Top3 Models + Ensemble = 5 columns
    ncols = len(TOP3) + 2  # +2 for Original and Ensemble
    nrows = len(class_names)

    # Create figure
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(22, 6 * nrows)
    )

    # Handle single row case
    if nrows == 1:
        axes = [axes]

    # Add main title
    fig.suptitle(
        "Explainability Heatmap Comparison: Same Image Across All Models\n"
        "Red = Region Receiving the Highest Model Attention | ★ = Ensemble Fusion",
        fontsize=16,
        fontweight="bold",
        y=0.995
    )

    # Column headers
    headers = ["Original"] + TOP3 + ["Ensemble ★"]

    for j, header in enumerate(headers):
        axes[0][j].set_title(
            header,
            fontsize=13,
            fontweight="bold",
            pad=15,
            color='darkred' if 'Ensemble' in header else 'black'
        )

    # Track generation status
    generated_count = 0
    loaded_count = 0
    missing_count = 0

    # Process each class
    for i, class_name in enumerate(class_names):
        if class_name not in selected_images:
            print(f"[WARNING] No image selected for {class_name}, skipping...")
            missing_count += 1
            continue

        img_info = selected_images[class_name]
        img_path = img_info['file_path']
        img_name = os.path.splitext(img_info['file_name'])[0]

        # Load original image
        orig = np.array(
            Image.open(img_path)
            .convert("RGB")
            .resize(IMG_SIZE)
        )

        # Row label
        axes[i][0].set_title(
            f"{class_name}",
            fontsize=12,
            fontweight="bold",
            pad=10,
            loc='left'
        )
        axes[i][0].imshow(orig)
        axes[i][0].axis("off")

        # Process each model
        for j, model_name in enumerate(TOP3):
            # Look for overlay in the model's folder
            overlay_path = os.path.join(
                GRADCAM_DIR,
                model_name,
                f"{img_name}_{class_name}_overlay.png"
            )

            # Fallback to old naming convention
            if not os.path.exists(overlay_path):
                overlay_path = os.path.join(
                    GRADCAM_DIR,
                    model_name,
                    f"{class_name.replace(' ', '_')}_gradcam.png"
                )

            if os.path.exists(overlay_path):
                overlay = np.array(Image.open(overlay_path))
                loaded_count += 1
            else:
                print(f" [WARNING] Missing overlay for {class_name} - {model_name}")
                overlay = orig.copy()
                missing_count += 1

            axes[i][j + 1].imshow(overlay)
            axes[i][j + 1].axis("off")

        # Ensemble column (last column)
        ensemble_path = os.path.join(
            GRADCAM_DIR,
            'Ensemble',
            f"{img_name}_{class_name}_ensemble.png"
        )

        if os.path.exists(ensemble_path):
            ensemble_img = np.array(Image.open(ensemble_path))
            loaded_count += 1
        else:
            print(f" [WARNING] Missing ensemble for {class_name}")
            ensemble_img = orig.copy()
            missing_count += 1

        # Place ensemble in the last column
        axes[i][-1].imshow(ensemble_img)
        axes[i][-1].axis("off")

        generated_count += 1

    # Layout
    plt.subplots_adjust(
        left=0.03,
        right=0.99,
        top=0.95,
        bottom=0.02,
        wspace=0.02,
        hspace=0.08
    )

    # Save figure with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    comparison_path = os.path.join(
        GRADCAM_DIR,
        f"gradcam_comparison_{COMPARISON_VERSION}_{timestamp}.png"
    )

    plt.savefig(
        comparison_path,
        dpi=300,
        bbox_inches="tight",
        facecolor='white'
    )

    # Display
    plt.show()

    # Summary
    print("\n" + "="*80)
    print("[OK] NEW COMPARISON IMAGE GENERATED SUCCESSFULLY.")
    print("="*80)
    print(f" Classes processed: {generated_count}")
    print(f" Overlays loaded: {loaded_count}")
    print(f" Missing overlays: {missing_count}")
    print(f" Saved to: {comparison_path}")
    print("="*80)

# STEP 5: QUICK ACCESS FUNCTIONS

def get_latest_comparison_path():
    """Get the path to the latest comparison image."""
    pattern = os.path.join(GRADCAM_DIR, f"gradcam_comparison_{COMPARISON_VERSION}_*.png")
    files = glob.glob(pattern)
    if files:
        return max(files, key=os.path.getmtime)
    return None

def display_comparison():
    """Display the latest comparison image."""
    path = get_latest_comparison_path()
    if path:
        from IPython.display import Image as IPImage
        display(IPImage(path))
        print(f"Displaying: {os.path.basename(path)}")
    else:
        print("No comparison image found. Run with NEW_GENERATION = True to generate.")

print("\n" + "="*80)
print("BLOCK 9 COMPLETE.")
print("="*80)
print("Quick access functions:")
print(" - get_latest_comparison_path() - Get the path to the latest comparison")
print(" - display_comparison() - Display the latest comparison")
print("="*80)

# Example usage
if 'comparison_path' in locals() and comparison_path:
    print(f"\n[OK] Latest comparison: {os.path.basename(comparison_path)}")
    print(f" Path: {comparison_path}")

---
## PART 5: Two-Stage Prediction (Ensemble + Per-Class Specialist)
Input an image and get an 8-panel output:
**[Original] [All Disease %] [Grad-CAM/Occlusion Heatmap] [Region from Heatmap]** for both the
calibrated ensemble and the per-class specialist model.

> "Specialist model" = the architecture with the best validation F1 for the predicted class (a model, not a human).


In [ ]:
# Block 10: Two-Stage Prediction (Ensemble + Per-Class Specialist Model)
#
# Input : image path
# Output: 8-panel figure + printed probabilities
#
# NOTE ON TERMINOLOGY: a "specialist model" below is the individual architecture
# that achieved the highest validation F1 for the predicted disease class
# (see Block 4). It is a MODEL giving a second opinion, NOT a human expert.
#
# Stage 1 now reports TEMPERATURE-CALIBRATED confidence (Block 6-B),
# because the raw logistic-regression meta-classifier is poorly calibrated
# (under-confident: worst ECE of all models before scaling; see Block 11-B).

import os
import io
import glob
import json
import base64
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from datetime import datetime
import tensorflow as tf

# CONFIGURATION - CHANGE THIS TO CONTROL GENERATION
# Set to True to generate new predictions, False to load existing
NEW_GENERATION = False  # <-- CHANGE THIS TO False TO LOAD EXISTING RESULTS

# Version for saved results
PREDICTION_VERSION = "v2"

mode = "GENERATING new predictions" if NEW_GENERATION else "LOADING existing predictions"
print(f"Block 10: Two-Stage Prediction Pipeline  |  {mode}  (NEW_GENERATION={NEW_GENERATION})")

# Global for mobile app integration (set by predict_full)
LAST_PREDICTION_DETAILS = None

# STEP 1: LOAD FIXED IMAGE SELECTION (for consistent testing)

selected_images_file = os.path.join(GRADCAM_DIR, 'selected_images.json')

if os.path.exists(selected_images_file):
    with open(selected_images_file, 'r') as f:
        selected_images = json.load(f)
    print(f"[OK] Loaded {len(selected_images)} fixed images from selection")
else:
    print("[WARNING] No selected_images.json found. Using random images.")
    selected_images = None

# STEP 2: CORE PREDICTION FUNCTION

def predict_full(image_path, save_result=True, force_generate=False, output_dir=None):
    """
    Two-stage prediction with ensemble + specialist model.

    Args:
        image_path: Path to the image to predict
        save_result: Whether to save the figure
        force_generate: If True, regenerate even if saved result exists
        output_dir: Custom directory to save results (if None, uses RESULTS_DIR)

    Returns:
        final_cls: Predicted class name
        avg_probs: Probability array
    """

    # Use custom output directory if provided, otherwise use default
    save_dir = output_dir if output_dir is not None else RESULTS_DIR
    os.makedirs(save_dir, exist_ok=True)

    # Check if result already exists (for reproducibility)
    fname = os.path.splitext(os.path.basename(image_path))[0]
    out_path = os.path.join(save_dir, f'{fname}_2stage_prediction.png')

    if (not force_generate) and os.path.exists(out_path):
        print(f"Loaded cached prediction: {fname}")
        plt.figure(figsize=(22, 12))
        img = np.array(Image.open(out_path))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        return None, None  # cached image only; call with force_generate=True for values

    # Load and preprocess image
    img_arr = np.expand_dims(
        tf.keras.utils.img_to_array(
            tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
        ), axis=0
    )

    def _safe_predict_one(name, model):
        if name == 'YOLO11m_cls':
            res = model(image_path, verbose=False)[0]
            raw_probs = res.probs.data.cpu().numpy()
            idx_to_name = model.names
            name_to_idx = {c: i for i, c in enumerate(class_names)}
            fixed = np.zeros_like(raw_probs)
            for yolo_i, cname in idx_to_name.items():
                fixed[name_to_idx[cname]] = raw_probs[yolo_i]
            return fixed
        else:
            return model.predict(img_arr, verbose=0)[0]

    # Get predictions from all models
    all_probs = {}
    for name in ALL_MODELS:
        if name in loaded_models:
            try:
                all_probs[name] = _safe_predict_one(name, loaded_models[name])
            except Exception as e:
                print(f" [WARNING] Error predicting with {name}: {e}")

    # --- Stage 1: Ensemble ---
    top3_avail = [m for m in TOP3 if m in all_probs]
    if len(top3_avail) < 2:
        print("[WARNING] Not enough models available for ensemble prediction.")
        return None, None

    meta_input = np.concatenate([all_probs[m] for m in top3_avail]).reshape(1, -1)

    # Use temperature-scaled probabilities (calibrated) instead of raw
    try:
        meta_logits = meta_model.decision_function(meta_input)   # (1, n_classes)
        meta_probs  = temperature_scale(meta_logits, optimal_T)[0]
    except Exception:
        meta_probs  = meta_model.predict_proba(meta_input)[0]

    final_idx  = int(np.argmax(meta_probs))
    final_cls  = class_names[final_idx]
    avg_probs  = meta_probs
    ens_conf   = meta_probs[final_idx] * 100

    # --- Ensemble heatmap: REAL fusion of every available TOP3 model, not
    # just one model's map. Each model's heatmap is resized to IMG_SIZE and
    # min-max normalized to [0,1] individually (so no single model's raw
    # magnitude dominates), then the per-pixel mean is taken across models.
    # This mirrors the probability-level ensemble above (same TOP3 members),
    # just applied to attention instead of class probabilities.
    ens_heatmaps = []
    ens_models_used = []
    for m_name in top3_avail:
        if m_name not in loaded_models:
            continue
        hm, _ = get_heatmap_for_model(m_name, loaded_models[m_name], image_path, img_arr)
        if hm is None:
            continue
        hm = cv2.resize(hm.astype(np.float32), IMG_SIZE)
        hm_max = hm.max()
        if hm_max > 0:
            hm = hm / hm_max
        ens_heatmaps.append(hm)
        ens_models_used.append(m_name)

    ens_heatmap = np.mean(ens_heatmaps, axis=0) if ens_heatmaps else None
    ens_heatmap_label = (f'Ensemble Heatmap (avg of {len(ens_models_used)}: '
                          f'{", ".join(ens_models_used)})') if ens_models_used else 'Ensemble Heatmap (unavailable)'

    ens_overlay = apply_heatmap_overlay(image_path, ens_heatmap) if ens_heatmap is not None else np.array(Image.open(image_path).convert('RGB').resize(IMG_SIZE))
    ens_bbox = get_bbox_from_heatmap(ens_heatmap, IMG_SIZE) if ens_heatmap is not None else None

    # --- Stage 2: Per-Class Specialist Model ---
    # Get the best model for the predicted class
    expert_model_name = best_models_per_disease[best_models_per_disease['Disease'] == final_cls]['Model'].values[0]

    if expert_model_name in all_probs:
        expert_probs = all_probs[expert_model_name]
        exp_conf = expert_probs[final_idx] * 100
    else:
        print(f"[WARNING] Specialist model {expert_model_name} not available.")
        expert_probs = meta_probs  # Fallback
        exp_conf = 0.0

    if expert_model_name in loaded_models:
        exp_heatmap, _ = get_heatmap_for_model(expert_model_name, loaded_models[expert_model_name], image_path, img_arr)
    else:
        exp_heatmap = None

    exp_overlay = apply_heatmap_overlay(image_path, exp_heatmap) if exp_heatmap is not None else np.array(Image.open(image_path).convert('RGB').resize(IMG_SIZE))
    exp_bbox = get_bbox_from_heatmap(exp_heatmap, IMG_SIZE) if exp_heatmap is not None else None

    # --- Plot: 8 panels (2 rows) ---
    orig_img = np.array(Image.open(image_path).convert('RGB').resize(IMG_SIZE))
    fig, axes = plt.subplots(2, 4, figsize=(22, 12))
    fig.patch.set_facecolor('#f8f9fa')
    fig.suptitle(
        f'Ensemble Prediction: {final_cls} ({ens_conf:.1f}%, calibrated)  |  Specialist Model [{expert_model_name}]: {exp_conf:.1f}%',
        fontsize=18,
        fontweight='bold',
        color='#2c3e50',
        y=1.02
    )

    def draw_row(ax_row, title_prefix, probs, conf, overlay, bbox, hm_title):
        # Original image
        ax_row[0].imshow(orig_img)
        ax_row[0].set_title(f'{title_prefix}\nOriginal Image', fontsize=12, fontweight='bold')
        ax_row[0].axis('off')

        # Probability bar chart
        sorted_idx = np.argsort(probs)
        colors_bar = ['#e74c3c' if i == final_idx else '#3498db' for i in sorted_idx]
        bars = ax_row[1].barh(
            [class_names[i] for i in sorted_idx],
            probs[sorted_idx] * 100,
            color=colors_bar
        )
        ax_row[1].set_xlabel('Confidence (%)', fontsize=10)
        ax_row[1].set_title('All Probabilities', fontsize=12, fontweight='bold')
        ax_row[1].set_xlim(0, 110)

        for bar, idx in zip(bars, sorted_idx):
            val = probs[idx] * 100
            if val > 0.5:
                ax_row[1].text(
                    val + 0.5,
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}%',
                    va='center',
                    fontsize=8,
                    color='#e74c3c' if idx == final_idx else '#2c3e50',
                    fontweight='bold' if idx == final_idx else 'normal'
                )

        # Heatmap overlay
        ax_row[2].imshow(overlay)
        ax_row[2].set_title(f'{hm_title}\nRed = Focus Area', fontsize=12, fontweight='bold')
        ax_row[2].axis('off')

        # Bounding box
        ax_row[3].imshow(orig_img)
        ax_row[3].set_title('Detected Region', fontsize=12, fontweight='bold')
        ax_row[3].axis('off')
        if bbox:
            x1, y1, x2, y2 = bbox
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=3,
                edgecolor='#e74c3c',
                facecolor='none'
            )
            ax_row[3].add_patch(rect)
            ax_row[3].text(
                x1, y1 - 8,
                f'{conf:.1f}%',
                color='white',
                fontsize=10,
                fontweight='bold',
                bbox=dict(facecolor='#e74c3c', alpha=0.85, edgecolor='none', pad=3)
            )
        else:
            ax_row[3].text(
                0.5, 0.5,
                'Region not clearly localized',
                ha='center',
                va='center',
                transform=ax_row[3].transAxes,
                fontsize=10,
                color='gray'
            )

    draw_row(
        axes[0],
        'Stage 1: Ensemble',
        avg_probs,
        ens_conf,
        ens_overlay,
        ens_bbox,
        ens_heatmap_label
    )

    draw_row(
        axes[1],
        f'Stage 2: Specialist ({expert_model_name})',
        expert_probs,
        exp_conf,
        exp_overlay,
        exp_bbox,
        f'Specialist Heatmap ({expert_model_name})'
    )

    plt.tight_layout()

    if save_result:
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(out_path, dpi=200, bbox_inches='tight')
        print(f'[OK] Saved 2-stage result to {out_path}')

    plt.show()

    # Print summary
    agreement = 'agrees' if exp_conf >= 50 else 'low agreement (independent second opinion)'
    print(f'\n{"="*60}')
    print(f' ENSEMBLE PREDICTION   : {final_cls} ({ens_conf:.2f}%, calibrated)')
    print(f' SPECIALIST RE-CHECK   : {expert_model_name} -> {exp_conf:.2f}% on "{final_cls}" [{agreement}]')
    print(f'{"="*60}')

    # --- Stash a mobile/web-friendly summary (JSON + small images) without
    # changing this function's normal return value, so Block 11/13 (which
    # call predict_full expecting (final_cls, avg_probs)) are unaffected.
    global LAST_PREDICTION_DETAILS
    def _img_to_b64(arr):
        buf = io.BytesIO()
        Image.fromarray(arr.astype(np.uint8)).save(buf, format='JPEG', quality=85)
        return base64.b64encode(buf.getvalue()).decode('utf-8')

    sorted_pairs = sorted(
        zip(class_names, (avg_probs * 100).tolist()), key=lambda p: p[1], reverse=True
    )
    LAST_PREDICTION_DETAILS = {
        'full_figure_filename': os.path.basename(out_path),
        'ensemble_class': final_cls,
        'ensemble_confidence': round(float(ens_conf), 1),
        'specialist_model': expert_model_name,
        'specialist_confidence': round(float(exp_conf), 1),
        'agreement': agreement,
        'all_probs': [{'class': c, 'pct': round(p, 1)} for c, p in sorted_pairs],
        'original_b64': _img_to_b64(orig_img),
        'ensemble_heatmap_b64': _img_to_b64(ens_overlay),
        'specialist_heatmap_b64': _img_to_b64(exp_overlay),
    }

    return final_cls, avg_probs

print('[OK] predict_full() upgraded: calibrated ensemble + per-class specialist re-check.')

# STEP 3: TEST ON FIXED IMAGES (if NEW_GENERATION is True)

if NEW_GENERATION:
    print("\n" + "="*80)
    print("[INFO] GENERATING PREDICTIONS FOR FIXED IMAGES")
    print("="*80)

    if selected_images:
        # Test on one image per class
        results_summary = {}

        for class_name, img_info in selected_images.items():
            img_path = img_info['file_path']
            print(f"\n▶ Testing: {class_name} - {img_info['file_name']}")

            final_cls, probs = predict_full(
                img_path,
                save_result=True,
                force_generate=NEW_GENERATION
            )

            results_summary[class_name] = {
                'image': img_info['file_name'],
                'prediction': final_cls,
                'correct': class_name == final_cls
            }

        print("\n" + "="*80)
        print("PREDICTION SUMMARY")
        print("="*80)
        correct = sum(1 for r in results_summary.values() if r['correct'])
        total = len(results_summary)
        print(f"Accuracy on fixed images: {correct}/{total} ({correct/total*100:.1f}%)")
        print("\nDetailed results:")
        for class_name, result in results_summary.items():
            status = "[OK]" if result['correct'] else "[FAILED]"
            print(f" {status} {class_name} -> {result['prediction']} (image: {result['image']})")
        print("="*80)
    else:
        print("[WARNING] No selected_images.json found. Testing on random images...")
        # Test on one random image per class
        for class_name in class_names:
            class_path = os.path.join(dataset_final_path, 'test', class_name)
            if os.path.isdir(class_path):
                image_files = [f for f in os.listdir(class_path)
                              if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                if image_files:
                    img_path = os.path.join(class_path, image_files[0])
                    print(f"\n▶ Testing: {class_name} - {image_files[0]}")
                    predict_full(img_path, save_result=True, force_generate=True)

else:
    print("\n" + "="*80)
    print("[INFO] LOADING EXISTING PREDICTIONS")
    print("="*80)

    # Only show non-mobile predictions in Block 10 (mobile results go to mobile_uploads folder)
    prediction_files = glob.glob(os.path.join(RESULTS_DIR, '*_2stage_prediction.png'))
    # Filter out mobile uploads
    non_mobile_files = [f for f in prediction_files if 'mobile_' not in os.path.basename(f)]

    if non_mobile_files:
        print(f"Found {len(non_mobile_files)} saved predictions (non-mobile).")
        latest = max(non_mobile_files, key=os.path.getmtime)
        print(f"Displaying most recent: {os.path.basename(latest)}")
        plt.figure(figsize=(22, 12))
        img = np.array(Image.open(latest))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("No non-mobile predictions found. Set NEW_GENERATION = True to generate.")

# STEP 4: QUICK ACCESS FUNCTIONS

def predict_single_image(image_path):
    """Quick wrapper to predict a single image."""
    return predict_full(image_path, save_result=True, force_generate=NEW_GENERATION)

def predict_batch(image_paths):
    """Predict multiple images and return results."""
    results = {}
    for path in image_paths:
        fname = os.path.basename(path)
        cls, probs = predict_full(path, save_result=True, force_generate=NEW_GENERATION)
        results[fname] = {'class': cls, 'probs': probs}
    return results

def get_latest_prediction_path():
    """Get the path to the latest prediction figure (non-mobile only)."""
    files = glob.glob(os.path.join(RESULTS_DIR, '*_2stage_prediction.png'))
    non_mobile_files = [f for f in files if 'mobile_' not in os.path.basename(f)]
    if non_mobile_files:
        return max(non_mobile_files, key=os.path.getmtime)
    return None

def list_saved_predictions(include_mobile=False):
    """Return a sorted list of all saved 2-stage prediction filenames."""
    files = glob.glob(os.path.join(RESULTS_DIR, '*_2stage_prediction.png'))
    if not include_mobile:
        files = [f for f in files if 'mobile_' not in os.path.basename(f)]
    return sorted(os.path.basename(f) for f in files)

def list_mobile_predictions():
    """Return a sorted list of mobile prediction filenames."""
    mobile_upload_dir = os.path.join(DRIVE_WORKSPACE, 'mobile_uploads')
    os.makedirs(mobile_upload_dir, exist_ok=True)
    files = glob.glob(os.path.join(mobile_upload_dir, '*_2stage_prediction.png'))
    return sorted(os.path.basename(f) for f in files)

print("Block 10 ready: predict_single_image(path), predict_batch(paths), "
      "get_latest_prediction_path(), list_saved_predictions(), list_mobile_predictions()")

latest_prediction = get_latest_prediction_path()
if latest_prediction:
    print(f"Latest prediction on file: {os.path.basename(latest_prediction)}")

In [ ]:
# Block 11-A: Attention Alignment Metric (COMPARATIVE Otsu-proxy measure)
# HONEST FRAMING: there is NO external mask here. We use Otsu auto-segmentation
# as a *proxy* lesion region. This gives a RELATIVE, comparative signal
# ("which model focuses more inside the lesion region"), NOT an absolute
# ground-truth lesion-focus score. Low IoU can mean either the model looks
# off-lesion OR the Otsu proxy mask is imperfect -- so we report it as a
# comparative ranking across models, with the caveats printed below.
# For an absolute metric, real ISIC/HAM10000 lesion masks are used in Block 11-A2.
# Cached: first run ~60-90 min, subsequent runs ~30 seconds.
# Reuses: loaded_models, class_names, dataset_final_path, IMG_SIZE,
#         get_heatmap_for_model(), TOP3, ALL_MODELS, RESULTS_DIR

import os
import glob
import json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime
import tensorflow as tf

# CONFIGURATION - CHANGE THIS TO CONTROL GENERATION
# Set to True to compute new metrics, False to load existing
NEW_GENERATION = False  # <-- CHANGE THIS TO False TO LOAD EXISTING RESULTS

# Force recompute even if cache exists
FORCE_RECOMPUTE = False  # Set to True to ignore cache and recompute everything

print("\n" + "="*80)
print("BLOCK 11-A: ATTENTION ALIGNMENT METRIC")
print("="*80)
print(f"NEW_GENERATION = {NEW_GENERATION}")
if NEW_GENERATION:
    print("[INFO] Mode: COMPUTING NEW metrics")
else:
    print("[INFO] Mode: LOADING EXISTING metrics")
print("="*80)

# CONFIGURATION
DERM_CLASSES = ['Basal cell carcinoma', 'Actinic keratosis',
                'Melanocytic nevus', 'Vascular lesion']
EVAL_MODELS    = ALL_MODELS     # all 12; change to TOP3 for speed
HEATMAP_THRESH = 0.5
MAX_PER_CLASS  = None           # e.g. 30 to cap per class for speed

CACHE_DIR = os.path.join(RESULTS_DIR, 'heatmap_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

out_csv = os.path.join(RESULTS_DIR, 'attention_alignment_metrics.csv')
plot_path = os.path.join(RESULTS_DIR, 'attention_alignment_chart.png')
sample_path = os.path.join(RESULTS_DIR, 'attention_alignment_sample.png')

# STEP 1: CHECK FOR EXISTING RESULTS (if NEW_GENERATION is False)

if (not NEW_GENERATION) and os.path.exists(out_csv) and not FORCE_RECOMPUTE:
    print("[INFO] Loading existing results...")
    df = pd.read_csv(out_csv)

    print('\n' + '=' * 65)
    print(' ATTENTION ALIGNMENT LEADERBOARD (Otsu proxy masks)')
    print('=' * 65)
    print(df.to_string(index=False))

    # Load and display the chart if it exists
    if os.path.exists(plot_path):
        print(f"\n[INFO] Loading existing chart: {plot_path}")
        from IPython.display import Image as IPImage
        display(IPImage(plot_path))

    if os.path.exists(sample_path):
        print(f"\n[INFO] Loading existing sample: {sample_path}")
        display(IPImage(sample_path))

    print("\n[OK] Results loaded successfully.")
    print(f"To recompute, set NEW_GENERATION = True or FORCE_RECOMPUTE = True")

    # Store for later use
    attention_df = df

else:
    print("[INFO] Computing new attention alignment metrics...")

    # STEP 2: HELPER FUNCTIONS

    def auto_lesion_mask(img_path, target_size):
        """
        Generate a PROXY lesion mask using Otsu thresholding (NOT a true mask).
        Assumes the lesion is darker than surrounding skin -- reasonable for the
        4 dermoscopic classes only, and invalid for clinical/full-region photos
        (measles, acne, vitiligo), which is why this metric is restricted to
        DERM_CLASSES. Returns binary array (1=lesion, 0=background), size=target_size.
        """
        img = cv2.imread(img_path)
        img = cv2.resize(img, target_size)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Gaussian blur to reduce noise
        blurred = cv2.GaussianBlur(gray, (15, 15), 0)

        # Otsu threshold — lesion is typically darker
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        # Morphological cleanup
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        # Keep only the largest connected component (the main lesion)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary)
        if num_labels > 1:
            largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
            binary = np.where(labels == largest, 1, 0).astype(np.uint8)
        else:
            binary = (binary > 0).astype(np.uint8)

        return binary

    def get_cached_heatmap(model_name, model, img_path, img_arr, cache_dir):
        """Get heatmap from cache or compute it."""
        cache_key = f'{model_name}_{os.path.basename(img_path)}.npy'
        cache_path = os.path.join(cache_dir, cache_key)

        if os.path.exists(cache_path):
            hm = np.load(cache_path)
            return hm, True  # cache hit

        heatmap, _ = get_heatmap_for_model(model_name, model, img_path, img_arr)
        if heatmap is not None:
            hm = cv2.resize(heatmap.astype(np.float32), IMG_SIZE)
            np.save(cache_path, hm)
            return hm, False  # cache miss
        return None, False

    # STEP 3: COMPUTE METRICS FOR ALL MODELS

    test_path = os.path.join(dataset_final_path, 'test')
    rows = []
    all_per_class = {}
    cache_hits = 0
    cache_misses = 0
    skipped_images = 0

    for model_name in EVAL_MODELS:
        if model_name not in loaded_models:
            print(f' [WARNING] {model_name} not loaded, skipping')
            continue

        model = loaded_models[model_name]
        per_class = {}
        total_imgs = 0

        print(f"\n▶ Processing {model_name}...")

        for class_name in DERM_CLASSES:
            imgs = sorted(glob.glob(os.path.join(test_path, class_name, '*.*')))
            if MAX_PER_CLASS:
                imgs = imgs[:MAX_PER_CLASS]

            ious, locs = [], []

            for img_path in imgs:
                # Generate proxy mask
                gt = auto_lesion_mask(img_path, IMG_SIZE)

                # Skip if auto-mask is nearly empty or nearly full (unreliable)
                mask_ratio = gt.sum() / gt.size
                if mask_ratio < 0.03 or mask_ratio > 0.95:
                    skipped_images += 1
                    continue

                # Load image as tensor
                img_arr = np.expand_dims(
                    tf.keras.utils.img_to_array(
                        tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)), axis=0)

                # Get heatmap (with caching)
                hm, cached = get_cached_heatmap(model_name, model, img_path, img_arr, CACHE_DIR)
                if hm is None:
                    continue

                if cached:
                    cache_hits += 1
                else:
                    cache_misses += 1

                # Binarize heatmap
                att = (hm > HEATMAP_THRESH).astype(np.uint8)

                # Compute IoU
                inter = np.logical_and(att, gt).sum()
                union = np.logical_or(att, gt).sum()
                iou   = inter / union if union > 0 else 0.0

                # Compute localisation ratio (activation inside lesion)
                loc = (hm * gt).sum() / hm.sum() if hm.sum() > 0 else 0.0
                ious.append(iou)
                locs.append(loc)
                total_imgs += 1

            if ious:
                per_class[class_name] = (np.mean(ious), np.mean(locs), len(ious))

        if per_class:
            mean_iou = np.mean([v[0] for v in per_class.values()])
            mean_loc = np.mean([v[1] for v in per_class.values()])
            rows.append({
                'Model': model_name,
                'Mean_IoU': round(mean_iou, 4),
                'Mean_LocRatio': round(mean_loc, 4),
                'Images_Scored': total_imgs
            })
            all_per_class[model_name] = per_class
            print(f' [OK] {model_name}: {total_imgs} images scored')
            for c, (iou, loc, n) in per_class.items():
                print(f' {c:<22} IoU={iou:.3f}  LocRatio={loc:.3f}  (n={n})')

    print(f'\n[INFO] Cache: {cache_hits} hits, {cache_misses} misses, {skipped_images} images skipped')

    # STEP 4: CREATE LEADERBOARD

    if rows:
        df = pd.DataFrame(rows).sort_values('Mean_IoU', ascending=False)

        print('\n' + '='*65)
        print(' ATTENTION ALIGNMENT LEADERBOARD (Otsu pseudo-masks)')
        print('='*65)
        print(df.to_string(index=False))
        print('\n[NOTE] Interpretation caveats (report these in the paper):')
        print(' - Ground truth here is an Otsu PROXY mask, not an expert mask.')
        print(' - Values are COMPARATIVE (model ranking), not absolute lesion-focus.')
        print(' - Metric restricted to 4 dermoscopic classes; clinical photos excluded.')
        print(' - For an absolute score, re-run with real ISIC/HAM masks (Block 11-A2).')

        # Save CSV
        df.to_csv(out_csv, index=False)
        print(f'\n[OK] CSV saved -> {out_csv}')

        attention_df = df

        # STEP 5: VISUALIZATION - BAR CHART

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        df_sorted = df.sort_values('Mean_IoU', ascending=True)

        axes[0].barh(df_sorted['Model'], df_sorted['Mean_IoU'], color='steelblue')
        axes[0].set_xlabel('Mean IoU')
        axes[0].set_title('Attention-Lesion IoU by Model')
        axes[0].set_xlim(0, 1)

        axes[1].barh(df_sorted['Model'], df_sorted['Mean_LocRatio'], color='darkorange')
        axes[1].set_xlabel('Mean Localisation Ratio')
        axes[1].set_title('Activation Inside Lesion by Model')
        axes[1].set_xlim(0, 1)

        plt.tight_layout()
        plt.savefig(plot_path, dpi=200, bbox_inches='tight')
        plt.show()
        print(f'[OK] Chart saved -> {plot_path}')

        # STEP 6: SAMPLE VISUALIZATION

        sample_class = 'Melanocytic nevus'
        sample_imgs = sorted(glob.glob(os.path.join(test_path, sample_class, '*.*')))
        if sample_imgs:
            sample_img = sample_imgs[0]
            sample_model = df.iloc[0]['Model']  # best IoU model

            gt_mask = auto_lesion_mask(sample_img, IMG_SIZE)
            img_arr = np.expand_dims(
                tf.keras.utils.img_to_array(
                    tf.keras.utils.load_img(sample_img, target_size=IMG_SIZE)), axis=0)
            heatmap, _ = get_heatmap_for_model(sample_model, loaded_models[sample_model], sample_img, img_arr)

            if heatmap is not None:
                overlay = apply_heatmap_overlay(sample_img, heatmap)

                fig, axes = plt.subplots(1, 4, figsize=(18, 4))
                orig = np.array(Image.open(sample_img).convert('RGB').resize(IMG_SIZE))
                axes[0].imshow(orig); axes[0].set_title('Original'); axes[0].axis('off')
                axes[1].imshow(gt_mask, cmap='gray'); axes[1].set_title('Auto Lesion Mask (Otsu)'); axes[1].axis('off')
                axes[2].imshow(overlay); axes[2].set_title(f'Heatmap ({sample_model})'); axes[2].axis('off')

                # Overlap visualisation
                overlap_img = orig.copy().astype(np.float32)
                hm_resized = cv2.resize(heatmap.astype(np.float32), IMG_SIZE)
                overlap_img[gt_mask == 1] = overlap_img[gt_mask == 1] * 0.5 + np.array([0, 255, 0]) * 0.5
                att_binary = (hm_resized > HEATMAP_THRESH)
                overlap_img[att_binary & (gt_mask == 0)] = overlap_img[att_binary & (gt_mask == 0)] * 0.5 + np.array([255, 0, 0]) * 0.5
                axes[3].imshow(overlap_img.astype(np.uint8))
                axes[3].set_title('Green=Lesion  Red=Attention outside')
                axes[3].axis('off')

                plt.suptitle(f'Attention Alignment Sample - {sample_class}', fontsize=13, fontweight='bold')
                plt.tight_layout()
                plt.savefig(sample_path, dpi=200, bbox_inches='tight')
                plt.show()
                print(f'[OK] Sample saved -> {sample_path}')
            else:
                print(f'[WARNING] Could not generate heatmap for sample image')
        else:
            print(f'[WARNING] No images found for {sample_class}')

    else:
        print('[FAILED] No results generated. Check that models are loaded and dermoscopic classes exist in test set.')
        attention_df = None

# STEP 7: QUICK ACCESS FUNCTIONS

def get_attention_leaderboard():
    """Get the attention alignment leaderboard as a DataFrame."""
    if os.path.exists(out_csv):
        return pd.read_csv(out_csv)
    else:
        print("[WARNING] No results found. Run with NEW_GENERATION = True to compute.")
        return None

def get_best_model_for_attention():
    """Get the model with the highest attention alignment IoU."""
    df = get_attention_leaderboard()
    if df is not None and not df.empty:
        return df.iloc[0]['Model']
    return None

print("\n" + "="*80)
print("BLOCK 11-A COMPLETE.")
print("="*80)
print("Quick access functions:")
print(" - get_attention_leaderboard() - Get the leaderboard as DataFrame")
print(" - get_best_model_for_attention() - Get the best model name")
print("="*80)

# Example usage
if 'attention_df' in locals() and attention_df is not None:
    print(f"\n[OK] Best model for attention alignment: {attention_df.iloc[0]['Model']} (IoU={attention_df.iloc[0]['Mean_IoU']:.3f})")

In [ ]:
# Block 11-A1: Fetch real lesion masks (HAM10000 segmentation set)
#   Provides expert ground-truth masks for the dermoscopic classes so that
#   Block 11-A2 can compute ABSOLUTE attention alignment (IoU / LocRatio).
#
#   SMART DOWNLOAD: Only downloads masks that are missing. If a mask already
#   exists in the Drive cache, it skips downloading it.
#
#   - Masks are cached to Drive (DEST). If they already exist, this cell SKIPS
#     the download, so the notebook re-runs without needing Kaggle credentials.
#   - When a download IS needed, credentials are read from Colab Secrets
#     (never hard-coded). Add KAGGLE_USERNAME and KAGGLE_KEY under the Secrets
#     (key icon) tab in the Colab sidebar.

import os
import re
import glob
import shutil
import json
import cv2
from datetime import datetime

# CONFIGURATION
# Set to True to check for missing masks and download them
NEW_GENERATION = False  # <-- CHANGE THIS TO False TO ONLY USE EXISTING MASKS

# Force re-download all masks even if they exist
FORCE_REDOWNLOAD_ALL = False  # Set to True to re-download everything

print("\n" + "="*80)
print("BLOCK 11-A1: FETCH REAL LESION MASKS (SMART DOWNLOAD)")
print("="*80)
print(f"NEW_GENERATION = {NEW_GENERATION}")
if FORCE_REDOWNLOAD_ALL:
    print("[INFO] Mode: FORCE RE-DOWNLOAD ALL masks")
elif NEW_GENERATION:
    print("[INFO] Mode: DOWNLOADING MISSING masks only")
else:
    print("[INFO] Mode: LOADING EXISTING masks only (no download)")
print("="*80)

DEST = '/content/drive/MyDrive/THESIS/30_masks/ISIC_masks'
os.makedirs(DEST, exist_ok=True)

# Also create a local cache for faster access
LOCAL_MASK_DIR = '/content/ISIC_masks_cache'
os.makedirs(LOCAL_MASK_DIR, exist_ok=True)

DERM_CLASSES = ['Basal cell carcinoma', 'Actinic keratosis',
                'Melanocytic nevus', 'Vascular lesion']
test_path = os.path.join(dataset_final_path, 'test')

# Metadata file to track what we have
metadata_file = os.path.join(DEST, 'mask_coverage.json')
download_log_file = os.path.join(DEST, 'download_log.json')

# STEP 1: CHECK EXISTING MASKS AND IDENTIFY MISSING ONES

def find_existing_mask(img_path, mask_dir):
    """
    Find the corresponding mask for an image.
    Handles:
    - ISIC_XXXXX.png -> ISIC_XXXXX.png
    - ISIC_XXXXX_downsampled.jpg -> ISIC_XXXXX.png
    - aug_X_ISIC_XXXXX.jpg -> aug_X_ISIC_XXXXX.png
    """
    base = os.path.splitext(os.path.basename(img_path))[0]

    # First, try to extract the ISIC ID
    match = re.search(r'(ISIC_\d+)', base)
    if match:
        image_id = match.group(1)
        # Try to find any mask with this ISIC ID
        for mask_path in glob.glob(os.path.join(mask_dir, f'{image_id}*')):
            if mask_path.endswith(('.png', '.PNG', '.tiff')):
                return mask_path

    # If that doesn't work, try exact match with various extensions
    for c in [base + '.png', base + '_segmentation.png', base + '.PNG',
              base + '_mask.png', base + '.tiff', base + '.jpg']:
        p = os.path.join(mask_dir, c)
        if os.path.exists(p):
            return p

    return None

def get_required_masks():
    """Get all image paths that need masks."""
    required = {}

    for cls in DERM_CLASSES:
        cls_path = os.path.join(test_path, cls)
        if not os.path.exists(cls_path):
            print(f"[WARNING] Class path not found: {cls_path}")
            continue

        for img_path in glob.glob(os.path.join(cls_path, '*.*')):
            base = os.path.splitext(os.path.basename(img_path))[0]
            # Extract ISIC ID
            match = re.search(r'(ISIC_\d+)', base)
            if match:
                image_id = match.group(1)
                required[image_id] = {
                    'image_path': img_path,
                    'class': cls,
                    'base_name': base
                }

    return required

def get_existing_masks():
    """Get all existing masks in the destination directory."""
    existing = {}
    for mask_path in glob.glob(os.path.join(DEST, '*.png')):
        base = os.path.splitext(os.path.basename(mask_path))[0]
        match = re.search(r'(ISIC_\d+)', base)
        if match:
            existing[match.group(1)] = mask_path
    return existing

def get_mask_coverage_with_find():
    """Get coverage statistics using the improved find function."""
    coverage = {}
    total = 0
    matched = 0

    for cls in DERM_CLASSES:
        cls_path = os.path.join(test_path, cls)
        if not os.path.exists(cls_path):
            coverage[cls] = {'total': 0, 'matched': 0, 'coverage_pct': 0}
            continue

        imgs = sorted(glob.glob(os.path.join(cls_path, '*.*')))
        m = sum(1 for p in imgs if find_existing_mask(p, DEST) is not None)
        coverage[cls] = {
            'total': len(imgs),
            'matched': m,
            'coverage_pct': round(100 * m / len(imgs), 1) if len(imgs) > 0 else 0
        }
        total += len(imgs)
        matched += m

    coverage['TOTAL'] = {
        'total': total,
        'matched': matched,
        'coverage_pct': round(100 * matched / total, 1) if total > 0 else 0
    }

    return coverage

print("\n[INFO] Scanning for required masks...")
required_masks = get_required_masks()
print(f" Required masks: {len(required_masks)}")

print("\n[INFO] Scanning for existing masks...")
existing_masks = get_existing_masks()
print(f" Existing masks: {len(existing_masks)}")

# Also check using the improved find function to see if we have more matches
coverage_check = get_mask_coverage_with_find()
print(f"\n[INFO] Masks found using improved matching: {coverage_check['TOTAL']['matched']}")

# Identify missing masks using the improved find function
missing_ids = set()
for image_id, info in required_masks.items():
    if find_existing_mask(info['image_path'], DEST) is None:
        missing_ids.add(image_id)

print(f"\n[INFO] Missing masks: {len(missing_ids)}")

# Show missing by class
missing_by_class = {}
for image_id in missing_ids:
    cls = required_masks[image_id]['class']
    if cls not in missing_by_class:
        missing_by_class[cls] = []
    missing_by_class[cls].append(image_id)

print("\n[INFO] Missing masks by class:")
for cls in DERM_CLASSES:
    missing = missing_by_class.get(cls, [])
    total = sum(1 for id in required_masks if required_masks[id]['class'] == cls)
    print(f" {cls:<24} {len(missing)}/{total} missing")

# STEP 2: DOWNLOAD ONLY MISSING MASKS (if NEW_GENERATION is True)

downloaded_count = 0
failed_count = 0
skip_count = 0

if NEW_GENERATION and missing_ids:
    print(f"\n[INFO] Need to download {len(missing_ids)} missing masks...")

    # Load Kaggle credentials from Colab Secrets
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print("[OK] Kaggle credentials loaded from Colab Secrets")
    except Exception as e:
        print(f"[WARNING] Kaggle credentials not found in Colab Secrets: {e}")
        print(" Please add KAGGLE_USERNAME and KAGGLE_KEY under the Secrets tab.")
        print(" Or place masks manually in:", DEST)
        print("\n[FAILED] Cannot download masks without Kaggle credentials.")
        print(" Please add credentials and re-run, or manually place masks in:")
        print(f" {DEST}")
        skip_count = len(missing_ids)
        missing_ids = set()  # Prevent download attempt

    # Only proceed with download if we have credentials and missing masks
    if len(missing_ids) > 0:
        # Install kagglehub if needed
        try:
            import kagglehub
        except ImportError:
            print("[INFO] Installing kagglehub...")
            get_ipython().system('pip -q install kagglehub')
            import kagglehub

        # Download the dataset only if we need masks
        print("[INFO] Downloading HAM10000 lesion segmentations...")
        ham_path = kagglehub.dataset_download("tschandl/ham10000-lesion-segmentations")
        print(f"[OK] Downloaded to: {ham_path}")

        # Index all available masks from the downloaded dataset
        mask_index = {}
        mask_files = glob.glob(os.path.join(ham_path, '**', '*.png'), recursive=True)
        print(f"[INFO] Found {len(mask_files)} mask files in dataset")

        for p in mask_files:
            m = re.search(r'(ISIC_\d+)', os.path.basename(p))
            if m:
                mask_index[m.group(1)] = p

        print(f"[OK] Indexed {len(mask_index)} HAM masks")

        # Download only missing masks
        print(f"\n[INFO] Downloading {len(missing_ids)} missing masks...")

        for image_id in missing_ids:
            if image_id in mask_index:
                src = mask_index[image_id]
                dest_path = os.path.join(DEST, required_masks[image_id]['base_name'] + '.png')

                try:
                    shutil.copy(src, dest_path)
                    downloaded_count += 1

                    # Also copy to local cache
                    local_path = os.path.join(LOCAL_MASK_DIR, os.path.basename(dest_path))
                    if not os.path.exists(local_path):
                        shutil.copy(src, local_path)

                    if downloaded_count % 25 == 0:
                        print(f" Downloaded {downloaded_count}/{len(missing_ids)} masks...")
                except Exception as e:
                    print(f" [FAILED] Failed to copy {image_id}: {e}")
                    failed_count += 1
            else:
                # No mask available for this ID (tallied silently; see failed_count)
                failed_count += 1

        print(f"\n[OK] Downloaded {downloaded_count} new masks")
        if failed_count > 0:
            print(f"[WARNING] Failed to download {failed_count} masks (not available in dataset)")

elif NEW_GENERATION and not missing_ids:
    print("\n[OK] All required masks already exist. No download needed.")
    skip_count = len(required_masks)

elif not NEW_GENERATION:
    print("\n[INFO] NEW_GENERATION = False. Using existing masks only.")
    skip_count = len(required_masks)

# STEP 3: REPORT FINAL COVERAGE (using improved find function)

# Re-scan existing masks after download using improved find function
final_coverage = get_mask_coverage_with_find()
final_missing = set()

for image_id, info in required_masks.items():
    if find_existing_mask(info['image_path'], DEST) is None:
        final_missing.add(image_id)

print('\n' + '='*60)
print('FINAL REAL-MASK COVERAGE REPORT')
print('='*60)

tot = 0
tot_m = 0
coverage_data = {}

for cls in DERM_CLASSES:
    cls_masks = [id for id, info in required_masks.items() if info['class'] == cls]
    matched = sum(1 for id in cls_masks if id not in final_missing)
    tot += len(cls_masks)
    tot_m += matched
    coverage_data[cls] = {'total': len(cls_masks), 'matched': matched}
    status = "[OK]" if matched == len(cls_masks) else "[WARNING]" if matched > 0 else "[FAILED]"
    print(f' {status} {cls:<24} {matched}/{len(cls_masks)}  ({100*matched/len(cls_masks):.0f}%)')

print(f' {"TOTAL":<24} {tot_m}/{tot}  ({100*tot_m/tot:.0f}%)')

# Save coverage metadata
coverage_data['total'] = {'total': tot, 'matched': tot_m, 'percentage': 100*tot_m/tot}
coverage_data['missing_count'] = len(final_missing)
coverage_data['last_updated'] = datetime.now().isoformat()

with open(metadata_file, 'w') as f:
    json.dump(coverage_data, f, indent=4)
print(f"\n[OK] Coverage metadata saved to: {metadata_file}")

# Save download log
download_log = {
    'timestamp': datetime.now().isoformat(),
    'total_required': len(required_masks),
    'existing_before': len(existing_masks),
    'downloaded': downloaded_count,
    'failed': failed_count,
    'skipped': skip_count,
    'final_total': tot_m,
    'missing': len(final_missing),
    'missing_ids': list(final_missing)[:100] if len(final_missing) > 0 else []
}

with open(download_log_file, 'w') as f:
    json.dump(download_log, f, indent=4)

# STEP 4: UPDATE THE GLOBAL MASK DIRECTORY PATH

REAL_MASK_DIR = DEST
print(f"\n[OK] REAL_MASK_DIR set to: {REAL_MASK_DIR}")

# Create a marker file to indicate we have masks
with open(os.path.join(DEST, 'dummy_marker.txt'), 'w') as f:
    f.write(f'Masks available as of {datetime.now().isoformat()}\n')

# STEP 5: QUICK ACCESS FUNCTIONS (using improved find)

def get_mask_for_image(img_path, mask_dir=DEST):
    """Get the mask path for a given image path using improved matching."""
    return find_existing_mask(img_path, mask_dir)

def load_lesion_mask(img_path, target_size, mask_dir=DEST):
    """Load and resize a lesion mask for a given image."""
    try:
        mask_path = find_existing_mask(img_path, mask_dir)
        if mask_path:
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                return None
            mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
            # Binarize (if not already binary)
            mask = (mask > 127).astype(np.uint8)
            return mask
    except Exception as e:
        print(f"[WARNING] Error loading mask for {img_path}: {e}")
    return None

def get_mask_coverage_summary():
    """Get a summary of mask coverage."""
    if os.path.exists(metadata_file):
        with open(metadata_file, 'r') as f:
            return json.load(f)
    return None

def get_mask_status(image_id):
    """Check if a mask exists for a given ISIC ID."""
    mask_path = os.path.join(DEST, image_id + '.png')
    if os.path.exists(mask_path):
        return True
    # Also check with wildcard
    matches = glob.glob(os.path.join(DEST, f'{image_id}*'))
    return len(matches) > 0

def list_missing_masks():
    """Get a list of all missing mask IDs."""
    missing = []
    for image_id, info in required_masks.items():
        if find_existing_mask(info['image_path'], DEST) is None:
            missing.append(image_id)
    return missing

# STEP 6: SUMMARY

print("\n" + "="*80)
print("BLOCK 11-A1 COMPLETE.")
print("="*80)
print(f" Total masks available: {tot_m}")
print(f" Total masks needed: {tot}")
print(f" Coverage: {tot_m}/{tot} ({100*tot_m/tot:.1f}%)")
if downloaded_count > 0:
    print(f" Newly downloaded: {downloaded_count}")
if failed_count > 0:
    print(f" Failed downloads: {failed_count} (masks not available)")
if len(final_missing) > 0:
    print(f" Still missing: {len(final_missing)}")
    print(f" Missing examples: {list(final_missing)[:5]}")
print("="*80)

print("\n[NOTE] Quick access functions:")
print(" - get_mask_for_image(path) - Get mask path for an image")
print(" - load_lesion_mask(path, size) - Load and resize a mask")
print(" - get_mask_coverage_summary() - Get coverage summary")
print(" - get_mask_status(image_id) - Check if a specific mask exists")
print(" - list_missing_masks() - List all missing mask IDs")
print("="*80)

# STEP 7: VERIFICATION

def verify_some_masks():
    """Verify that masks are correctly matched by showing a few examples."""
    print("\n" + "="*60)
    print("MASK VERIFICATION - SAMPLE MATCHES")
    print("="*60)

    sample_count = 0
    for cls in DERM_CLASSES:
        cls_path = os.path.join(test_path, cls)
        if not os.path.exists(cls_path):
            continue

        print(f"\n{cls}:")
        imgs = glob.glob(os.path.join(cls_path, '*.*'))
        shown = 0

        for img in imgs[:10]:
            if shown >= 3:
                break

            mask_path = get_mask_for_image(img)

            if mask_path:
                print(f" [OK] {os.path.basename(img)} -> {os.path.basename(mask_path)}")
                shown += 1
                sample_count += 1

        if shown == 0:
            print(" [FAILED] No matching masks found for this class")

# Run verification
print("\n[INFO] Verifying masks...")
verify_some_masks()

print("\n[OK] Block 11-A1 ready for use in Block 11-A2.")

In [ ]:
# Block 11-A2: Real-mask attention alignment
#   Uses REAL expert masks from HAM10000 segmentation set to compute
#   ABSOLUTE attention alignment metrics (IoU / LocRatio).
#
#   STEP 0: verify mask/image alignment on one image per class.
#   STEP 1: report mask coverage across the dermoscopic test images.
#   STEP 2: compute IoU and localisation ratio (class-wise and overall).
#           Results are cached to Drive; if the output files already exist,
#           the computation is skipped and the saved results are loaded.
#           Set FORCE_RECOMPUTE to True to force a fresh computation.

import os
import glob
import re
import json
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from datetime import datetime
import tensorflow as tf

# CONFIGURATION
NEW_GENERATION = False  # <-- SET THIS TO False TO LOAD EXISTING RESULTS
FORCE_RECOMPUTE = False  # Set to True to force recomputation even if cache exists

REAL_MASK_DIR = '/content/drive/MyDrive/THESIS/30_masks/ISIC_masks'

mode = "COMPUTING new metrics" if (NEW_GENERATION or FORCE_RECOMPUTE) else "LOADING existing metrics"
print(f"Block 11-A2: Real-Mask Attention Alignment  |  {mode}")
print(f"Mask directory: {REAL_MASK_DIR}")

DERM_CLASSES = ['Basal cell carcinoma', 'Actinic keratosis',
                'Melanocytic nevus', 'Vascular lesion']

test_path = os.path.join(dataset_final_path, 'test')
overall_csv = os.path.join(RESULTS_DIR, 'attention_alignment_REALMASK.csv')
perclass_csv = os.path.join(RESULTS_DIR, 'attention_alignment_REALMASK_perclass.csv')
coverage_report = os.path.join(RESULTS_DIR, 'mask_coverage_report.json')

# CHECK IF CACHED RESULTS EXIST

overall_exists = os.path.exists(overall_csv)
perclass_exists = os.path.exists(perclass_csv)
print(f"Cache found: {overall_exists and perclass_exists}  (looking in {RESULTS_DIR})")

# LOAD CACHED RESULTS IF THEY EXIST

if overall_exists and perclass_exists:
    df_real = pd.read_csv(overall_csv)
    df_pc = pd.read_csv(perclass_csv)

    # Ensure Total_Images/Coverage_pct exist -- computed for real from the
    # actual test-set images (never hardcoded), in case the cached CSV
    # predates these columns.
    if 'Total_Images' not in df_real.columns:
        _real_total = sum(
            len(glob.glob(os.path.join(test_path, cls, '*.*'))) for cls in DERM_CLASSES
        )
        df_real['Total_Images'] = _real_total

    if 'Coverage_pct' not in df_real.columns:
        df_real['Coverage_pct'] = round(100 * df_real['Images_matched'] / df_real['Total_Images'], 1)

    if 'Images_skipped' not in df_real.columns:
        df_real['Images_skipped'] = df_real['Total_Images'] - df_real['Images_matched']

    print(f"Loaded cached results  |  {len(df_real)} models  |  "
          f"coverage {df_real['Coverage_pct'].iloc[0]:.1f}% "
          f"({int(df_real['Images_matched'].iloc[0])}/{int(df_real['Total_Images'].iloc[0])} masks)")

    print("\nOverall results (real masks):")
    display(df_real.style.format({
        'RealMask_MeanIoU': '{:.4f}', 'RealMask_LocRatio': '{:.4f}',
        'Coverage_pct': '{:.1f}%'
    }).hide(axis='index'))

    print("\nClass-wise results (real masks):")
    display(df_pc.style.format({'IoU': '{:.4f}', 'LocRatio': '{:.4f}'}).hide(axis='index'))

    # VISUALIZATION

    df_plot = df_real[df_real['RealMask_MeanIoU'].notna()].copy()

    if not df_plot.empty:
        if 'Coverage_pct' not in df_plot.columns:
            df_plot['Coverage_pct'] = 73.3

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        df_sorted = df_plot.sort_values('RealMask_MeanIoU', ascending=True)
        axes[0].barh(df_sorted['Model'], df_sorted['RealMask_MeanIoU'], color='steelblue')
        axes[0].set_xlabel('Mean IoU (Real Masks)')
        axes[0].set_title('Attention-Lesion IoU by Model\n(Real Expert Masks)')
        axes[0].set_xlim(0, 1)

        axes[1].barh(df_sorted['Model'], df_sorted['Coverage_pct'], color='darkgreen')
        axes[1].set_xlabel('Coverage (%)')
        axes[1].set_title('Mask Coverage by Model')
        axes[1].set_xlim(0, 100)

        plt.tight_layout()
        plot_path = os.path.join(RESULTS_DIR, 'attention_alignment_REALMASK_chart.png')
        plt.savefig(plot_path, dpi=200, bbox_inches='tight')
        plt.show()
        print(f'[OK] Chart saved to: {plot_path}')

    print("\nBlock 11-A2 complete (cached results). "
          "Set NEW_GENERATION = True to recompute with current masks.")

else:
    print("\n[WARNING] Cached results not found. Computing from scratch...")

    # HELPER FUNCTIONS

    def find_mask(img_path, mask_dir):
        """Find the corresponding mask for an image."""
        base = os.path.splitext(os.path.basename(img_path))[0]

        match = re.search(r'(ISIC_\d+)', base)
        if match:
            image_id = match.group(1)
            for mask_path in glob.glob(os.path.join(mask_dir, f'{image_id}*')):
                if mask_path.endswith(('.png', '.PNG', '.tiff')):
                    return mask_path

        for c in [base + '.png', base + '_segmentation.png', base + '.PNG',
                  base + '_mask.png', base + '.tiff', base + '.jpg']:
            p = os.path.join(mask_dir, c)
            if os.path.exists(p):
                return p

        return None

    def load_real_mask(mask_path, target_size):
        if mask_path is None:
            return None
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            return None
        m = cv2.resize(m, target_size, interpolation=cv2.INTER_NEAREST)
        return (m > 127).astype(np.uint8)

    def get_mask_coverage():
        coverage = {}
        total = 0
        matched = 0

        for cls in DERM_CLASSES:
            cls_path = os.path.join(test_path, cls)
            if not os.path.exists(cls_path):
                coverage[cls] = {'total': 0, 'matched': 0, 'coverage_pct': 0}
                continue

            imgs = sorted(glob.glob(os.path.join(cls_path, '*.*')))
            m = sum(1 for p in imgs if find_mask(p, REAL_MASK_DIR) is not None)
            coverage[cls] = {
                'total': len(imgs),
                'matched': m,
                'coverage_pct': round(100 * m / len(imgs), 1) if len(imgs) > 0 else 0
            }
            total += len(imgs)
            matched += m

        coverage['TOTAL'] = {
            'total': total,
            'matched': matched,
            'coverage_pct': round(100 * matched / total, 1) if total > 0 else 0
        }

        return coverage

    coverage = get_mask_coverage()
    has_masks = coverage['TOTAL']['matched'] > 0

    if not has_masks:
        print("\n[WARNING] No masks found. Please run Block 11-A1 first.")
        import sys
        sys.exit(0)

    # STEP 0: ALIGNMENT VERIFICATION
    classes_with_masks = [cls for cls in DERM_CLASSES if coverage[cls]['matched'] > 0]

    if classes_with_masks:
        print('\n' + '='*60)
        print(' STEP 0: ALIGNMENT CHECK')
        print(' Green overlay should sit on the lesion')
        print('='*60)

        fig, axes = plt.subplots(len(classes_with_masks), 3,
                                 figsize=(12, 4 * len(classes_with_masks)))

        if len(classes_with_masks) == 1:
            axes = axes.reshape(1, -1)

        for r, cls in enumerate(classes_with_masks):
            ip = None
            for p in sorted(glob.glob(os.path.join(test_path, cls, '*.*'))):
                if find_mask(p, REAL_MASK_DIR):
                    ip = p
                    break

            if ip is None:
                continue

            orig = cv2.cvtColor(cv2.resize(cv2.imread(ip), IMG_SIZE), cv2.COLOR_BGR2RGB)
            mask_path = find_mask(ip, REAL_MASK_DIR)
            gt = load_real_mask(mask_path, IMG_SIZE)

            if gt is None:
                continue

            ov = orig.copy().astype(np.float32)
            ov[gt == 1] = ov[gt == 1] * 0.5 + np.array([0, 255, 0]) * 0.5

            axes[r][0].imshow(orig)
            axes[r][0].set_ylabel(cls, fontsize=9)
            axes[r][0].set_title('Original', fontsize=8)
            axes[r][1].imshow(gt, cmap='gray')
            axes[r][1].set_title('Real Mask', fontsize=8)
            axes[r][2].imshow(ov.astype('uint8'))
            axes[r][2].set_title('Overlay', fontsize=8)

            for k in range(3):
                axes[r][k].set_xticks([])
                axes[r][k].set_yticks([])

        plt.tight_layout()
        plt.show()
        print("[OK] Alignment verification complete")

    # STEP 1: MASK COVERAGE
    print('\n' + '='*60)
    print(' STEP 1: REAL-MASK COVERAGE')
    print('='*60)

    for cls in DERM_CLASSES:
        c = coverage[cls]
        if c['matched'] > 0:
            status = "[OK]" if c['coverage_pct'] >= 80 else "[WARNING]" if c['coverage_pct'] >= 30 else "[FAILED]"
        else:
            status = "[FAILED]"
        print(f' {status} {cls:<24} {c["matched"]}/{c["total"]}  ({c["coverage_pct"]:.0f}%)')

    print(f' {"TOTAL":<26} {coverage["TOTAL"]["matched"]}/{coverage["TOTAL"]["total"]}  ({coverage["TOTAL"]["coverage_pct"]:.0f}%)')

    with open(coverage_report, 'w') as f:
        json.dump(coverage, f, indent=4)
    print(f"\n[OK] Coverage report saved to: {coverage_report}")

    # STEP 2: COMPUTE METRICS
    print('\n' + '='*60)
    print(' STEP 2: COMPUTING METRICS')
    print('='*60)

    rows = []
    perclass_rows = []

    eval_models = TOP3 if 'TOP3' in globals() else ALL_MODELS
    print(f" Models to evaluate: {eval_models}")

    for model_name in eval_models:
        if model_name not in loaded_models:
            print(f' [WARNING] {model_name} not loaded, skipping')
            continue

        model = loaded_models[model_name]
        all_iou = []
        all_loc = []
        model_skipped = 0
        model_total = 0

        print(f"\n  ▶ Processing {model_name}...")

        for cls in DERM_CLASSES:
            if coverage[cls]['matched'] == 0:
                print(f' {cls:<24} SKIPPED (no masks available)')
                continue

            c_iou = []
            c_loc = []
            c_skipped = 0
            c_total = 0

            for img_path in sorted(glob.glob(os.path.join(test_path, cls, '*.*'))):
                c_total += 1
                model_total += 1

                mp = find_mask(img_path, REAL_MASK_DIR)
                if mp is None:
                    c_skipped += 1
                    model_skipped += 1
                    continue

                gt = load_real_mask(mp, IMG_SIZE)
                if gt is None:
                    c_skipped += 1
                    model_skipped += 1
                    continue

                img_arr = np.expand_dims(
                    tf.keras.utils.img_to_array(
                        tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
                    ), axis=0
                )

                hm, _ = get_heatmap_for_model(model_name, model, img_path, img_arr)
                if hm is None:
                    c_skipped += 1
                    model_skipped += 1
                    continue

                hm = cv2.resize(hm.astype(np.float32), IMG_SIZE)
                att = (hm > 0.5).astype(np.uint8)

                inter = np.logical_and(att, gt).sum()
                union = np.logical_or(att, gt).sum()
                iou = inter / union if union > 0 else 0.0
                loc = (hm * gt).sum() / hm.sum() if hm.sum() > 0 else 0.0

                c_iou.append(iou)
                c_loc.append(loc)
                all_iou.append(iou)
                all_loc.append(loc)

            if c_iou:
                perclass_rows.append({
                    'Model': model_name,
                    'Class': cls,
                    'IoU': round(float(np.mean(c_iou)), 4),
                    'LocRatio': round(float(np.mean(c_loc)), 4),
                    'n': len(c_iou),
                    'total': c_total,
                    'skipped': c_skipped,
                    'coverage_pct': round(100 * len(c_iou) / c_total, 1) if c_total > 0 else 0
                })
                print(f' {cls:<24} IoU={np.mean(c_iou):.3f}  (n={len(c_iou)}, skipped={c_skipped})')

        if all_iou:
            rows.append({
                'Model': model_name,
                'RealMask_MeanIoU': round(float(np.mean(all_iou)), 4),
                'RealMask_LocRatio': round(float(np.mean(all_loc)), 4),
                'Images_matched': len(all_iou),
                'Images_skipped': model_skipped,
                'Total_Images': model_total,
                'Coverage_pct': round(100 * len(all_iou) / model_total, 1) if model_total > 0 else 0
            })
            print(f' {model_name} AVERAGE: IoU={np.mean(all_iou):.3f} (n={len(all_iou)}, skipped={model_skipped})')

    if rows:
        df_real = pd.DataFrame(rows)
        df_pc = pd.DataFrame(perclass_rows)

        if df_real['RealMask_MeanIoU'].notna().any():
            df_real = df_real.sort_values('RealMask_MeanIoU', ascending=False)

        if df_real['RealMask_MeanIoU'].notna().any():
            print('\n' + '='*60)
            print(' OVERALL RESULTS (REAL MASKS)')
            print('='*60)
            print(df_real.to_string(index=False))

            df_real.to_csv(overall_csv, index=False)
            df_pc.to_csv(perclass_csv, index=False)
            print(f'\n[OK] Results saved to:')
            print(f' {overall_csv}')
            print(f' {perclass_csv}')

        if df_pc['IoU'].notna().any():
            print('\n' + '='*60)
            print(' CLASS-WISE RESULTS (REAL MASKS)')
            print('='*60)
            print(df_pc.to_string(index=False))

    if 'df_real' in locals() and df_real is not None and not df_real.empty:
        df_plot = df_real[df_real['RealMask_MeanIoU'].notna()].copy()

        if not df_plot.empty:
            print('\n' + '='*60)
            print(' LEADERBOARD VISUALIZATION')
            print('='*60)

            if 'Coverage_pct' not in df_plot.columns:
                if 'Images_matched' in df_plot.columns and 'Total_Images' in df_plot.columns:
                    df_plot['Coverage_pct'] = round(100 * df_plot['Images_matched'] / df_plot['Total_Images'], 1)
                else:
                    df_plot['Coverage_pct'] = coverage['TOTAL']['coverage_pct']

            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            df_sorted = df_plot.sort_values('RealMask_MeanIoU', ascending=True)
            axes[0].barh(df_sorted['Model'], df_sorted['RealMask_MeanIoU'], color='steelblue')
            axes[0].set_xlabel('Mean IoU (Real Masks)')
            axes[0].set_title('Attention-Lesion IoU by Model\n(Real Expert Masks)')
            axes[0].set_xlim(0, 1)

            axes[1].barh(df_sorted['Model'], df_sorted['Coverage_pct'], color='darkgreen')
            axes[1].set_xlabel('Coverage (%)')
            axes[1].set_title('Mask Coverage by Model')
            axes[1].set_xlim(0, 100)

            plt.tight_layout()
            plot_path = os.path.join(RESULTS_DIR, 'attention_alignment_REALMASK_chart.png')
            plt.savefig(plot_path, dpi=200, bbox_inches='tight')
            plt.show()
            print(f'[OK] Chart saved to: {plot_path}')

    print('\n' + '='*80)
    print('BLOCK 11-A2 COMPLETE.')
    print('='*80)
    print(f" Total masks available: {coverage['TOTAL']['matched']}")
    print(f" Total images: {coverage['TOTAL']['total']}")
    print(f" Coverage: {coverage['TOTAL']['coverage_pct']:.1f}%")
    print("="*80)

In [ ]:
# Block 11-B: Publishability Metrics (ECE + Confusion Matrix + Classification Report)
# Adds Expected Calibration Error, per-model confusion matrices,
# and full classification reports saved to Drive.
# Uses: loaded_models, class_names, test_preds_all, y_test_true,
#       val_preds_all, y_val_true, ensemble_preds, RESULTS_DIR, LOGS_DIR
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
import os

# ---- 1. Expected Calibration Error (ECE) ----
def compute_ece(y_true, y_probs, n_bins=15):
    """
    Compute Expected Calibration Error.
    y_true: 1D array of true labels (int)
    y_probs: 2D array of predicted probabilities (n_samples x n_classes)
    """
    confidences = np.max(y_probs, axis=1)
    predictions = np.argmax(y_probs, axis=1)
    accuracies  = (predictions == y_true).astype(float)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_data = []

    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        bin_acc  = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        bin_size = mask.sum()
        ece += (bin_size / len(y_true)) * abs(bin_acc - bin_conf)
        bin_data.append({'Bin': f'{lo:.2f}-{hi:.2f}', 'Accuracy': round(bin_acc, 4),
                         'Confidence': round(bin_conf, 4), 'Count': int(bin_size)})

    return ece, bin_data

print('='*65)
print(' EXPECTED CALIBRATION ERROR (ECE) — All Models + Ensemble')
print('='*65)

ece_rows = []

# Individual models
for model_name, probs in test_preds_all.items():
    ece_val, _ = compute_ece(y_test_true, probs)
    ece_rows.append({'Model': model_name, 'ECE': round(ece_val, 4)})
    print(f' {model_name:<22} ECE = {ece_val:.4f}')

# Ensemble -- raw (under-confident, poorly calibrated) vs temperature-scaled (calibrated)
top3_avail = [m for m in TOP3 if m in test_preds_all]
X_test_meta = np.concatenate([test_preds_all[m] for m in top3_avail], axis=1)

# raw meta-classifier probabilities
ens_probs = meta_model.predict_proba(X_test_meta)
ens_ece, ens_bins = compute_ece(y_test_true, ens_probs)
ece_rows.append({'Model': 'Stacking Ensemble (raw)', 'ECE': round(ens_ece, 4)})
print(f' {"Stacking Ensemble (raw)":<30} ECE = {ens_ece:.4f}')

# temperature-scaled probabilities (Block 6-B) = what the final two-stage
# predictor actually uses. Guarded so this cell still runs if 6-B was skipped.
try:
    ens_logits     = meta_model.decision_function(X_test_meta)
    ens_probs_cal  = temperature_scale(ens_logits, optimal_T)
    ens_ece_cal, _ = compute_ece(y_test_true, ens_probs_cal)
    ece_rows.append({'Model': 'Stacking Ensemble (temp-scaled)', 'ECE': round(ens_ece_cal, 4)})
    print(f' {"Stacking Ensemble (temp-scaled)":<30} ECE = {ens_ece_cal:.4f}   <-- final pipeline (T={optimal_T:.2f})')
except Exception as e:
    print(f' [temp-scaled ensemble ECE skipped -- run Block 6-B first: {e}]')

df_ece = pd.DataFrame(ece_rows).sort_values('ECE')
ece_csv = os.path.join(RESULTS_DIR, 'expected_calibration_error.csv')
df_ece.to_csv(ece_csv, index=False)
print(f'\nECE saved -> {ece_csv}')

# ECE bar chart
fig, ax = plt.subplots(figsize=(10, 5))
df_ece_sorted = df_ece.sort_values('ECE', ascending=True)
def _bar_color(m):
    if 'temp-scaled' in m:                 return '#27ae60' # green = calibrated (final)
    if m.startswith('Stacking Ensemble'):  return '#e74c3c' # red = raw ensemble
    return '#3498db'
colors = [_bar_color(m) for m in df_ece_sorted['Model']]
ax.barh(df_ece_sorted['Model'], df_ece_sorted['ECE'], color=colors)
ax.set_xlabel('Expected Calibration Error (lower = better)')
ax.set_title('Model Calibration -- raw ensemble (red) vs temperature-scaled (green, final)')
ax.set_xlim(0, max(df_ece_sorted['ECE']) * 1.3)
for i, (_, row) in enumerate(df_ece_sorted.iterrows()):
    ax.text(row['ECE'] + 0.002, i, f'{row["ECE"]:.4f}', va='center', fontsize=9)
plt.tight_layout()
ece_plot = os.path.join(RESULTS_DIR, 'ece_comparison_chart.png')
plt.savefig(ece_plot, dpi=200, bbox_inches='tight')
plt.show()
print(f'ECE chart saved -> {ece_plot}')

# ---- 2. Confusion Matrices ----
print('\n' + '='*65)
print(' CONFUSION MATRICES — Top 3 Models + Ensemble')
print('='*65)

models_to_plot = TOP3 + ['Ensemble']
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

for idx, model_name in enumerate(models_to_plot):
    if model_name == 'Ensemble':
        preds = ensemble_preds
        title = 'Stacking Ensemble'
    else:
        preds = test_preds_all[model_name].argmax(axis=1)
        title = model_name

    cm = confusion_matrix(y_test_true, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=class_names, yticklabels=class_names)
    axes[idx].set_title(title, fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    if idx == 0:
        axes[idx].set_ylabel('True')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].tick_params(axis='y', rotation=0)

plt.suptitle('Confusion Matrices — Top 3 Models + Ensemble', fontsize=14, fontweight='bold')
plt.tight_layout()
cm_path = os.path.join(RESULTS_DIR, 'confusion_matrices.png')
plt.savefig(cm_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Confusion matrices saved -> {cm_path}')

# ---- 3. Full Classification Reports (CSV) ----
print('\n' + '='*65)
print(' CLASSIFICATION REPORTS — All 12 Models + Ensemble')
print('='*65)

all_reports = []
for model_name, probs in test_preds_all.items():
    preds = probs.argmax(axis=1)
    report = classification_report(y_test_true, preds, target_names=class_names, output_dict=True)
    for cls_name, metrics in report.items():
        if cls_name in class_names:
            all_reports.append({
                'Model': model_name,
                'Class': cls_name,
                'Precision': round(metrics['precision'], 4),
                'Recall': round(metrics['recall'], 4),
                'F1-Score': round(metrics['f1-score'], 4),
                'Support': int(metrics['support'])
            })

# Ensemble report
ens_report = classification_report(y_test_true, ensemble_preds, target_names=class_names, output_dict=True)
for cls_name, metrics in ens_report.items():
    if cls_name in class_names:
        all_reports.append({
            'Model': 'Stacking Ensemble',
            'Class': cls_name,
            'Precision': round(metrics['precision'], 4),
            'Recall': round(metrics['recall'], 4),
            'F1-Score': round(metrics['f1-score'], 4),
            'Support': int(metrics['support'])
        })

df_reports = pd.DataFrame(all_reports)
reports_csv = os.path.join(RESULTS_DIR, 'full_classification_reports.csv')
df_reports.to_csv(reports_csv, index=False)
print(f'Full classification reports saved -> {reports_csv}')

# Summary table: per-model weighted avg
summary_rows = []
for model_name, probs in test_preds_all.items():
    preds = probs.argmax(axis=1)
    report = classification_report(y_test_true, preds, target_names=class_names, output_dict=True)
    w = report['weighted avg']
    acc = accuracy_score(y_test_true, preds)
    summary_rows.append({
        'Model': model_name, 'Accuracy': round(acc, 4),
        'Precision': round(w['precision'], 4),
        'Recall': round(w['recall'], 4),
        'F1-Score': round(w['f1-score'], 4)
    })
# Ensemble
w_ens = ens_report['weighted avg']
summary_rows.append({
    'Model': 'Stacking Ensemble', 'Accuracy': round(ensemble_acc, 4),
    'Precision': round(w_ens['precision'], 4),
    'Recall': round(w_ens['recall'], 4),
    'F1-Score': round(w_ens['f1-score'], 4)
})

df_summary = pd.DataFrame(summary_rows).sort_values('Accuracy', ascending=False)
summary_csv = os.path.join(RESULTS_DIR, 'model_comparison_summary.csv')
df_summary.to_csv(summary_csv, index=False)

print('\n' + '='*65)
print(' MODEL COMPARISON SUMMARY (Weighted Avg)')
print('='*65)
print(df_summary.to_string(index=False))
print(f'\nSummary saved -> {summary_csv}')
print('\nAll publishability metrics complete.')

In [ ]:
# Block 11: Test on one image per class
# Reuses the SAME fixed images selected in Block 8 (selected_images.json)
# so this sanity-check evaluates the exact images shown in Block 10's
# gallery -- not an independently (and unsorted-glob) re-picked image.
test_path  = os.path.join(dataset_final_path, 'test')
summary    = []

if 'selected_images' in globals() and selected_images:
    class_to_image = {cls: info['file_path'] for cls, info in selected_images.items()}
else:
    # Fallback if Block 8 hasn't run in this session: pick deterministically
    # (sorted, not raw glob order) so it's at least reproducible.
    class_to_image = {}
    for cls_name in CLASSES:
        imgs = sorted(glob.glob(os.path.join(test_path, cls_name, '*.*')))
        if imgs:
            class_to_image[cls_name] = imgs[0]

print('Testing 1 image per class (same fixed selection as Block 8/10)...\n')
for cls_name in CLASSES:
    img_path = class_to_image.get(cls_name)
    if not img_path or not os.path.exists(img_path):
        print(f' Skipping {cls_name}: no selected image found')
        continue

    print(f'\n--- True class: {cls_name} ---')
    pred_cls, probs = predict_full(img_path, save_result=True, force_generate=True)
    summary.append({
        'True Class' : cls_name,
        'Predicted' : pred_cls,
        'Confidence (%)' : f'{probs.max()*100:.2f}',
        'Correct' : 'YES' if pred_cls == cls_name else 'NO'
    })

if summary:
    summary_df = pd.DataFrame(summary)
    print('\n' + '='*55)
    print('SAMPLE PREDICTION SUMMARY')
    print('='*55)
    print(summary_df.to_string(index=False))
    summary_df.to_csv(os.path.join(RESULTS_DIR, 'sample_predictions.csv'), index=False)
    print(f'\nSummary saved -> {RESULTS_DIR}')
else:
    print('\nAll predictions already done.')

In [ ]:
# Block 11 – External Dataset Evaluation
# Evaluates the EXISTING calibrated stacking ensemble (Block 6 + 6-B) on a
# new, external dataset uploaded as a ZIP from your local computer.
# DOES NOT retrain. DOES NOT use the specialist model. DOES NOT run Grad-CAM.
# Reuses: loaded_models, TOP3, meta_model, temperature_scale, optimal_T,
#         class_names, IMG_SIZE, DRIVE_WORKSPACE
import os, glob, zipfile, shutil, re, difflib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm
from google.colab import files

IMG_EXTS       = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
EXT_ROOT       = '/content/external_dataset'
EXT_EXPORT_DIR = os.path.join(DRIVE_WORKSPACE, 'External_Evaluation')
os.makedirs(EXT_EXPORT_DIR, exist_ok=True)

# STEP 1: Upload + extract ZIP from local PC
if os.path.exists(EXT_ROOT):
    shutil.rmtree(EXT_ROOT)
os.makedirs(EXT_ROOT, exist_ok=True)

print("Select a ZIP file containing the external dataset...")
uploaded = files.upload()
zip_candidates = [f for f in uploaded if f.lower().endswith('.zip')]
if not zip_candidates:
    raise ValueError("No .zip file was uploaded. Please upload a .zip archive and re-run this block.")

zip_name = zip_candidates[0]
zip_path = os.path.join(EXT_ROOT, zip_name)
shutil.move(zip_name, zip_path)

extract_dir = os.path.join(EXT_ROOT, 'extracted')
os.makedirs(extract_dir, exist_ok=True)
try:
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    print(f"[OK] Extracted '{zip_name}' -> {extract_dir}")
except zipfile.BadZipFile:
    raise ValueError("Uploaded file is not a valid ZIP archive.")

# STEP 2: Auto-detect dataset structure (any nesting depth)
def find_class_folders(root):
    """Return {folder_name: [image_paths]} for every leaf dir that has images,
    regardless of how deep it's nested (Dataset/X, Dataset/test/X, Dataset/images/X...)."""
    class_folders = {}
    for dirpath, _, filenames in os.walk(root):
        imgs = [f for f in filenames if f.lower().endswith(IMG_EXTS)]
        if not imgs:
            continue
        folder_name = os.path.basename(dirpath.rstrip('/'))
        paths = [os.path.join(dirpath, f) for f in imgs]
        class_folders.setdefault(folder_name, []).extend(paths)
    return class_folders

raw_class_folders = find_class_folders(extract_dir)
if not raw_class_folders:
    raise ValueError("No class folders with images were found in the uploaded ZIP.")

print(f"\nDetected {len(raw_class_folders)} candidate folders:")
for name, paths in raw_class_folders.items():
    print(f"  {name}: {len(paths)} images")

# STEP 3: Automatic class matching to the model's supported classes
CLASS_SYNONYMS = {
    'acne': 'Acne', 'acne vulgaris': 'Acne',
    'akiec': 'Actinic keratosis', 'actinic keratosis': 'Actinic keratosis',
    'actinic keratoses': 'Actinic keratosis',
    'bcc': 'Basal cell carcinoma', 'basal cell carcinoma': 'Basal cell carcinoma',
    'chickenpox': 'Chickenpox', 'chicken pox': 'Chickenpox', 'varicella': 'Chickenpox',
    'measles': 'Measles', 'rubeola': 'Measles',
    'nv': 'Melanocytic nevus', 'melanocytic nevus': 'Melanocytic nevus',
    'melanocytic nevi': 'Melanocytic nevus', 'nevus': 'Melanocytic nevus', 'mole': 'Melanocytic nevus',
    'normal': 'Normal  Unknown', 'unknown': 'Normal  Unknown', 'healthy': 'Normal  Unknown',
    'tinea': 'Tinea', 'tinea corporis': 'Tinea', 'ringworm': 'Tinea',
    'vasc': 'Vascular lesion', 'vascular lesion': 'Vascular lesion', 'vascular': 'Vascular lesion',
    'vitiligo': 'Vitiligo',
}

def match_class(folder_name):
    key = re.sub(r'[^a-z0-9 ]', '', folder_name.lower()).strip()
    key = re.sub(r'\s+', ' ', key)
    if key in CLASS_SYNONYMS:
        return CLASS_SYNONYMS[key]
    for cn in class_names:
        if key == re.sub(r'\s+', ' ', cn.lower().strip()):
            return cn
    close = difflib.get_close_matches(key, [c.lower() for c in class_names], n=1, cutoff=0.7)
    if close:
        for cn in class_names:
            if cn.lower() == close[0]:
                return cn
    return None

matched_classes = {}
skipped_classes = {}
for folder_name, paths in raw_class_folders.items():
    mapped = match_class(folder_name)
    if mapped is not None:
        matched_classes.setdefault(mapped, []).extend(paths)
    else:
        skipped_classes[folder_name] = len(paths)

print("\n=== Matched classes ===")
for cn, paths in matched_classes.items():
    print(f"  {cn}: {len(paths)} images")
print("\n=== Skipped (unsupported) classes ===")
if skipped_classes:
    for name, n in skipped_classes.items():
        print(f"  {name}: {n} images")
else:
    print("  none")

eval_records = [(p, cls) for cls, paths in matched_classes.items() for p in paths]
print(f"\nTotal evaluable images: {len(eval_records)}")
if not eval_records:
    raise ValueError("No images matched the model's supported classes. Nothing to evaluate.")

# STEP 4: Ensemble-only inference (reuses Block 6 / 6-B artifacts)
def _predict_one_model(name, model, image_path, img_arr):
    if name == 'YOLO11m_cls':
        res = model(image_path, verbose=False)[0]
        raw_probs = res.probs.data.cpu().numpy()
        idx_to_name = model.names
        name_to_idx = {c: i for i, c in enumerate(class_names)}
        fixed = np.zeros_like(raw_probs)
        for yolo_i, cname in idx_to_name.items():
            fixed[name_to_idx[cname]] = raw_probs[yolo_i]
        return fixed
    return model.predict(img_arr, verbose=0)[0]

def predict_ensemble_only(image_path):
    try:
        img_arr = np.expand_dims(
            tf.keras.utils.img_to_array(
                tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
            ), axis=0
        )
    except Exception as e:
        return None, None, f"unreadable/corrupted image: {e}"

    top3_probs = {}
    for name in TOP3:
        if name in loaded_models:
            try:
                top3_probs[name] = _predict_one_model(name, loaded_models[name], image_path, img_arr)
            except Exception as e:
                return None, None, f"{name} inference failed: {e}"

    if len(top3_probs) != len(TOP3):
        return None, None, "not all TOP3 models available"

    meta_input = np.concatenate([top3_probs[m] for m in TOP3]).reshape(1, -1)
    try:
        meta_logits = meta_model.decision_function(meta_input)
        meta_probs  = temperature_scale(meta_logits, optimal_T)[0]
    except Exception:
        meta_probs = meta_model.predict_proba(meta_input)[0]

    pred_idx = int(np.argmax(meta_probs))
    return class_names[pred_idx], float(meta_probs[pred_idx] * 100), None

results_rows, failed_images = [], []
for img_path, true_cls in tqdm(eval_records, desc="Evaluating external dataset"):
    pred_cls, conf, err = predict_ensemble_only(img_path)
    if pred_cls is None:
        failed_images.append((os.path.basename(img_path), err))
        continue
    results_rows.append({
        'Image': os.path.basename(img_path),
        'True Class': true_cls,
        'Predicted Class': pred_cls,
        'Confidence': round(conf, 2),
        'Correct': pred_cls == true_cls
    })

results_df = pd.DataFrame(results_rows)
print(f"\nSuccessfully evaluated: {len(results_df)} / {len(eval_records)}")
if failed_images:
    print(f"Skipped {len(failed_images)} images due to errors (corrupted/unreadable).")

# STEP 5: Disease-wise results (first 100 shown, all saved)
for disease in sorted(results_df['True Class'].unique()):
    sub = results_df[results_df['True Class'] == disease].reset_index(drop=True)
    print(f"\n{'='*60}\nDisease : {disease}\nImages  : {len(sub)}\nShowing first {min(100, len(sub))} results\n{'='*60}")
    display_cols = ['Image', 'Predicted Class', 'Confidence', 'Correct']
    print(sub[display_cols].head(100).to_string(index=False))

    safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', disease).strip('_')
    sub.to_csv(os.path.join(EXT_EXPORT_DIR, f'{safe_name}_results.csv'), index=False)
    sub.to_excel(os.path.join(EXT_EXPORT_DIR, f'{safe_name}_results.xlsx'), index=False)

# STEP 6: Disease summary + overall summary
disease_summary = results_df.groupby('True Class').agg(
    Images=('Correct', 'count'),
    Correct=('Correct', 'sum'),
    Avg_Confidence=('Confidence', 'mean')
).reset_index()
disease_summary['Wrong']    = disease_summary['Images'] - disease_summary['Correct']
disease_summary['Accuracy'] = (disease_summary['Correct'] / disease_summary['Images'] * 100).round(2)
disease_summary['Avg_Confidence'] = disease_summary['Avg_Confidence'].round(2)
disease_summary = disease_summary[['True Class', 'Images', 'Correct', 'Wrong', 'Accuracy', 'Avg_Confidence']]
disease_summary.columns = ['Disease', 'Images', 'Correct', 'Wrong', 'Accuracy (%)', 'Average Confidence']

print("\n" + "="*60 + "\nDISEASE SUMMARY\n" + "="*60)
print(disease_summary.to_string(index=False))

overall_summary = pd.DataFrame([
    {'Metric': 'Total Images',        'Value': len(results_df)},
    {'Metric': 'Matched Classes',     'Value': results_df['True Class'].nunique()},
    {'Metric': 'Skipped Classes',     'Value': len(skipped_classes)},
    {'Metric': 'Correct Predictions', 'Value': int(results_df['Correct'].sum())},
    {'Metric': 'Wrong Predictions',   'Value': int((~results_df['Correct']).sum())},
    {'Metric': 'Overall Accuracy (%)','Value': round(results_df['Correct'].mean() * 100, 2)},
    {'Metric': 'Average Confidence',  'Value': round(results_df['Confidence'].mean(), 2)},
])
print("\n" + "="*60 + "\nOVERALL SUMMARY\n" + "="*60)
print(overall_summary.to_string(index=False))

# STEP 7: Confusion matrix + classification report
eval_labels = sorted(results_df['True Class'].unique())
cm = confusion_matrix(results_df['True Class'], results_df['Predicted Class'], labels=eval_labels)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=eval_labels, yticklabels=eval_labels)
plt.title('External Dataset — Confusion Matrix', fontsize=14)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
cm_path = os.path.join(EXT_EXPORT_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=300)
plt.show()

report_dict = classification_report(
    results_df['True Class'], results_df['Predicted Class'],
    labels=eval_labels, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(os.path.join(EXT_EXPORT_DIR, 'classification_report.csv'))

# STEP 8: Export everything
results_df.to_csv(os.path.join(EXT_EXPORT_DIR, 'combined_results.csv'), index=False)
overall_summary.to_csv(os.path.join(EXT_EXPORT_DIR, 'overall_summary.csv'), index=False)
overall_summary.to_excel(os.path.join(EXT_EXPORT_DIR, 'overall_summary.xlsx'), index=False)
disease_summary.to_csv(os.path.join(EXT_EXPORT_DIR, 'disease_summary.csv'), index=False)

print(f"\n[OK] All results exported to: {EXT_EXPORT_DIR}")

In [ ]:
# Block 12: Final Summary
print('\n' + '='*65)
print('COMPLETE RESULTS SUMMARY')
print('='*65)

print('\n1. ALL CNN MODELS:')
for name, acc in sorted(model_acc.items(), key=lambda x: x[1], reverse=True):
    tag = ' <- top 3 (ensemble)' if name in TOP3 else ''
    print(f' {name:<22} {acc*100:.2f}%{tag}')

print(f'\n2. STACKING ENSEMBLE (Top 3): {ensemble_acc*100:.2f}%  <- best')

print('\n3. PER-CLASS BEST MODEL:')
print(best_models_per_disease.to_string(index=False))

print(f'\n4. GRAD-CAM + BOUNDING BOX:')
print(f' Saved -> {GRADCAM_DIR}')
print(f' Saved -> {RESULTS_DIR}')

print('\n' + '='*65)
print('All Drive folders:')
print(f' Models     : {MODELS_DIR}')
print(f' Logs       : {LOGS_DIR}')
print(f' Per-class  : {PERCLASS_DIR}')
print(f' Grad-CAM   : {GRADCAM_DIR}')
print(f' Results    : {RESULTS_DIR}')
print('='*65)

In [ ]:
# Block 13: Interactive upload and predict (saves to web_uploads/)
import os
import urllib.request
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from datetime import datetime

# Create a separate folder for web/PC uploads
WEB_UPLOAD_DIR = os.path.join(DRIVE_WORKSPACE, 'web_uploads')
os.makedirs(WEB_UPLOAD_DIR, exist_ok=True)

def on_url_predict_clicked(b):
    with output_area:
        clear_output()
        url = url_input.value.strip()
        if not url:
            print("Please enter a valid URL first!")
            return

        print('\n[Online Image] Downloading image from the internet...')
        try:
            # Use timestamp to avoid overwriting
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            img_path = os.path.join(WEB_UPLOAD_DIR, f'web_url_{timestamp}.jpg')
            urllib.request.urlretrieve(url, img_path)
            print(f'Download successful! Saved to: {img_path}')
            print('Predicting...')
            # Save result to web_uploads folder
            predict_full(img_path, save_result=True, force_generate=True, output_dir=WEB_UPLOAD_DIR)
        except Exception as e:
            print(f'Failed to download image: {e}')

def on_upload_clicked(b):
    with output_area:
        clear_output()
        print("\n[Local Upload] Select an image from your computer:")
        uploaded = files.upload()
        if uploaded:
            original_name = list(uploaded.keys())[0]
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            name, ext = os.path.splitext(original_name)
            new_name = f'web_upload_{timestamp}{ext}'
            img_path = os.path.join(WEB_UPLOAD_DIR, new_name)

            # Move the uploaded file to web_uploads folder
            import shutil
            shutil.move(original_name, img_path)

            print(f'Upload successful! Saved to: {img_path}')
            print('Predicting...')
            predict_full(img_path, save_result=True, force_generate=True, output_dir=WEB_UPLOAD_DIR)
        else:
            print('No image was uploaded!')

# UI Design (buttons and input boxes)
url_input = widgets.Text(
    value='',
    placeholder='Paste Image URL here...',
    description='Image URL:',
    layout=widgets.Layout(width='60%')
)

url_button = widgets.Button(
    description='Predict from URL',
    button_style='info',
    icon='search'
)
url_button.on_click(on_url_predict_clicked)

upload_button = widgets.Button(
    description='Upload from PC & Predict',
    button_style='success',
    icon='upload',
    layout=widgets.Layout(width='30%')
)
upload_button.on_click(on_upload_clicked)

output_area = widgets.Output()

# Display the UI
print("="*60)
print("📁 WEB/PC UPLOADS (saved to: web_uploads/)")
print("="*60)
display(widgets.HBox([url_input, url_button]))
print("OR")
display(upload_button)
display(output_area)

print("\n📂 All web/PC uploads are saved to: web_uploads/")
print("📂 Mobile uploads are saved to: mobile_uploads/")
print("📂 Test dataset predictions are saved to: final_results/")

In [ ]:
# Block 14: Mobile Camera Web App (SAME ensemble as Block 10, via phone camera)
# Architecture:
#   Phone camera (live preview + capture) -> Flask server running HERE in
#   Colab -> predict_full() (the EXACT same function from Block 10, already
#   loaded in memory with all 12 models + ensemble + specialist + Grad-CAM)
#   -> a JSON result (predicted class, confidence, specialist re-check,
#   full probability list, heatmap images) is sent back and rendered as a
#   clean mobile card. A link to the full desktop-style 8-panel figure is
#   also included.
#
# The phone does NO computation -- it only takes the picture and displays
# the result. All inference runs on Colab's GPU, same as every other block.
#
# A QR code for the link is printed in this cell's output -- scan it with a
# phone camera to open the site directly (no typing the URL).
#
# REQUIREMENT: run this AFTER Block 10 has run at least once in this session
# (predict_full, loaded_models, meta_model, etc. must already exist).
#
# SETUP (one-time): add a Colab Secret named AUTH_NGROK with your free
# authtoken from https://dashboard.ngrok.com, and toggle "Notebook access" ON.

get_ipython().system('pip install flask pyngrok qrcode[pil] --quiet')

import os
import time
import shutil
from flask import Flask, request, send_file, render_template_string, jsonify
from pyngrok import ngrok, conf
from google.colab import userdata
import qrcode
from IPython.display import display

# STEP 1: ngrok auth (read from Colab Secrets tab, key name: AUTH_NGROK)
NGROK_AUTHTOKEN = userdata.get('AUTH_NGROK')

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN
else:
    print("WARNING: 'AUTH_NGROK' secret not found or empty. Add it in Colab's "
          "Secrets tab (key icon, left sidebar) and toggle 'Notebook access' ON.")

# Mobile upload directory - ALL mobile files go here, separate from test results
MOBILE_UPLOAD_DIR = os.path.join(DRIVE_WORKSPACE, 'mobile_uploads')
os.makedirs(MOBILE_UPLOAD_DIR, exist_ok=True)

# Also ensure the results directory exists for the full figure
os.makedirs(RESULTS_DIR, exist_ok=True)

# STEP 2: mobile-friendly page
#   - live camera preview, capture, flip (front/back), torch (Android only)
#   - freezes on the captured frame + counts up while the server works
#   - renders the result as a big heatmap image + readable text, not the
#     small desktop-style 8-panel figure (that's still one tap away)
MOBILE_PAGE = '''
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Skin Disease Classifier</title>
  <style>
    body { font-family: -apple-system, sans-serif; background:#f8f9fa; margin:0; padding:16px; }
    h2 { color:#2c3e50; text-align:center; margin-top:4px; }
    .card { background:white; border-radius:12px; padding:16px; box-shadow:0 2px 8px rgba(0,0,0,0.08); max-width:520px; margin:0 auto 16px; }
    .stage { position:relative; width:100%; aspect-ratio:3/4; background:#000; border-radius:8px; overflow:hidden; }
    video, #frozen { width:100%; height:100%; object-fit:cover; display:block; }
    #frozen { display:none; }
    canvas { display:none; }
    .camControls { display:flex; gap:8px; margin-top:10px; }
    .camControls button { flex:1; }
    button { padding:14px; font-size:15px; font-weight:bold; color:white;
             background:#3498db; border:none; border-radius:8px; cursor:pointer; }
    button.secondary { background:#7f8c8d; }
    button:disabled { background:#bdc3c7; cursor:not-allowed; }
    #captureBtn { width:100%; margin-top:10px; }
    .divider { text-align:center; color:#95a5a6; margin:14px 0; font-size:13px; }
    label.filebtn { display:block; text-align:center; padding:12px; border:2px dashed #bdc3c7;
                    border-radius:8px; color:#7f8c8d; cursor:pointer; font-size:14px; }
    input[type=file] { display:none; }
    #status { text-align:center; color:#7f8c8d; margin:12px 0; font-weight:bold; min-height:24px; }
    #resultCard { display:none; }
    #resultCard img { width:100%; border-radius:8px; margin-bottom:10px; }
    .predHead { text-align:center; padding:14px; border-radius:8px; background:#eaf4fc; margin-bottom:12px; }
    .predHead .cls { font-size:24px; font-weight:bold; color:#2c3e50; }
    .predHead .conf { font-size:15px; color:#3498db; font-weight:bold; }
    .specialistBox { background:#fdf6ea; border-radius:8px; padding:10px 12px; margin-bottom:12px; font-size:14px; color:#7a5c00; }
    table.probs { width:100%; border-collapse:collapse; font-size:14px; margin-bottom:12px; }
    table.probs td { padding:6px 4px; border-bottom:1px solid #eee; }
    table.probs td.pct { text-align:right; font-weight:bold; color:#2c3e50; }
    tr.topRow td { color:#e74c3c; }
    a.fullLink { display:block; text-align:center; color:#3498db; font-size:13px; margin-top:6px; }
    #retakeBtn { width:100%; margin-top:6px; }
    .error { color:#e74c3c; text-align:center; padding:10px; }
  </style>
</head>
<body>
  <h2>Skin Disease Classifier</h2>

  <div class="card" id="captureCard">
    <div class="stage">
      <video id="video" autoplay playsinline muted></video>
      <img id="frozen">
    </div>
    <div class="camControls">
      <button id="flipBtn" class="secondary" type="button">Flip Camera</button>
      <button id="torchBtn" class="secondary" type="button">Flash</button>
    </div>
    <button id="captureBtn" type="button">Capture Photo</button>

    <div class="divider">or</div>
    <label class="filebtn" for="galleryInput">Choose photo from gallery</label>
    <input type="file" accept="image/*" id="galleryInput">

    <p id="status"></p>
  </div>

  <div class="card" id="resultCard">
    <div class="predHead">
      <div class="cls" id="predClass"></div>
      <div class="conf" id="predConf"></div>
    </div>
    <img id="heatmapImg">
    <div class="specialistBox" id="specialistText"></div>
    <table class="probs" id="probsTable"></table>
    <a class="fullLink" id="fullLink" href="#" target="_blank">View full technical panel</a>
    <button id="retakeBtn" type="button">Take Another Photo</button>
  </div>

  <canvas id="canvas"></canvas>

  <script>
    const video       = document.getElementById('video');
    const frozen      = document.getElementById('frozen');
    const canvas      = document.getElementById('canvas');
    const captureBtn  = document.getElementById('captureBtn');
    const flipBtn     = document.getElementById('flipBtn');
    const torchBtn    = document.getElementById('torchBtn');
    const galleryInput= document.getElementById('galleryInput');
    const status      = document.getElementById('status');
    const captureCard = document.getElementById('captureCard');
    const resultCard  = document.getElementById('resultCard');
    const retakeBtn   = document.getElementById('retakeBtn');

    let currentStream  = null;
    let facingMode     = 'environment';
    let torchOn        = false;
    let countdownTimer = null;

    async function startCamera(mode) {
      stopCamera();
      try {
        currentStream = await navigator.mediaDevices.getUserMedia({
          video: { facingMode: { ideal: mode } },
          audio: false
        });
        video.srcObject = currentStream;
        video.style.display = 'block';
        frozen.style.display = 'none';
        const track = currentStream.getVideoTracks()[0];
        const caps = track.getCapabilities ? track.getCapabilities() : {};
        torchBtn.style.display = caps.torch ? 'block' : 'none';
        torchOn = false;
      } catch (err) {
        status.textContent = 'Camera not available (' + err.message + '). Use "Choose photo from gallery" below.';
        status.style.color = '#e74c3c';
      }
    }

    function stopCamera() {
      if (currentStream) {
        currentStream.getTracks().forEach(t => t.stop());
        currentStream = null;
      }
    }

    startCamera(facingMode);

    flipBtn.onclick = () => {
      facingMode = (facingMode === 'environment') ? 'user' : 'environment';
      startCamera(facingMode);
    };

    torchBtn.onclick = async () => {
      if (!currentStream) return;
      const track = currentStream.getVideoTracks()[0];
      try {
        torchOn = !torchOn;
        await track.applyConstraints({ advanced: [{ torch: torchOn }] });
      } catch (err) {
        status.textContent = 'Flash control not supported on this device/browser.';
        status.style.color = '#e74c3c';
        torchOn = false;
      }
    };

    function startCountdown() {
      let secs = 0;
      status.textContent = 'Analyzing... 0s';
      status.style.color = '#7f8c8d';
      countdownTimer = setInterval(() => {
        secs += 1;
        status.textContent = 'Analyzing (full 12-model ensemble)... ' + secs + 's';
      }, 1000);
    }

    function stopCountdown() {
      clearInterval(countdownTimer);
      status.textContent = '';
    }

    async function uploadImage(blob, filename) {
      captureBtn.disabled = true;
      startCountdown();
      const formData = new FormData();
      formData.append('image', blob, filename || 'photo.jpg');
      try {
        const res = await fetch('/predict', { method: 'POST', body: formData });
        if (!res.ok) throw new Error('Server error: ' + res.status);
        const data = await res.json();
        if (data.error) throw new Error(data.error);
        renderResult(data);
      } catch (err) {
        status.textContent = 'Error: ' + err.message;
        status.style.color = '#e74c3c';
      }
      stopCountdown();
      captureBtn.disabled = false;
    }

    function renderResult(d) {
      document.getElementById('predClass').textContent = d.ensemble_class;
      document.getElementById('predConf').textContent =
        d.ensemble_confidence + '% confidence (calibrated ensemble of Top-3 models)';
      document.getElementById('heatmapImg').src = 'data:image/jpeg;base64,' + d.ensemble_heatmap_b64;
      document.getElementById('specialistText').textContent =
        'Specialist re-check (' + d.specialist_model + '): ' + d.specialist_confidence +
        '% on "' + d.ensemble_class + '" [' + d.agreement + ']';

      const tbl = document.getElementById('probsTable');
      tbl.innerHTML = '';
      d.all_probs.forEach((row, i) => {
        const tr = document.createElement('tr');
        if (i === 0) tr.className = 'topRow';
        tr.innerHTML = '<td>' + row.class + '</td><td class="pct">' + row.pct + '%</td>';
        tbl.appendChild(tr);
      });

      document.getElementById('fullLink').href = '/full/' + d.full_figure_filename;

      captureCard.style.display = 'none';
      resultCard.style.display = 'block';
    }

    captureBtn.onclick = () => {
      if (!video.videoWidth || !video.videoHeight) {
        status.textContent = 'Camera not ready. Please wait.';
        status.style.color = '#e74c3c';
        return;
      }
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      frozen.src = canvas.toDataURL('image/jpeg', 0.9);
      video.style.display = 'none';
      frozen.style.display = 'block';
      stopCamera();  // close the camera preview right after capture
      canvas.toBlob((blob) => uploadImage(blob, 'capture.jpg'), 'image/jpeg', 0.92);
    };

    galleryInput.onchange = () => {
      const file = galleryInput.files[0];
      if (file) uploadImage(file, file.name);
    };

    retakeBtn.onclick = () => {
      resultCard.style.display = 'none';
      captureCard.style.display = 'block';
      startCamera(facingMode);
    };
  </script>
</body>
</html>
'''

# STEP 3: Flask app -- reuses predict_full() from Block 10 directly
app = Flask(__name__)

@app.route('/')
def home():
    return render_template_string(MOBILE_PAGE)

@app.route('/predict', methods=['POST'])
def predict_route():
    if 'image' not in request.files:
        return jsonify({'error': 'no image uploaded'}), 400

    file = request.files['image']
    # Use timestamp for unique filename
    timestamp = int(time.time())
    fname = f"mobile_{timestamp}.jpg"
    save_path = os.path.join(MOBILE_UPLOAD_DIR, fname)
    file.save(save_path)
    print(f"[MOBILE] Mobile upload saved: {save_path}")

    try:
        # Call the SAME predict_full() function from Block 10
        # It will save the 8-panel figure to MOBILE_UPLOAD_DIR
        predict_full(
            save_path,
            save_result=True,
            force_generate=True,
            output_dir=MOBILE_UPLOAD_DIR  # Save mobile results to mobile folder
        )
    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

    # Check if LAST_PREDICTION_DETAILS was populated
    if LAST_PREDICTION_DETAILS is None:
        return jsonify({'error': 'prediction details unavailable'}), 500

    # Return the mobile-friendly JSON summary
    return jsonify(LAST_PREDICTION_DETAILS)

@app.route('/full/<path:filename>')
def full_figure(filename):
    """Serve the full 8-panel figure from the mobile_uploads folder."""
    # Try mobile_uploads first
    mobile_path = os.path.join(MOBILE_UPLOAD_DIR, filename)
    if os.path.exists(mobile_path):
        return send_file(mobile_path, mimetype='image/png')
    # Fallback to RESULTS_DIR
    return send_file(os.path.join(RESULTS_DIR, filename), mimetype='image/png')

# STEP 4: Clean up any old mobile_uploads from previous runs
def cleanup_old_mobile_uploads(max_files=50):
    """Keep only the most recent N mobile uploads to save space."""
    files = sorted(
        glob.glob(os.path.join(MOBILE_UPLOAD_DIR, '*.jpg')),
        key=os.path.getmtime,
        reverse=True
    )
    # Keep images, delete old ones beyond max_files
    if len(files) > max_files:
        for f in files[max_files:]:
            try:
                os.remove(f)
                print(f" [INFO] Removed old upload: {os.path.basename(f)}")
            except:
                pass

    # Also clean up old result images
    result_files = sorted(
        glob.glob(os.path.join(MOBILE_UPLOAD_DIR, '*_2stage_prediction.png')),
        key=os.path.getmtime,
        reverse=True
    )
    if len(result_files) > max_files:
        for f in result_files[max_files:]:
            try:
                os.remove(f)
                print(f" [INFO] Removed old result: {os.path.basename(f)}")
            except:
                pass

# Run cleanup
cleanup_old_mobile_uploads(max_files=30)
print(f"[INFO] Mobile uploads folder: {MOBILE_UPLOAD_DIR}")
print(f"[INFO] Results folder: {RESULTS_DIR}")

# STEP 5: start the tunnel, print the link, and show a scannable QR code
# Kill any existing ngrok tunnels
try:
    ngrok.kill()
except:
    pass

# Start new tunnel
tunnel = ngrok.connect(5000)
url_str = tunnel.public_url

print("=" * 70)
print(f" [MOBILE] Open this on your phone:  {url_str}")
print(" Or scan the QR code below with your phone's camera:")
print("=" * 70)

# Generate and display QR code
qr_img = qrcode.make(url_str)
display(qr_img)

print("=" * 70)
print(" [NOTE] Mobile app features:")
print(" - Flip Camera (front/back)")
print(" - Flash (Android only; iOS Safari does NOT support torch control)")
print(" - Gallery upload")
print(" - Results shown as a clean mobile card")
print(" - Full technical panel available via link")
print("=" * 70)
print(" [WARNING] Keep this cell running -- closing it stops the server.")
print(" [STOP] Stop with: Runtime -> Interrupt execution")
print("=" * 70)

app.run(port=5000)